# 06 — Agente Redactor v1.6
Shell transaccional con preservación del comportamiento del notebook antiguo.

In [1]:
from pathlib import Path

root = Path("/content/tesis_codigo")

paths = [
    root / "src/tools/draft_writing/hybrid_retrieval.py",
    root / "src/adapters/draft_writing_hybrid_runtime.py",
    root / "tests/v16/test_agent06_hybrid_retrieval_v16.py",
]

for path in paths:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text("", encoding="utf-8")
    print(path)

/content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py
/content/tesis_codigo/tests/v16/test_agent06_hybrid_retrieval_v16.py


In [3]:
import sys
from pathlib import Path

CODE_ROOT = Path("/content/tesis_codigo").resolve()

assert CODE_ROOT.exists(), f"No existe el repositorio: {CODE_ROOT}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("CODE_ROOT:", CODE_ROOT)
print("src existe:", (CODE_ROOT / "src").exists())
print("sys.path[0]:", sys.path[0])

CODE_ROOT: /content/tesis_codigo
src existe: True
sys.path[0]: /content/tesis_codigo


In [4]:
import inspect

from src.tools.draft_writing.retrieval import (
    query_chroma_restricted,
    query_csv_restricted,
    dedupe_evidence,
)

print("query_chroma_restricted")
print(inspect.signature(query_chroma_restricted))
print(inspect.getsource(query_chroma_restricted))

print("\nquery_csv_restricted")
print(inspect.signature(query_csv_restricted))
print(inspect.getsource(query_csv_restricted))

print("\ndedupe_evidence")
print(inspect.signature(dedupe_evidence))
print(inspect.getsource(dedupe_evidence))

query_chroma_restricted
(collection, chunks_df, query, source_filenames, top_k, max_evidence_chars=18000, valid_source_chunk_pairs=None)
def query_chroma_restricted(
    collection,
    chunks_df,
    query,
    source_filenames,
    top_k,
    max_evidence_chars=18000,
    valid_source_chunk_pairs=None,
):
    if not source_filenames:
        return []

    per_source_k = max(1, top_k // len(source_filenames) + 1)
    rows = []

    for source in source_filenames:
        source_chunk_count = int(
            (chunks_df["source_filename"].astype(str) == source).sum()
        )
        result = collection.query(
            query_texts=[query],
            n_results=min(per_source_k, max(1, source_chunk_count)),
            where={"source_filename": source},
            include=["documents", "metadatas", "distances"],
        )

        documents = (result.get("documents") or [[]])[0]
        metadatas = (result.get("metadatas") or [[]])[0]
        distances = (result.get("distances") 

In [5]:
from pathlib import Path

path = Path(
    "/content/tesis_codigo/src/tools/draft_writing/"
    "hybrid_retrieval.py"
)

code = r'''
from __future__ import annotations

from collections import defaultdict
from typing import Any, Callable


def _candidate_key(row: dict[str, Any]) -> tuple[str, str]:
    return (
        str(row.get("source_filename", "")).strip(),
        str(row.get("chunk_id", "")).strip(),
    )


def reciprocal_rank_fusion(
    chroma_rows: list[dict[str, Any]],
    csv_rows: list[dict[str, Any]],
    *,
    rrf_k: int = 60,
) -> list[dict[str, Any]]:
    """
    Fusiona dos rankings sin comparar directamente sus scores originales.
    """

    if rrf_k <= 0:
        raise ValueError("rrf_k debe ser mayor que cero")

    fused: dict[tuple[str, str], dict[str, Any]] = {}
    fused_scores: defaultdict[tuple[str, str], float] = defaultdict(float)

    rankings = (
        ("chroma_restricted", chroma_rows),
        ("csv_lexical_restricted", csv_rows),
    )

    for method, rows in rankings:
        for rank, original in enumerate(rows, start=1):
            row = dict(original)
            key = _candidate_key(row)

            if not key[0] or not key[1]:
                continue

            fused_scores[key] += 1.0 / (rrf_k + rank)

            if key not in fused:
                fused[key] = row
                fused[key]["retrieval_methods"] = []
                fused[key]["component_ranks"] = {}
                fused[key]["component_scores"] = {}

            item = fused[key]

            if method not in item["retrieval_methods"]:
                item["retrieval_methods"].append(method)

            item["component_ranks"][method] = rank
            item["component_scores"][method] = float(
                row.get("score", 0.0) or 0.0
            )

    output: list[dict[str, Any]] = []

    for key, item in fused.items():
        row = dict(item)
        row["hybrid_score"] = fused_scores[key]
        row["score"] = fused_scores[key]
        row["retrieval_method"] = "hybrid_rrf"
        output.append(row)

    output.sort(
        key=lambda row: (
            -float(row["hybrid_score"]),
            str(row.get("source_filename", "")),
            str(row.get("chunk_id", "")),
        )
    )

    return output


def retrieve_hybrid_evidence(
    *,
    query: str,
    source_filenames: list[str],
    top_k: int,
    query_chroma: Callable[..., list[dict[str, Any]]],
    query_csv: Callable[..., list[dict[str, Any]]],
    deduplicate: Callable[..., list[dict[str, Any]]],
    chroma_kwargs: dict[str, Any] | None = None,
    csv_kwargs: dict[str, Any] | None = None,
    dedupe_kwargs: dict[str, Any] | None = None,
    rrf_k: int = 60,
) -> list[dict[str, Any]]:
    """
    Variante experimental:

    Chroma + CSV
    -> Reciprocal Rank Fusion
    -> deduplicación/validación heredada
    -> top_k
    """

    if top_k <= 0:
        return []

    chroma_kwargs = dict(chroma_kwargs or {})
    csv_kwargs = dict(csv_kwargs or {})
    dedupe_kwargs = dict(dedupe_kwargs or {})

    chroma_rows = query_chroma(
        query=query,
        source_filenames=source_filenames,
        top_k=top_k,
        **chroma_kwargs,
    )

    csv_rows = query_csv(
        query=query,
        source_filenames=source_filenames,
        top_k=top_k,
        **csv_kwargs,
    )

    fused_rows = reciprocal_rank_fusion(
        chroma_rows,
        csv_rows,
        rrf_k=rrf_k,
    )

    deduplicated_rows = deduplicate(
        fused_rows,
        **dedupe_kwargs,
    )

    return deduplicated_rows[:top_k]
'''

path.write_text(code, encoding="utf-8")
print("Creado:", path)

Creado: /content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py


In [6]:
from src.tools.draft_writing.hybrid_retrieval import reciprocal_rank_fusion

chroma_test = [
    {
        "source_filename": "paper_a.pdf",
        "chunk_id": "a1",
        "score": 0.80,
        "text": "semántico",
    },
    {
        "source_filename": "paper_b.pdf",
        "chunk_id": "b1",
        "score": 0.70,
        "text": "semántico",
    },
]

csv_test = [
    {
        "source_filename": "paper_c.pdf",
        "chunk_id": "c1",
        "score": 0.12,
        "text": "10%, 58.7%, 6.11%",
    },
    {
        "source_filename": "paper_a.pdf",
        "chunk_id": "a1",
        "score": 0.10,
        "text": "semántico",
    },
]

result = reciprocal_rank_fusion(chroma_test, csv_test)

for row in result:
    print(
        row["source_filename"],
        row["chunk_id"],
        row["hybrid_score"],
        row["retrieval_methods"],
    )

paper_a.pdf a1 0.03252247488101534 ['chroma_restricted', 'csv_lexical_restricted']
paper_c.pdf c1 0.01639344262295082 ['csv_lexical_restricted']
paper_b.pdf b1 0.016129032258064516 ['chroma_restricted']


In [7]:
query_chroma_restricted
query_csv_restricted
dedupe_evidence

<function src.tools.draft_writing.retrieval.dedupe_evidence(rows, valid_source_chunk_pairs=None)>

In [8]:
import inspect

from src.tools.draft_writing.retrieval import (
    query_chroma_restricted,
    query_csv_restricted,
)

print("CHROMA")
print(inspect.signature(query_chroma_restricted))
print(inspect.getsource(query_chroma_restricted))

print("\nCSV")
print(inspect.signature(query_csv_restricted))
print(inspect.getsource(query_csv_restricted))

CHROMA
(collection, chunks_df, query, source_filenames, top_k, max_evidence_chars=18000, valid_source_chunk_pairs=None)
def query_chroma_restricted(
    collection,
    chunks_df,
    query,
    source_filenames,
    top_k,
    max_evidence_chars=18000,
    valid_source_chunk_pairs=None,
):
    if not source_filenames:
        return []

    per_source_k = max(1, top_k // len(source_filenames) + 1)
    rows = []

    for source in source_filenames:
        source_chunk_count = int(
            (chunks_df["source_filename"].astype(str) == source).sum()
        )
        result = collection.query(
            query_texts=[query],
            n_results=min(per_source_k, max(1, source_chunk_count)),
            where={"source_filename": source},
            include=["documents", "metadatas", "distances"],
        )

        documents = (result.get("documents") or [[]])[0]
        metadatas = (result.get("metadatas") or [[]])[0]
        distances = (result.get("distances") or [[]])[0]

    

In [10]:
from pathlib import Path

path = Path(
    "/content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py"
)

code = r'''
from __future__ import annotations

from collections import defaultdict
from typing import Any


def _candidate_key(row: dict[str, Any]) -> tuple[str, str]:
    return (
        str(row.get("source_filename", "")).strip(),
        str(row.get("chunk_id", "")).strip(),
    )


def reciprocal_rank_fusion(
    chroma_rows: list[dict[str, Any]],
    csv_rows: list[dict[str, Any]],
    *,
    rrf_k: int = 60,
) -> list[dict[str, Any]]:
    if rrf_k <= 0:
        raise ValueError("rrf_k debe ser mayor que cero")

    fused: dict[tuple[str, str], dict[str, Any]] = {}
    fused_scores: defaultdict[tuple[str, str], float] = defaultdict(float)

    rankings = (
        ("chroma_restricted", chroma_rows),
        ("csv_lexical_restricted", csv_rows),
    )

    for method, rows in rankings:
        for rank, original in enumerate(rows, start=1):
            row = dict(original)
            key = _candidate_key(row)

            if not key[0] or not key[1]:
                continue

            fused_scores[key] += 1.0 / (rrf_k + rank)

            if key not in fused:
                fused[key] = row
                fused[key]["retrieval_methods"] = []
                fused[key]["component_ranks"] = {}
                fused[key]["component_scores"] = {}

            item = fused[key]

            if method not in item["retrieval_methods"]:
                item["retrieval_methods"].append(method)

            item["component_ranks"][method] = rank
            item["component_scores"][method] = float(
                row.get("score", 0.0) or 0.0
            )

    output = []

    for key, item in fused.items():
        row = dict(item)
        row["hybrid_score"] = fused_scores[key]
        row["score"] = fused_scores[key]
        row["retrieval_method"] = "hybrid_rrf"
        output.append(row)

    output.sort(
        key=lambda row: (
            -float(row["hybrid_score"]),
            str(row.get("source_filename", "")),
            str(row.get("chunk_id", "")),
        )
    )

    return output
'''

path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(code, encoding="utf-8")

print("Creado:", path)
print("Existe:", path.exists())

Creado: /content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py
Existe: True


In [11]:
from pathlib import Path

path = Path(
    "/content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py"
)

code = r'''
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path


TARGET_CHUNKS = {
    "696fb1df0f31a0b4_chunk_0003",
    "42891eb1891ec233_chunk_0003",
}


def safe_str(value):
    return "" if value is None else str(value).strip()


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--project-dir", required=True)
    parser.add_argument("--code-root", default="/content/tesis_codigo")
    parser.add_argument("--section-id", default="S2")
    parser.add_argument("--output-dir")
    args = parser.parse_args()

    code_root = Path(args.code_root).resolve()
    if str(code_root) not in sys.path:
        sys.path.insert(0, str(code_root))

    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
    )
    from src.tools.draft_writing.retrieval import (
        query_chroma_restricted,
        query_csv_restricted,
        dedupe_evidence,
    )
    from src.adapters.draft_writing_runtime import (
        build_chroma_collection,
        build_draft_agent_input,
        load_draft_configuration,
    )
    from src.tools.draft_writing import (
        build_section_query,
        validate_draft_dependencies,
    )

    project_dir = Path(args.project_dir).resolve()

    cfg = load_draft_configuration(
        project_dir,
        attempt_number=1,
    )

    bundle = validate_draft_dependencies(
        build_draft_agent_input(cfg)
    )

    collection = build_chroma_collection(cfg)

    section = next(
        (
            item
            for item in bundle["outline"].get("sections", [])
            if safe_str(item.get("section_id")) == args.section_id
        ),
        None,
    )

    if section is None:
        raise SystemExit(f"SECTION_NOT_FOUND:{args.section_id}")

    source_filenames = [
        safe_str(
            paper.get("source_filename")
            if isinstance(paper, dict)
            else paper
        )
        for paper in section.get("papers_to_use", [])
    ]
    source_filenames = [
        source for source in source_filenames if source
    ]

    query = build_section_query(section)

    top_k = int(
        cfg["policy"].get(
            "top_k_evidence_per_section",
            8,
        )
    )

    max_chars = int(
        cfg["policy"].get(
            "max_evidence_chars",
            18000,
        )
    )

    valid_pairs = {
        (
            safe_str(row["source_filename"]),
            safe_str(row["chunk_id"]),
        )
        for _, row in bundle["chunks"].iterrows()
    }

    chroma_rows = query_chroma_restricted(
        collection,
        bundle["chunks"],
        query,
        source_filenames,
        top_k,
        max_evidence_chars=max_chars,
        valid_source_chunk_pairs=valid_pairs,
    )

    csv_rows = query_csv_restricted(
        bundle["chunks"],
        query,
        source_filenames,
        top_k,
        max_evidence_chars=max_chars,
        valid_source_chunk_pairs=valid_pairs,
    )

    fused_rows = reciprocal_rank_fusion(
        chroma_rows,
        csv_rows,
        rrf_k=60,
    )

    final_rows = dedupe_evidence(
        fused_rows,
        valid_pairs,
    )[:top_k]

    final_chunk_ids = {
        safe_str(row.get("chunk_id"))
        for row in final_rows
    }

    target_presence = {
        chunk_id: chunk_id in final_chunk_ids
        for chunk_id in sorted(TARGET_CHUNKS)
    }

    output_dir = (
        Path(args.output_dir).resolve()
        if args.output_dir
        else Path(cfg["output_dir"])
        / "diagnostic_hybrid_retrieval"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    payload = {
        "status": "DIAGNOSTIC_HYBRID_RETRIEVAL_COMPLETED",
        "diagnostic_only": True,
        "openai_called": False,
        "pipeline_state_modified": False,
        "contractual_attempt_created": False,
        "section_id": args.section_id,
        "top_k": top_k,
        "query": query,
        "chroma_candidates": chroma_rows,
        "csv_candidates": csv_rows,
        "fused_candidates": fused_rows,
        "final_top_k": final_rows,
        "target_chunk_presence": target_presence,
    }

    output_file = (
        output_dir
        / f"{args.section_id}_hybrid_retrieval.json"
    )

    output_file.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print(
        json.dumps(
            {
                "status": payload["status"],
                "output_file": str(output_file),
                "final_top_k_count": len(final_rows),
                "target_chunk_presence": target_presence,
                "pipeline_state_modified": False,
                "openai_called": False,
            },
            indent=2,
            ensure_ascii=False,
        )
    )

    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

path.write_text(code, encoding="utf-8")

print("Creado:", path)
print("Existe:", path.exists())

Creado: /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py
Existe: True


In [12]:
!python /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py \
  --project-dir /content/proyecto_estado_arte \
  --code-root /content/tesis_codigo \
  --section-id S2 \
  --output-dir /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/hybrid_retrieval

2026-07-20 17:12:28.453738: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 17:12:28.469644: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784567548.485919 3780128 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784567548.490930 3780128 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-20 17:12:28.509870: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [13]:
from pathlib import Path
import json
import re

path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/hybrid_retrieval/"
    "S2_hybrid_retrieval.json"
)

data = json.loads(path.read_text(encoding="utf-8"))

target_values = ["0.96", "1.34", "10%", "58.7%", "6.11%", "99%"]

print("TOP-K HÍBRIDO\n")

for rank, row in enumerate(data["final_top_k"], start=1):
    text = str(row.get("text", ""))
    found = [value for value in target_values if value in text]

    print(
        f"Rank {rank}:",
        row.get("chunk_id"),
        "| métodos:", row.get("retrieval_methods"),
        "| valores:", found,
    )

print("\nCOBERTURA POR VALOR\n")

for value in target_values:
    matching = [
        row
        for row in data["final_top_k"]
        if value in str(row.get("text", ""))
    ]

    print(
        value,
        "→",
        "CUBIERTO" if matching else "NO CUBIERTO",
        [row.get("chunk_id") for row in matching],
    )

TOP-K HÍBRIDO

Rank 1: 12ed2391bde9cd16_chunk_0005 | métodos: ['csv_lexical_restricted'] | valores: []
Rank 2: 696fb1df0f31a0b4_chunk_0015 | métodos: ['chroma_restricted'] | valores: []
Rank 3: 696fb1df0f31a0b4_chunk_0002 | métodos: ['csv_lexical_restricted'] | valores: []
Rank 4: 696fb1df0f31a0b4_chunk_0014 | métodos: ['chroma_restricted'] | valores: []
Rank 5: 42891eb1891ec233_chunk_0008 | métodos: ['chroma_restricted'] | valores: []
Rank 6: 696fb1df0f31a0b4_chunk_0003 | métodos: ['csv_lexical_restricted'] | valores: ['10%', '58.7%', '6.11%']
Rank 7: 42891eb1891ec233_chunk_0002 | métodos: ['csv_lexical_restricted'] | valores: []
Rank 8: 696fb1df0f31a0b4_chunk_0016 | métodos: ['chroma_restricted'] | valores: []

COBERTURA POR VALOR

0.96 → NO CUBIERTO []
1.34 → NO CUBIERTO []
10% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
58.7% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
6.11% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
99% → NO CUBIERTO []


In [14]:
from pathlib import Path
import json

report_path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/hybrid_retrieval/"
    "S2_hybrid_retrieval.json"
)

data = json.loads(report_path.read_text(encoding="utf-8"))

target_values = ["0.96", "1.34", "10%", "58.7%", "6.11%", "99%"]


def print_value_positions(name, rows):
    print(f"\n{name}")
    print("=" * len(name))

    for value in target_values:
        matches = []

        for rank, row in enumerate(rows, start=1):
            text = str(row.get("text", ""))

            if value in text:
                matches.append(
                    {
                        "rank": rank,
                        "chunk_id": row.get("chunk_id"),
                        "source": row.get("source_filename"),
                        "score": row.get("score"),
                        "methods": row.get(
                            "retrieval_methods",
                            [row.get("retrieval_method")],
                        ),
                    }
                )

        print(
            value,
            "→",
            matches if matches else "NO APARECE",
        )


print_value_positions(
    "CANDIDATOS CHROMA",
    data["chroma_candidates"],
)

print_value_positions(
    "CANDIDATOS CSV",
    data["csv_candidates"],
)

print_value_positions(
    "CANDIDATOS FUSIONADOS",
    data["fused_candidates"],
)

print_value_positions(
    "TOP-K FINAL",
    data["final_top_k"],
)


CANDIDATOS CHROMA
0.96 → NO APARECE
1.34 → [{'rank': 6, 'chunk_id': '42891eb1891ec233_chunk_0026', 'source': '1/33.An-LM-BP-Neural-Network-Approach-to-Estimate-Monthly-Mean-Daily-Global-Solar-Radiation-Using-MODIS-Atmospheric-Products.pdf', 'score': 0.5180349946022034, 'methods': ['chroma_restricted']}]
10% → NO APARECE
58.7% → NO APARECE
6.11% → NO APARECE
99% → NO APARECE

CANDIDATOS CSV
0.96 → NO APARECE
1.34 → [{'rank': 5, 'chunk_id': '42891eb1891ec233_chunk_0003', 'source': '1/33.An-LM-BP-Neural-Network-Approach-to-Estimate-Monthly-Mean-Daily-Global-Solar-Radiation-Using-MODIS-Atmospheric-Products.pdf', 'score': 0.056338028169014086, 'methods': ['csv_lexical_restricted']}]
10% → [{'rank': 3, 'chunk_id': '696fb1df0f31a0b4_chunk_0003', 'source': '1/39.MLP_Back_Propagation_Artificial_Neural_Network_for.pdf', 'score': 0.07042253521126761, 'methods': ['csv_lexical_restricted']}]
58.7% → [{'rank': 3, 'chunk_id': '696fb1df0f31a0b4_chunk_0003', 'source': '1/39.MLP_Back_Propagation_Artifi

In [15]:
from pathlib import Path

runner_path = Path(
    "/content/tesis_codigo/"
    "run_agent06_diagnostic_hybrid_retrieval.py"
)

code = runner_path.read_text(encoding="utf-8")

old = '''
    top_k = int(
        cfg["policy"].get(
            "top_k_evidence_per_section",
            8,
        )
    )
'''

new = '''
    final_top_k = int(
        cfg["policy"].get(
            "top_k_evidence_per_section",
            8,
        )
    )

    candidate_multiplier = 3
    candidate_k = final_top_k * candidate_multiplier
'''

assert old in code, "No encontré el bloque original de top_k"

code = code.replace(old, new)

code = code.replace(
    '''
        top_k,
        max_evidence_chars=max_chars,
''',
    '''
        candidate_k,
        max_evidence_chars=max_chars,
'''
)

code = code.replace(
    '''
    )[:top_k]
''',
    '''
    )[:final_top_k]
'''
)

code = code.replace(
    '''
        "top_k": top_k,
''',
    '''
        "candidate_k_per_retriever": candidate_k,
        "final_top_k": final_top_k,
        "candidate_multiplier": candidate_multiplier,
'''
)

runner_path.write_text(code, encoding="utf-8")

print("Runner actualizado:", runner_path)

Runner actualizado: /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py


In [16]:
text = runner_path.read_text(encoding="utf-8")

for fragment in [
    "final_top_k",
    "candidate_multiplier = 3",
    "candidate_k = final_top_k * candidate_multiplier",
    "candidate_k_per_retriever",
]:
    print(fragment, "→", fragment in text)

final_top_k → True
candidate_multiplier = 3 → True
candidate_k = final_top_k * candidate_multiplier → True
candidate_k_per_retriever → True


In [17]:
!python /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py \
  --project-dir /content/proyecto_estado_arte \
  --code-root /content/tesis_codigo \
  --section-id S2 \
  --output-dir /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/hybrid_retrieval_candidate24

2026-07-20 17:21:10.732996: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 17:21:10.748001: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784568070.762905 3850050 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784568070.767754 3850050 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-20 17:21:10.785242: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [18]:
from pathlib import Path
import json

report_path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/"
    "hybrid_retrieval_candidate24/"
    "S2_hybrid_retrieval.json"
)

data = json.loads(report_path.read_text(encoding="utf-8"))

target_values = [
    "0.96",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]

print("CONFIGURACIÓN")
print(
    "Candidatos por recuperador:",
    data.get("candidate_k_per_retriever"),
)
print(
    "Top-k final:",
    data.get("final_top_k"),
)

print("\nTOP-K HÍBRIDO\n")

for rank, row in enumerate(data["final_top_k"], start=1):
    text = str(row.get("text", ""))
    found = [
        value
        for value in target_values
        if value in text
    ]

    print(
        f"Rank {rank}:",
        row.get("chunk_id"),
        "| métodos:",
        row.get("retrieval_methods"),
        "| valores:",
        found,
    )

print("\nCOBERTURA POR VALOR\n")

for value in target_values:
    matching = [
        row
        for row in data["final_top_k"]
        if value in str(row.get("text", ""))
    ]

    print(
        value,
        "→",
        "CUBIERTO" if matching else "NO CUBIERTO",
        [row.get("chunk_id") for row in matching],
    )

CONFIGURACIÓN
Candidatos por recuperador: 24
Top-k final: [{'source_filename': '1/33.An-LM-BP-Neural-Network-Approach-to-Estimate-Monthly-Mean-Daily-Global-Solar-Radiation-Using-MODIS-Atmospheric-Products.pdf', 'chunk_id': '42891eb1891ec233_chunk_0002', 'text': 'its development and utilization are being integrated into people’s lives. Therefore, accurate solar radiation data are of great signiﬁcance for site-selection of photovoltaic (PV) power generation, design of solar furnaces and energy-efﬁcient buildings. Practically, it is challenging to get accurate solar radiation data because of scarce and uneven distribution of ground-based observation sites throughout the country. Many artiﬁcial neural network (ANN) estimation models are therefore developed to estimate solar radiation, but the existing ANN models are mostly based on conventional meteorological data; clouds, aerosols, and water vapor are rarely considered because of a lack of instrumental observations at the conventional met

In [19]:
from pathlib import Path

path = Path(
    "/content/tesis_codigo/src/tools/draft_writing/"
    "hybrid_retrieval.py"
)

code = path.read_text(encoding="utf-8")

addition = r'''


def query_csv_ranked_restricted(
    chunks_df,
    query,
    source_filenames,
    top_k,
    *,
    max_evidence_chars=18000,
    valid_source_chunk_pairs=None,
):
    """
    Variante experimental del retrieval CSV.

    A diferencia de la implementación behavior-preserving:
    1. calcula el solapamiento léxico para todos los chunks permitidos;
    2. ordena por score descendente;
    3. deduplica;
    4. aplica top_k.
    """
    from src.tools.draft_writing.retrieval import (
        dedupe_evidence,
        safe_str,
        tokenize_for_overlap,
    )

    if not source_filenames or top_k <= 0:
        return []

    query_tokens = tokenize_for_overlap(query)

    subset = chunks_df[
        chunks_df["source_filename"]
        .astype(str)
        .isin(source_filenames)
    ]

    rows = []

    for _, source_row in subset.iterrows():
        text = safe_str(source_row["text"])
        text_tokens = tokenize_for_overlap(text)

        overlap = len(query_tokens & text_tokens)
        score = overlap / max(len(query_tokens), 1)

        rows.append(
            {
                "source_filename": safe_str(
                    source_row["source_filename"]
                ),
                "chunk_id": safe_str(
                    source_row["chunk_id"]
                ),
                "text": text[:max_evidence_chars],
                "score": float(score),
                "retrieval_method": (
                    "csv_lexical_ranked_experimental"
                ),
            }
        )

    rows.sort(
        key=lambda row: (
            -float(row.get("score", 0.0)),
            str(row.get("source_filename", "")),
            str(row.get("chunk_id", "")),
        )
    )

    return dedupe_evidence(
        rows,
        valid_source_chunk_pairs,
    )[:top_k]
'''

if "def query_csv_ranked_restricted(" not in code:
    code += addition
    path.write_text(code, encoding="utf-8")
    print("Función experimental agregada:", path)
else:
    print("La función ya existía:", path)

Función experimental agregada: /content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py


In [20]:
from pathlib import Path

runner_path = Path(
    "/content/tesis_codigo/"
    "run_agent06_diagnostic_hybrid_retrieval.py"
)

code = runner_path.read_text(encoding="utf-8")

old_import = '''    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
    )
'''

new_import = '''    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
        query_csv_ranked_restricted,
    )
'''

assert old_import in code, (
    "No encontré el import esperado del módulo híbrido"
)

code = code.replace(old_import, new_import)

old_call = '''    csv_rows = query_csv_restricted(
        bundle["chunks"],
        query,
        source_filenames,
        candidate_k,
        max_evidence_chars=max_chars,
        valid_source_chunk_pairs=valid_pairs,
    )
'''

new_call = '''    csv_rows = query_csv_ranked_restricted(
        bundle["chunks"],
        query,
        source_filenames,
        candidate_k,
        max_evidence_chars=max_chars,
        valid_source_chunk_pairs=valid_pairs,
    )
'''

assert old_call in code, (
    "No encontré la llamada original a query_csv_restricted"
)

code = code.replace(old_call, new_call)

runner_path.write_text(code, encoding="utf-8")

print("Runner actualizado:", runner_path)

Runner actualizado: /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py


In [21]:
!python /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py \
  --project-dir /content/proyecto_estado_arte \
  --code-root /content/tesis_codigo \
  --section-id S2 \
  --output-dir /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/hybrid_ranked_csv_candidate24

2026-07-20 17:24:00.192362: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 17:24:00.208123: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784568240.223636 3874147 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784568240.228665 3874147 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-20 17:24:00.246302: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [22]:
from pathlib import Path
import json

report_path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/"
    "hybrid_ranked_csv_candidate24/"
    "S2_hybrid_retrieval.json"
)

data = json.loads(
    report_path.read_text(encoding="utf-8")
)

target_values = [
    "0.96",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]

print("TOP-K HÍBRIDO CON CSV ORDENADO\n")

for rank, row in enumerate(data["final_top_k"], start=1):
    text = str(row.get("text", ""))

    values = [
        value
        for value in target_values
        if value in text
    ]

    print(
        f"Rank {rank}:",
        row.get("chunk_id"),
        "| métodos:",
        row.get("retrieval_methods"),
        "| valores:",
        values,
    )

print("\nCOBERTURA\n")

for value in target_values:
    matches = [
        row.get("chunk_id")
        for row in data["final_top_k"]
        if value in str(row.get("text", ""))
    ]

    print(
        value,
        "→",
        "CUBIERTO" if matches else "NO CUBIERTO",
        matches,
    )

TOP-K HÍBRIDO CON CSV ORDENADO

Rank 1: 42891eb1891ec233_chunk_0002 | métodos: ['chroma_restricted', 'csv_lexical_restricted'] | valores: []
Rank 2: 12ed2391bde9cd16_chunk_0005 | métodos: ['chroma_restricted', 'csv_lexical_restricted'] | valores: []
Rank 3: 42891eb1891ec233_chunk_0026 | métodos: ['chroma_restricted', 'csv_lexical_restricted'] | valores: ['1.34']
Rank 4: 696fb1df0f31a0b4_chunk_0008 | métodos: ['chroma_restricted', 'csv_lexical_restricted'] | valores: []
Rank 5: 696fb1df0f31a0b4_chunk_0015 | métodos: ['chroma_restricted'] | valores: []
Rank 6: 696fb1df0f31a0b4_chunk_0002 | métodos: ['csv_lexical_restricted'] | valores: []
Rank 7: 696fb1df0f31a0b4_chunk_0014 | métodos: ['chroma_restricted'] | valores: []
Rank 8: 42891eb1891ec233_chunk_0008 | métodos: ['chroma_restricted'] | valores: []

COBERTURA

0.96 → NO CUBIERTO []
1.34 → CUBIERTO ['42891eb1891ec233_chunk_0026']
10% → NO CUBIERTO []
58.7% → NO CUBIERTO []
6.11% → NO CUBIERTO []
99% → NO CUBIERTO []


In [23]:
from pathlib import Path
import json

runner_path = Path(
    "/content/tesis_codigo/"
    "run_agent06_diagnostic_hybrid_retrieval.py"
)

report_path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/"
    "hybrid_ranked_csv_candidate24/"
    "S2_hybrid_retrieval.json"
)

runner_text = runner_path.read_text(encoding="utf-8")
data = json.loads(report_path.read_text(encoding="utf-8"))

print("VERIFICACIÓN DEL RUNNER")
print("=======================")
print(
    "Importa query_csv_ranked_restricted:",
    "query_csv_ranked_restricted" in runner_text,
)

print(
    "Llama query_csv_ranked_restricted:",
    "csv_rows = query_csv_ranked_restricted(" in runner_text,
)

print(
    "Todavía llama query_csv_restricted:",
    "csv_rows = query_csv_restricted(" in runner_text,
)

targets = {
    "42891eb1891ec233_chunk_0003",
    "42891eb1891ec233_chunk_0026",
    "696fb1df0f31a0b4_chunk_0003",
}

for collection_name in [
    "chroma_candidates",
    "csv_candidates",
    "fused_candidates",
    "final_top_k",
]:
    print(f"\n{collection_name.upper()}")
    print("=" * len(collection_name))

    rows = data[collection_name]

    for rank, row in enumerate(rows, start=1):
        chunk_id = str(row.get("chunk_id", ""))

        if chunk_id in targets:
            print(
                "rank:", rank,
                "| chunk:", chunk_id,
                "| score:", row.get("score"),
                "| método original:",
                row.get("retrieval_method"),
                "| métodos fusionados:",
                row.get("retrieval_methods"),
            )

VERIFICACIÓN DEL RUNNER
Importa query_csv_ranked_restricted: True
Llama query_csv_ranked_restricted: True
Todavía llama query_csv_restricted: False

CHROMA_CANDIDATES
rank: 10 | chunk: 42891eb1891ec233_chunk_0026 | score: 0.5180349946022034 | método original: chroma_restricted | métodos fusionados: None

CSV_CANDIDATES
rank: 3 | chunk: 696fb1df0f31a0b4_chunk_0003 | score: 0.07042253521126761 | método original: csv_lexical_ranked_experimental | métodos fusionados: None
rank: 5 | chunk: 42891eb1891ec233_chunk_0003 | score: 0.056338028169014086 | método original: csv_lexical_ranked_experimental | métodos fusionados: None
rank: 13 | chunk: 42891eb1891ec233_chunk_0026 | score: 0.04225352112676056 | método original: csv_lexical_ranked_experimental | métodos fusionados: None

FUSED_CANDIDATES
rank: 3 | chunk: 42891eb1891ec233_chunk_0026 | score: 0.027984344422700584 | método original: hybrid_rrf | métodos fusionados: ['chroma_restricted', 'csv_lexical_restricted']
rank: 9 | chunk: 696fb1df0f3

In [24]:
from pathlib import Path

path = Path(
    "/content/tesis_codigo/src/tools/draft_writing/"
    "hybrid_retrieval.py"
)

code = path.read_text(encoding="utf-8")

addition = r'''


def quota_hybrid_selection(
    chroma_rows,
    csv_rows,
    *,
    final_top_k=8,
    chroma_quota=4,
    csv_quota=4,
    valid_source_chunk_pairs=None,
):
    """
    Selección híbrida experimental con cuotas mínimas.

    1. Selecciona los primeros resultados de Chroma.
    2. Selecciona los primeros resultados de CSV.
    3. Deduplica por fuente y chunk.
    4. Completa los espacios restantes con candidatos de ambos rankings.
    """

    from src.tools.draft_writing.retrieval import dedupe_evidence

    if final_top_k <= 0:
        return []

    selected = []

    selected.extend(chroma_rows[:chroma_quota])
    selected.extend(csv_rows[:csv_quota])

    selected = dedupe_evidence(
        selected,
        valid_source_chunk_pairs,
    )

    if len(selected) >= final_top_k:
        return selected[:final_top_k]

    selected_keys = {
        (
            str(row.get("source_filename", "")).strip(),
            str(row.get("chunk_id", "")).strip(),
        )
        for row in selected
    }

    remaining = []

    for method, rows in (
        ("chroma_restricted", chroma_rows),
        ("csv_lexical_ranked_experimental", csv_rows),
    ):
        for rank, original in enumerate(rows, start=1):
            row = dict(original)

            key = (
                str(row.get("source_filename", "")).strip(),
                str(row.get("chunk_id", "")).strip(),
            )

            if not key[0] or not key[1]:
                continue

            if key in selected_keys:
                continue

            row["hybrid_selection_method"] = "quota_hybrid"
            row["component_rank"] = rank
            row["component_method"] = method

            remaining.append(row)

    remaining.sort(
        key=lambda row: (
            int(row.get("component_rank", 999999)),
            str(row.get("component_method", "")),
            str(row.get("source_filename", "")),
            str(row.get("chunk_id", "")),
        )
    )

    for row in remaining:
        key = (
            str(row.get("source_filename", "")).strip(),
            str(row.get("chunk_id", "")).strip(),
        )

        if key in selected_keys:
            continue

        selected.append(row)
        selected_keys.add(key)

        if len(selected) >= final_top_k:
            break

    final_rows = dedupe_evidence(
        selected,
        valid_source_chunk_pairs,
    )

    for row in final_rows:
        row["retrieval_method"] = "hybrid_quota"
        row["hybrid_selection_method"] = "quota_hybrid"

    return final_rows[:final_top_k]
'''

if "def quota_hybrid_selection(" not in code:
    code += addition
    path.write_text(code, encoding="utf-8")
    print("Función agregada:", path)
else:
    print("La función ya existe")

Función agregada: /content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py


In [25]:
from pathlib import Path

runner_path = Path(
    "/content/tesis_codigo/"
    "run_agent06_diagnostic_hybrid_retrieval.py"
)

code = runner_path.read_text(encoding="utf-8")

old = '''    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
        query_csv_ranked_restricted,
    )
'''

new = '''    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
        query_csv_ranked_restricted,
        quota_hybrid_selection,
    )
'''

assert old in code, "No encontré el import esperado"

code = code.replace(old, new)

runner_path.write_text(code, encoding="utf-8")

print("Import actualizado")

Import actualizado


In [26]:
code = runner_path.read_text(encoding="utf-8")

old = '''    fused_rows = reciprocal_rank_fusion(
        chroma_rows,
        csv_rows,
        rrf_k=60,
    )

    final_rows = dedupe_evidence(
        fused_rows,
        valid_pairs,
    )[:final_top_k]
'''

new = '''    fused_rows = reciprocal_rank_fusion(
        chroma_rows,
        csv_rows,
        rrf_k=60,
    )

    final_rows = quota_hybrid_selection(
        chroma_rows,
        csv_rows,
        final_top_k=final_top_k,
        chroma_quota=4,
        csv_quota=4,
        valid_source_chunk_pairs=valid_pairs,
    )
'''

assert old in code, "No encontré el bloque RRF final"

code = code.replace(old, new)

runner_path.write_text(code, encoding="utf-8")

print("Selección final cambiada a quota_hybrid")

Selección final cambiada a quota_hybrid


In [27]:
code = runner_path.read_text(encoding="utf-8")

old = '''        "candidate_multiplier": candidate_multiplier,
'''

new = '''        "candidate_multiplier": candidate_multiplier,
        "final_selection_method": "quota_hybrid",
        "chroma_quota": 4,
        "csv_quota": 4,
'''

assert old in code, "No encontré candidate_multiplier en el payload"

code = code.replace(old, new)

runner_path.write_text(code, encoding="utf-8")

print("Metadatos experimentales agregados")

Metadatos experimentales agregados


In [28]:
!python /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py \
  --project-dir /content/proyecto_estado_arte \
  --code-root /content/tesis_codigo \
  --section-id S2 \
  --output-dir /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/hybrid_quota_4_4

2026-07-20 17:27:43.232981: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 17:27:43.248344: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784568463.264484 3905599 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784568463.269636 3905599 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-20 17:27:43.287686: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [29]:
from pathlib import Path
import json

report_path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/"
    "hybrid_quota_4_4/"
    "S2_hybrid_retrieval.json"
)

data = json.loads(report_path.read_text(encoding="utf-8"))

target_values = [
    "0.96",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]

print("TOP-K HÍBRIDO CON CUOTAS 4+4\n")

for rank, row in enumerate(data["final_top_k"], start=1):
    text = str(row.get("text", ""))

    values = [
        value
        for value in target_values
        if value in text
    ]

    print(
        f"Rank {rank}:",
        row.get("chunk_id"),
        "| método:",
        row.get("retrieval_method"),
        "| valores:",
        values,
    )

print("\nCOBERTURA\n")

for value in target_values:
    matches = [
        row.get("chunk_id")
        for row in data["final_top_k"]
        if value in str(row.get("text", ""))
    ]

    print(
        value,
        "→",
        "CUBIERTO" if matches else "NO CUBIERTO",
        matches,
    )

TOP-K HÍBRIDO CON CUOTAS 4+4

Rank 1: 696fb1df0f31a0b4_chunk_0015 | método: chroma_restricted | valores: []
Rank 2: 696fb1df0f31a0b4_chunk_0014 | método: chroma_restricted | valores: []
Rank 3: 42891eb1891ec233_chunk_0008 | método: chroma_restricted | valores: []
Rank 4: 696fb1df0f31a0b4_chunk_0016 | método: chroma_restricted | valores: []
Rank 5: 12ed2391bde9cd16_chunk_0005 | método: csv_lexical_ranked_experimental | valores: []
Rank 6: 696fb1df0f31a0b4_chunk_0002 | método: csv_lexical_ranked_experimental | valores: []
Rank 7: 696fb1df0f31a0b4_chunk_0003 | método: csv_lexical_ranked_experimental | valores: ['10%', '58.7%', '6.11%']
Rank 8: 42891eb1891ec233_chunk_0002 | método: csv_lexical_ranked_experimental | valores: []

COBERTURA

0.96 → NO CUBIERTO []
1.34 → NO CUBIERTO []
10% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
58.7% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
6.11% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
99% → NO CUBIERTO []


In [30]:
from pathlib import Path

path = Path(
    "/content/tesis_codigo/src/tools/draft_writing/"
    "hybrid_retrieval.py"
)

code = path.read_text(encoding="utf-8")

addition = r'''


def balanced_hybrid_selection(
    chroma_rows,
    csv_rows,
    fused_rows,
    *,
    final_top_k=8,
    chroma_quota=3,
    csv_quota=3,
    valid_source_chunk_pairs=None,
):
    """
    Selección híbrida balanceada:

    1. Reserva una cuota mínima para Chroma.
    2. Reserva una cuota mínima para CSV.
    3. Completa los lugares restantes mediante el ranking RRF.
    4. Deduplica por source_filename y chunk_id.
    """

    from src.tools.draft_writing.retrieval import dedupe_evidence

    if final_top_k <= 0:
        return []

    selected = []
    selected_keys = set()

    def add_rows(rows, limit, selection_source):
        added = 0

        for original in rows:
            if added >= limit:
                break

            row = dict(original)

            key = (
                str(row.get("source_filename", "")).strip(),
                str(row.get("chunk_id", "")).strip(),
            )

            if not key[0] or not key[1]:
                continue

            if key in selected_keys:
                continue

            row["hybrid_selection_method"] = "balanced_quota_rrf"
            row["selection_source"] = selection_source

            selected.append(row)
            selected_keys.add(key)
            added += 1

    # Cuotas iniciales
    add_rows(
        chroma_rows,
        chroma_quota,
        "chroma_quota",
    )

    add_rows(
        csv_rows,
        csv_quota,
        "csv_quota",
    )

    # Completar con el ranking fusionado
    remaining_slots = final_top_k - len(selected)

    if remaining_slots > 0:
        add_rows(
            fused_rows,
            remaining_slots,
            "rrf_completion",
        )

    final_rows = dedupe_evidence(
        selected,
        valid_source_chunk_pairs,
    )

    return final_rows[:final_top_k]
'''

if "def balanced_hybrid_selection(" not in code:
    code += addition
    path.write_text(code, encoding="utf-8")
    print("Función agregada:", path)
else:
    print("La función ya existe")

Función agregada: /content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py


In [31]:
from pathlib import Path

runner_path = Path(
    "/content/tesis_codigo/"
    "run_agent06_diagnostic_hybrid_retrieval.py"
)

code = runner_path.read_text(encoding="utf-8")

old = '''    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
        query_csv_ranked_restricted,
        quota_hybrid_selection,
    )
'''

new = '''    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
        query_csv_ranked_restricted,
        balanced_hybrid_selection,
    )
'''

assert old in code, "No encontré el import actual"

code = code.replace(old, new)

runner_path.write_text(code, encoding="utf-8")

print("Import actualizado")

Import actualizado


In [32]:
code = runner_path.read_text(encoding="utf-8")

old = '''    final_rows = quota_hybrid_selection(
        chroma_rows,
        csv_rows,
        final_top_k=final_top_k,
        chroma_quota=4,
        csv_quota=4,
        valid_source_chunk_pairs=valid_pairs,
    )
'''

new = '''    final_rows = balanced_hybrid_selection(
        chroma_rows,
        csv_rows,
        fused_rows,
        final_top_k=final_top_k,
        chroma_quota=3,
        csv_quota=3,
        valid_source_chunk_pairs=valid_pairs,
    )
'''

assert old in code, "No encontré el bloque quota_hybrid"

code = code.replace(old, new)

code = code.replace(
    '"final_selection_method": "quota_hybrid",',
    '"final_selection_method": "balanced_quota_rrf",',
)

code = code.replace(
    '"chroma_quota": 4,',
    '"chroma_quota": 3,',
)

code = code.replace(
    '"csv_quota": 4,',
    '"csv_quota": 3,',
)

runner_path.write_text(code, encoding="utf-8")

print("Selección cambiada a 3+3+RRF")

Selección cambiada a 3+3+RRF


In [33]:
!python /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py \
  --project-dir /content/proyecto_estado_arte \
  --code-root /content/tesis_codigo \
  --section-id S2 \
  --output-dir /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/hybrid_balanced_3_3_rrf

2026-07-20 17:29:39.155955: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 17:29:39.170804: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784568579.186041 3921070 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784568579.190877 3921070 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-20 17:29:39.208165: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [34]:
from pathlib import Path
import json

report_path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/"
    "hybrid_balanced_3_3_rrf/"
    "S2_hybrid_retrieval.json"
)

data = json.loads(
    report_path.read_text(encoding="utf-8")
)

target_values = [
    "0.96",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]

print("TOP-K BALANCEADO 3+3+RRF\n")

for rank, row in enumerate(data["final_top_k"], start=1):
    text = str(row.get("text", ""))

    values = [
        value
        for value in target_values
        if value in text
    ]

    print(
        f"Rank {rank}:",
        row.get("chunk_id"),
        "| selección:",
        row.get("selection_source"),
        "| valores:",
        values,
    )

print("\nCOBERTURA\n")

for value in target_values:
    matches = [
        row.get("chunk_id")
        for row in data["final_top_k"]
        if value in str(row.get("text", ""))
    ]

    print(
        value,
        "→",
        "CUBIERTO" if matches else "NO CUBIERTO",
        matches,
    )

TOP-K BALANCEADO 3+3+RRF

Rank 1: 696fb1df0f31a0b4_chunk_0015 | selección: chroma_quota | valores: []
Rank 2: 696fb1df0f31a0b4_chunk_0014 | selección: chroma_quota | valores: []
Rank 3: 42891eb1891ec233_chunk_0008 | selección: chroma_quota | valores: []
Rank 4: 12ed2391bde9cd16_chunk_0005 | selección: csv_quota | valores: []
Rank 5: 696fb1df0f31a0b4_chunk_0002 | selección: csv_quota | valores: []
Rank 6: 696fb1df0f31a0b4_chunk_0003 | selección: csv_quota | valores: ['10%', '58.7%', '6.11%']
Rank 7: 42891eb1891ec233_chunk_0002 | selección: rrf_completion | valores: []
Rank 8: 42891eb1891ec233_chunk_0026 | selección: rrf_completion | valores: ['1.34']

COBERTURA

0.96 → NO CUBIERTO []
1.34 → CUBIERTO ['42891eb1891ec233_chunk_0026']
10% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
58.7% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
6.11% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
99% → NO CUBIERTO []


In [35]:
from pathlib import Path
import json
import pandas as pd

report_path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/"
    "hybrid_balanced_3_3_rrf/"
    "S2_hybrid_retrieval.json"
)

data = json.loads(report_path.read_text(encoding="utf-8"))

# La información completa de chunks ya está disponible en los artefactos
# que usa el runner. Volvemos a cargarla mediante el runtime existente.

from src.adapters.draft_writing_runtime import (
    build_draft_agent_input,
    load_draft_configuration,
)

from src.tools.draft_writing import validate_draft_dependencies

project_dir = Path("/content/proyecto_estado_arte")

cfg = load_draft_configuration(
    project_dir,
    attempt_number=1,
)

bundle = validate_draft_dependencies(
    build_draft_agent_input(cfg)
)

chunks_df = bundle["chunks"].copy()

targets = ["0.96", "99%"]

for value in targets:
    matches = chunks_df[
        chunks_df["text"]
        .astype(str)
        .str.contains(value, regex=False, na=False)
    ]

    print(f"\nVALOR: {value}")
    print("=" * 60)
    print("Coincidencias:", len(matches))

    for _, row in matches.iterrows():
        print(
            "source_filename:",
            row.get("source_filename"),
        )
        print(
            "chunk_id:",
            row.get("chunk_id"),
        )
        print(
            "texto:",
            str(row.get("text", ""))[:700],
        )
        print("-" * 60)


VALOR: 0.96
Coincidencias: 51
source_filename: 1/33.An-LM-BP-Neural-Network-Approach-to-Estimate-Monthly-Mean-Daily-Global-Solar-Radiation-Using-MODIS-Atmospheric-Products.pdf
chunk_id: 42891eb1891ec233_chunk_0022
texto: a is a the constant, and 1 < a < 10. Different numbers of neurons in the hidden layer were tested
in order to select a relatively optimized network structure. After training, a comparison was
performed between simulated M-GSR and observed ones at the training sites. R, RMSE and MBE
were used as error metrics for comparison; these statistics are deﬁned as follows:
R =
∑N
i=1(xi −x)(yi −y)
q
∑N
i=1(xi −x)2q
∑N
i=1(yi −y)2
(4)
RMSE =
q
∑
N
i=1 (H e−Hm)2/N
(5)
MBE =∑
N
i=1(H e−Hm)/N
(6)
where He the estimated M-GSR, and Hm the measured M-GSR, and N the number of samples. If
RMSE and MBE are smaller, the simulation precision is higher. The performance of the number of
neurons in hidden layer is s
------------------------------------------------------------
source_filename:

In [36]:
from pathlib import Path

path = Path(
    "/content/tesis_codigo/src/tools/draft_writing/"
    "hybrid_retrieval.py"
)

code = path.read_text(encoding="utf-8")

addition = r'''


def build_quantitative_query(base_query: str) -> str:
    """
    Construye una subconsulta general orientada a métricas,
    resultados experimentales y comparaciones cuantitativas.
    No utiliza valores concretos producidos por el LLM.
    """
    quantitative_terms = (
        "experimental results performance metrics accuracy "
        "prediction error comparison models "
        "RMSE MAE MBE MAPE NRMSE correlation coefficient "
        "R R2 percentage percent forecast results"
    )

    return f"{base_query} {quantitative_terms}".strip()
'''

if "def build_quantitative_query(" not in code:
    code += addition
    path.write_text(code, encoding="utf-8")
    print("Función agregada:", path)
else:
    print("La función ya existe")

Función agregada: /content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py


In [37]:
from pathlib import Path

runner_path = Path(
    "/content/tesis_codigo/"
    "run_agent06_diagnostic_hybrid_retrieval.py"
)

code = runner_path.read_text(encoding="utf-8")

old = '''    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
        query_csv_ranked_restricted,
        balanced_hybrid_selection,
    )
'''

new = '''    from src.tools.draft_writing.hybrid_retrieval import (
        reciprocal_rank_fusion,
        query_csv_ranked_restricted,
        balanced_hybrid_selection,
        build_quantitative_query,
    )
'''

assert old in code, "No encontré el import híbrido actual"

code = code.replace(old, new)
runner_path.write_text(code, encoding="utf-8")

print("Import actualizado")

Import actualizado


In [38]:
code = runner_path.read_text(encoding="utf-8")

old = '''    query = build_section_query(section)
'''

new = '''    query = build_section_query(section)
    quantitative_query = build_quantitative_query(query)
'''

assert old in code, "No encontré la creación de query"

code = code.replace(old, new)
runner_path.write_text(code, encoding="utf-8")

print("Subconsulta cuantitativa agregada")

Subconsulta cuantitativa agregada


In [39]:
code = runner_path.read_text(encoding="utf-8")

start = code.index("    chroma_rows = query_chroma_restricted(")
end = code.index("    fused_rows = reciprocal_rank_fusion(", start)

old = code[start:end]

new = '''    chroma_thematic_rows = query_chroma_restricted(
        collection,
        bundle["chunks"],
        query,
        source_filenames,
        candidate_k,
        max_evidence_chars=max_chars,
        valid_source_chunk_pairs=valid_pairs,
    )

    chroma_quantitative_rows = query_chroma_restricted(
        collection,
        bundle["chunks"],
        quantitative_query,
        source_filenames,
        candidate_k,
        max_evidence_chars=max_chars,
        valid_source_chunk_pairs=valid_pairs,
    )

    csv_thematic_rows = query_csv_ranked_restricted(
        bundle["chunks"],
        query,
        source_filenames,
        candidate_k,
        max_evidence_chars=max_chars,
        valid_source_chunk_pairs=valid_pairs,
    )

    csv_quantitative_rows = query_csv_ranked_restricted(
        bundle["chunks"],
        quantitative_query,
        source_filenames,
        candidate_k,
        max_evidence_chars=max_chars,
        valid_source_chunk_pairs=valid_pairs,
    )

    chroma_rows = dedupe_evidence(
        chroma_thematic_rows + chroma_quantitative_rows,
        valid_pairs,
    )[:candidate_k]

    csv_rows = dedupe_evidence(
        csv_thematic_rows + csv_quantitative_rows,
        valid_pairs,
    )[:candidate_k]

'''

code = code[:start] + new + code[end:]
runner_path.write_text(code, encoding="utf-8")

print("Retrieval temático + cuantitativo agregado")

Retrieval temático + cuantitativo agregado


In [40]:
code = runner_path.read_text(encoding="utf-8")

old = '''        "query": query,
'''

new = '''        "query": query,
        "quantitative_query": quantitative_query,
        "retrieval_queries": {
            "thematic": query,
            "quantitative": quantitative_query,
        },
'''

assert old in code, "No encontré query en el payload"

code = code.replace(old, new)
runner_path.write_text(code, encoding="utf-8")

print("Trazabilidad de consultas agregada")

Trazabilidad de consultas agregada


In [41]:
!python /content/tesis_codigo/run_agent06_diagnostic_hybrid_retrieval.py \
  --project-dir /content/proyecto_estado_arte \
  --code-root /content/tesis_codigo \
  --section-id S2 \
  --output-dir /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/hybrid_balanced_quantitative

2026-07-20 18:07:09.683489: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 18:07:09.697878: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784570829.713415   31504 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784570829.718223   31504 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-20 18:07:09.736407: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [42]:
from pathlib import Path
import json

report_path = Path(
    "/content/proyecto_estado_arte/experimento_paper_02/"
    "05_outputs/05_draft_experiments/"
    "hybrid_balanced_quantitative/"
    "S2_hybrid_retrieval.json"
)

data = json.loads(report_path.read_text(encoding="utf-8"))

target_values = [
    "0.96",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]

print("TOP-K TEMÁTICO + CUANTITATIVO\n")

for rank, row in enumerate(data["final_top_k"], start=1):
    text = str(row.get("text", ""))

    values = [
        value
        for value in target_values
        if value in text
    ]

    print(
        f"Rank {rank}:",
        row.get("chunk_id"),
        "| selección:",
        row.get("selection_source"),
        "| valores:",
        values,
    )

print("\nCOBERTURA\n")

for value in target_values:
    matches = [
        row.get("chunk_id")
        for row in data["final_top_k"]
        if value in str(row.get("text", ""))
    ]

    print(
        value,
        "→",
        "CUBIERTO" if matches else "NO CUBIERTO",
        matches,
    )

TOP-K TEMÁTICO + CUANTITATIVO

Rank 1: 696fb1df0f31a0b4_chunk_0015 | selección: chroma_quota | valores: []
Rank 2: 696fb1df0f31a0b4_chunk_0014 | selección: chroma_quota | valores: []
Rank 3: 42891eb1891ec233_chunk_0008 | selección: chroma_quota | valores: []
Rank 4: 12ed2391bde9cd16_chunk_0024 | selección: csv_quota | valores: []
Rank 5: 12ed2391bde9cd16_chunk_0026 | selección: csv_quota | valores: []
Rank 6: 12ed2391bde9cd16_chunk_0023 | selección: csv_quota | valores: []
Rank 7: 42891eb1891ec233_chunk_0030 | selección: rrf_completion | valores: ['0.96', '1.34']
Rank 8: 12ed2391bde9cd16_chunk_0005 | selección: rrf_completion | valores: []

COBERTURA

0.96 → CUBIERTO ['42891eb1891ec233_chunk_0030']
1.34 → CUBIERTO ['42891eb1891ec233_chunk_0030']
10% → NO CUBIERTO []
58.7% → NO CUBIERTO []
6.11% → NO CUBIERTO []
99% → NO CUBIERTO []


In [43]:
from pathlib import Path

path = Path(
    "/content/tesis_codigo/src/tools/draft_writing/"
    "hybrid_retrieval.py"
)

code = path.read_text(encoding="utf-8")

addition = r'''


def reciprocal_rank_fusion_many(
    rankings,
    *,
    rrf_k=60,
):
    """
    Fusiona múltiples rankings sin comparar directamente
    sus scores originales.

    rankings:
        iterable de pares (nombre_metodo, lista_de_rows)
    """
    from collections import defaultdict

    if rrf_k <= 0:
        raise ValueError("rrf_k debe ser mayor que cero")

    fused = {}
    fused_scores = defaultdict(float)

    for method_name, rows in rankings:
        for rank, original in enumerate(rows, start=1):
            row = dict(original)

            key = (
                str(row.get("source_filename", "")).strip(),
                str(row.get("chunk_id", "")).strip(),
            )

            if not key[0] or not key[1]:
                continue

            fused_scores[key] += 1.0 / (rrf_k + rank)

            if key not in fused:
                fused[key] = row
                fused[key]["retrieval_methods"] = []
                fused[key]["component_ranks"] = {}
                fused[key]["component_scores"] = {}

            item = fused[key]

            if method_name not in item["retrieval_methods"]:
                item["retrieval_methods"].append(method_name)

            item["component_ranks"][method_name] = rank
            item["component_scores"][method_name] = float(
                row.get("score", 0.0) or 0.0
            )

    output = []

    for key, item in fused.items():
        row = dict(item)
        row["score"] = fused_scores[key]
        row["hybrid_score"] = fused_scores[key]
        row["retrieval_method"] = "multi_query_rrf"
        output.append(row)

    output.sort(
        key=lambda row: (
            -float(row["hybrid_score"]),
            str(row.get("source_filename", "")),
            str(row.get("chunk_id", "")),
        )
    )

    return output
'''

if "def reciprocal_rank_fusion_many(" not in code:
    code += addition
    path.write_text(code, encoding="utf-8")
    print("Función agregada:", path)
else:
    print("La función ya existe")

Función agregada: /content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py


In [44]:
from pathlib import Path

runner_path = Path(
    "/content/tesis_codigo/"
    "run_agent06_diagnostic_hybrid_retrieval.py"
)

code = runner_path.read_text(encoding="utf-8")

old = '''        build_quantitative_query,
'''

new = '''        build_quantitative_query,
        reciprocal_rank_fusion_many,
'''

assert old in code, "No encontré build_quantitative_query en el import"

code = code.replace(old, new)

runner_path.write_text(code, encoding="utf-8")

print("Import actualizado")

Import actualizado


In [46]:
code = runner_path.read_text(encoding="utf-8")

old = '''    chroma_rows = dedupe_evidence(
        chroma_thematic_rows + chroma_quantitative_rows,
        valid_pairs,
    )[:candidate_k]

    csv_rows = dedupe_evidence(
        csv_thematic_rows + csv_quantitative_rows,
        valid_pairs,
    )[:candidate_k]
'''

new = '''    chroma_rows = reciprocal_rank_fusion_many(
        [
            ("chroma_thematic", chroma_thematic_rows),
            ("chroma_quantitative", chroma_quantitative_rows),
        ],
        rrf_k=60,
    )[:candidate_k]

    csv_rows = reciprocal_rank_fusion_many(
        [
            ("csv_thematic", csv_thematic_rows),
            ("csv_quantitative", csv_quantitative_rows),
        ],
        rrf_k=60,
    )[:candidate_k]
'''

assert old in code, "No encontré el bloque de concatenación actual"

code = code.replace(old, new)

runner_path.write_text(code, encoding="utf-8")

print("Concatenación sustituida por fusión multi-query")

Concatenación sustituida por fusión multi-query


In [49]:
import sys
import importlib
from pathlib import Path

# ============================================================
# 1. CONFIGURAR RUTA DEL REPOSITORIO
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()
MODULE_PATH = (
    CODE_ROOT
    / "src"
    / "tools"
    / "draft_writing"
    / "hybrid_retrieval.py"
)

assert CODE_ROOT.exists(), f"No existe: {CODE_ROOT}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

MODULE_PATH.parent.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. RECONSTRUIR hybrid_retrieval.py COMPLETO
# ============================================================

module_code = r'''
from __future__ import annotations

from collections import defaultdict
from typing import Any, Iterable


def _safe_str(value: Any) -> str:
    return "" if value is None else str(value).strip()


def _candidate_key(
    row: dict[str, Any],
) -> tuple[str, str]:
    return (
        _safe_str(row.get("source_filename")),
        _safe_str(row.get("chunk_id")),
    )


def build_quantitative_query(
    base_query: str,
) -> str:
    """
    Construye una subconsulta general orientada a resultados,
    métricas y comparaciones cuantitativas.

    No utiliza números concretos generados por el LLM.
    """
    quantitative_terms = (
        "experimental results performance metrics accuracy "
        "prediction error comparison models "
        "RMSE MAE MBE MAPE NRMSE correlation coefficient "
        "R R2 percentage percent forecast results"
    )

    return (
        f"{_safe_str(base_query)} {quantitative_terms}"
    ).strip()


def query_csv_ranked_restricted(
    chunks_df,
    query,
    source_filenames,
    top_k,
    max_evidence_chars=18000,
    valid_source_chunk_pairs=None,
):
    """
    Recuperación léxica experimental ordenada por score.

    Diferencia respecto del baseline:
    el baseline calcula score pero conserva el orden original
    del CSV. Esta variante ordena de mayor a menor relevancia.
    """
    from src.tools.draft_writing.retrieval import (
        dedupe_evidence,
        safe_str,
        tokenize_for_overlap,
    )

    if not source_filenames or top_k <= 0:
        return []

    query_tokens = tokenize_for_overlap(query)

    subset = chunks_df[
        chunks_df["source_filename"]
        .astype(str)
        .isin(source_filenames)
    ]

    rows = []

    for _, source_row in subset.iterrows():
        text = safe_str(source_row["text"])
        text_tokens = tokenize_for_overlap(text)

        overlap = len(query_tokens & text_tokens)
        score = overlap / max(len(query_tokens), 1)

        rows.append(
            {
                "source_filename": safe_str(
                    source_row["source_filename"]
                ),
                "chunk_id": safe_str(
                    source_row["chunk_id"]
                ),
                "text": text[:max_evidence_chars],
                "score": float(score),
                "retrieval_method": (
                    "csv_lexical_ranked_experimental"
                ),
            }
        )

    rows.sort(
        key=lambda row: (
            -float(row.get("score", 0.0)),
            _safe_str(row.get("source_filename")),
            _safe_str(row.get("chunk_id")),
        )
    )

    return dedupe_evidence(
        rows,
        valid_source_chunk_pairs,
    )[:top_k]


def reciprocal_rank_fusion(
    chroma_rows: list[dict[str, Any]],
    csv_rows: list[dict[str, Any]],
    *,
    rrf_k: int = 60,
) -> list[dict[str, Any]]:
    """
    Fusiona los rankings Chroma y CSV mediante RRF.
    """
    return reciprocal_rank_fusion_many(
        [
            ("chroma_restricted", chroma_rows),
            ("csv_lexical_restricted", csv_rows),
        ],
        rrf_k=rrf_k,
    )


def reciprocal_rank_fusion_many(
    rankings: Iterable[
        tuple[str, list[dict[str, Any]]]
    ],
    *,
    rrf_k: int = 60,
) -> list[dict[str, Any]]:
    """
    Fusiona múltiples rankings sin comparar directamente
    sus scores originales.
    """
    if rrf_k <= 0:
        raise ValueError(
            "rrf_k debe ser mayor que cero"
        )

    fused: dict[
        tuple[str, str],
        dict[str, Any],
    ] = {}

    fused_scores: defaultdict[
        tuple[str, str],
        float,
    ] = defaultdict(float)

    for method_name, rows in rankings:
        for rank, original in enumerate(
            rows,
            start=1,
        ):
            row = dict(original)
            key = _candidate_key(row)

            if not key[0] or not key[1]:
                continue

            fused_scores[key] += (
                1.0 / (rrf_k + rank)
            )

            if key not in fused:
                fused[key] = row
                fused[key]["retrieval_methods"] = []
                fused[key]["component_ranks"] = {}
                fused[key]["component_scores"] = {}

            item = fused[key]

            if (
                method_name
                not in item["retrieval_methods"]
            ):
                item["retrieval_methods"].append(
                    method_name
                )

            item["component_ranks"][
                method_name
            ] = rank

            item["component_scores"][
                method_name
            ] = float(
                row.get("score", 0.0) or 0.0
            )

    output = []

    for key, item in fused.items():
        row = dict(item)

        row["score"] = fused_scores[key]
        row["hybrid_score"] = fused_scores[key]
        row["retrieval_method"] = (
            "multi_query_rrf"
        )

        output.append(row)

    output.sort(
        key=lambda row: (
            -float(
                row.get("hybrid_score", 0.0)
            ),
            _safe_str(
                row.get("source_filename")
            ),
            _safe_str(
                row.get("chunk_id")
            ),
        )
    )

    return output


def balanced_hybrid_selection(
    chroma_rows,
    csv_rows,
    fused_rows,
    *,
    final_top_k=8,
    chroma_quota=3,
    csv_quota=3,
    valid_source_chunk_pairs=None,
):
    """
    Selección híbrida balanceada:

    1. reserva una cuota para Chroma;
    2. reserva una cuota para CSV;
    3. completa los lugares restantes con RRF;
    4. deduplica por fuente y chunk.
    """
    from src.tools.draft_writing.retrieval import (
        dedupe_evidence,
    )

    if final_top_k <= 0:
        return []

    if chroma_quota < 0 or csv_quota < 0:
        raise ValueError(
            "Las cuotas no pueden ser negativas"
        )

    selected = []
    selected_keys = set()

    def add_rows(
        rows,
        limit,
        selection_source,
    ):
        added = 0

        for original in rows:
            if added >= limit:
                break

            row = dict(original)
            key = _candidate_key(row)

            if not key[0] or not key[1]:
                continue

            if key in selected_keys:
                continue

            row["hybrid_selection_method"] = (
                "balanced_quota_rrf"
            )
            row["selection_source"] = (
                selection_source
            )

            selected.append(row)
            selected_keys.add(key)
            added += 1

    add_rows(
        chroma_rows,
        chroma_quota,
        "chroma_quota",
    )

    add_rows(
        csv_rows,
        csv_quota,
        "csv_quota",
    )

    remaining_slots = (
        final_top_k - len(selected)
    )

    if remaining_slots > 0:
        add_rows(
            fused_rows,
            remaining_slots,
            "rrf_completion",
        )

    final_rows = dedupe_evidence(
        selected,
        valid_source_chunk_pairs,
    )

    return final_rows[:final_top_k]


def quota_hybrid_selection(
    chroma_rows,
    csv_rows,
    *,
    final_top_k=8,
    chroma_quota=4,
    csv_quota=4,
    valid_source_chunk_pairs=None,
):
    """
    Variante sencilla con cuotas fijas.
    Se conserva para reproducir experimentos anteriores.
    """
    from src.tools.draft_writing.retrieval import (
        dedupe_evidence,
    )

    selected = (
        list(chroma_rows[:chroma_quota])
        + list(csv_rows[:csv_quota])
    )

    return dedupe_evidence(
        selected,
        valid_source_chunk_pairs,
    )[:final_top_k]
'''

MODULE_PATH.write_text(
    module_code,
    encoding="utf-8",
)

print("Módulo reconstruido:")
print(MODULE_PATH)

# ============================================================
# 3. LIMPIAR CACHÉ DE IMPORTACIÓN
# ============================================================

module_name = (
    "src.tools.draft_writing.hybrid_retrieval"
)

if module_name in sys.modules:
    del sys.modules[module_name]

importlib.invalidate_caches()

# ============================================================
# 4. VERIFICAR IMPORTACIONES
# ============================================================

from src.tools.draft_writing.hybrid_retrieval import (
    build_quantitative_query,
    query_csv_ranked_restricted,
    reciprocal_rank_fusion,
    reciprocal_rank_fusion_many,
    balanced_hybrid_selection,
    quota_hybrid_selection,
)

print("\nIMPORTACIONES CORRECTAS")

functions = [
    build_quantitative_query,
    query_csv_ranked_restricted,
    reciprocal_rank_fusion,
    reciprocal_rank_fusion_many,
    balanced_hybrid_selection,
    quota_hybrid_selection,
]

for function in functions:
    print("✓", function.__name__)

Módulo reconstruido:
/content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py

IMPORTACIONES CORRECTAS
✓ build_quantitative_query
✓ query_csv_ranked_restricted
✓ reciprocal_rank_fusion
✓ reciprocal_rank_fusion_many
✓ balanced_hybrid_selection
✓ quota_hybrid_selection


In [51]:
import sys
import json
import importlib
from pathlib import Path

# ============================================================
# 1. RUTAS
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()
PROJECT_DIR = Path("/content/proyecto_estado_arte").resolve()

SECTION_ID = "S2"

MODULE_PATH = (
    CODE_ROOT
    / "src"
    / "tools"
    / "draft_writing"
    / "hybrid_retrieval.py"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "experimento_paper_02"
    / "05_outputs"
    / "05_draft_experiments"
    / "hybrid_multiquery_rrf_complete"
)

assert CODE_ROOT.exists(), f"No existe CODE_ROOT: {CODE_ROOT}"
assert PROJECT_DIR.exists(), f"No existe PROJECT_DIR: {PROJECT_DIR}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

MODULE_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

# ============================================================
# 2. RECONSTRUIR hybrid_retrieval.py COMPLETO
# ============================================================

module_code = r'''
from __future__ import annotations

from collections import defaultdict
from typing import Any, Iterable


def _safe_str(value: Any) -> str:
    return "" if value is None else str(value).strip()


def _candidate_key(
    row: dict[str, Any],
) -> tuple[str, str]:
    return (
        _safe_str(row.get("source_filename")),
        _safe_str(row.get("chunk_id")),
    )


def build_quantitative_query(
    base_query: str,
) -> str:
    quantitative_terms = (
        "experimental results performance metrics accuracy "
        "prediction error comparison models "
        "RMSE MAE MBE MAPE NRMSE correlation coefficient "
        "R R2 percentage percent forecast results"
    )

    return (
        f"{_safe_str(base_query)} {quantitative_terms}"
    ).strip()


def query_csv_ranked_restricted(
    chunks_df,
    query,
    source_filenames,
    top_k,
    max_evidence_chars=18000,
    valid_source_chunk_pairs=None,
):
    from src.tools.draft_writing.retrieval import (
        dedupe_evidence,
        safe_str,
        tokenize_for_overlap,
    )

    if not source_filenames or top_k <= 0:
        return []

    query_tokens = tokenize_for_overlap(query)

    subset = chunks_df[
        chunks_df["source_filename"]
        .astype(str)
        .isin(source_filenames)
    ]

    rows = []

    for _, source_row in subset.iterrows():
        text = safe_str(source_row["text"])
        text_tokens = tokenize_for_overlap(text)

        overlap = len(query_tokens & text_tokens)
        score = overlap / max(len(query_tokens), 1)

        rows.append(
            {
                "source_filename": safe_str(
                    source_row["source_filename"]
                ),
                "chunk_id": safe_str(
                    source_row["chunk_id"]
                ),
                "text": text[:max_evidence_chars],
                "score": float(score),
                "retrieval_method": (
                    "csv_lexical_ranked_experimental"
                ),
            }
        )

    rows.sort(
        key=lambda row: (
            -float(row.get("score", 0.0)),
            _safe_str(row.get("source_filename")),
            _safe_str(row.get("chunk_id")),
        )
    )

    return dedupe_evidence(
        rows,
        valid_source_chunk_pairs,
    )[:top_k]


def reciprocal_rank_fusion_many(
    rankings: Iterable[
        tuple[str, list[dict[str, Any]]]
    ],
    *,
    rrf_k: int = 60,
) -> list[dict[str, Any]]:
    if rrf_k <= 0:
        raise ValueError("rrf_k debe ser mayor que cero")

    fused = {}
    fused_scores = defaultdict(float)

    for method_name, rows in rankings:
        for rank, original in enumerate(rows, start=1):
            row = dict(original)
            key = _candidate_key(row)

            if not key[0] or not key[1]:
                continue

            fused_scores[key] += 1.0 / (rrf_k + rank)

            if key not in fused:
                fused[key] = row
                fused[key]["retrieval_methods"] = []
                fused[key]["component_ranks"] = {}
                fused[key]["component_scores"] = {}

            item = fused[key]

            if method_name not in item["retrieval_methods"]:
                item["retrieval_methods"].append(method_name)

            item["component_ranks"][method_name] = rank

            item["component_scores"][method_name] = float(
                row.get("score", 0.0) or 0.0
            )

    output = []

    for key, item in fused.items():
        row = dict(item)
        row["score"] = fused_scores[key]
        row["hybrid_score"] = fused_scores[key]
        row["retrieval_method"] = "multi_query_rrf"
        output.append(row)

    output.sort(
        key=lambda row: (
            -float(row.get("hybrid_score", 0.0)),
            _safe_str(row.get("source_filename")),
            _safe_str(row.get("chunk_id")),
        )
    )

    return output


def reciprocal_rank_fusion(
    chroma_rows,
    csv_rows,
    *,
    rrf_k=60,
):
    return reciprocal_rank_fusion_many(
        [
            ("chroma_restricted", chroma_rows),
            ("csv_lexical_restricted", csv_rows),
        ],
        rrf_k=rrf_k,
    )


def balanced_hybrid_selection(
    chroma_rows,
    csv_rows,
    fused_rows,
    *,
    final_top_k=8,
    chroma_quota=3,
    csv_quota=3,
    valid_source_chunk_pairs=None,
):
    from src.tools.draft_writing.retrieval import (
        dedupe_evidence,
    )

    if final_top_k <= 0:
        return []

    selected = []
    selected_keys = set()

    def add_rows(rows, limit, selection_source):
        added = 0

        for original in rows:
            if added >= limit:
                break

            row = dict(original)
            key = _candidate_key(row)

            if not key[0] or not key[1]:
                continue

            if key in selected_keys:
                continue

            row["hybrid_selection_method"] = (
                "balanced_quota_rrf"
            )
            row["selection_source"] = selection_source

            selected.append(row)
            selected_keys.add(key)
            added += 1

    add_rows(
        chroma_rows,
        chroma_quota,
        "chroma_quota",
    )

    add_rows(
        csv_rows,
        csv_quota,
        "csv_quota",
    )

    remaining_slots = final_top_k - len(selected)

    if remaining_slots > 0:
        add_rows(
            fused_rows,
            remaining_slots,
            "rrf_completion",
        )

    final_rows = dedupe_evidence(
        selected,
        valid_source_chunk_pairs,
    )

    return final_rows[:final_top_k]
'''

MODULE_PATH.write_text(
    module_code,
    encoding="utf-8",
)

print("Módulo híbrido reconstruido:", MODULE_PATH)

# ============================================================
# 3. LIMPIAR CACHÉ E IMPORTAR
# ============================================================

module_name = "src.tools.draft_writing.hybrid_retrieval"

if module_name in sys.modules:
    del sys.modules[module_name]

importlib.invalidate_caches()

from src.adapters.draft_writing_runtime import (
    build_chroma_collection,
    build_draft_agent_input,
    load_draft_configuration,
)

from src.tools.draft_writing import (
    build_section_query,
    validate_draft_dependencies,
)

from src.tools.draft_writing.retrieval import (
    query_chroma_restricted,
    dedupe_evidence,
)

from src.tools.draft_writing.hybrid_retrieval import (
    build_quantitative_query,
    query_csv_ranked_restricted,
    reciprocal_rank_fusion,
    reciprocal_rank_fusion_many,
    balanced_hybrid_selection,
)

print("Importaciones correctas")

# ============================================================
# 4. CARGAR CONFIGURACIÓN
# ============================================================

cfg = load_draft_configuration(
    PROJECT_DIR,
    attempt_number=1,
)

bundle = validate_draft_dependencies(
    build_draft_agent_input(cfg)
)

chunks_df = bundle["chunks"]
outline = bundle["outline"]

collection = build_chroma_collection(cfg)

print("Chunks cargados:", len(chunks_df))

# ============================================================
# 5. BUSCAR SECCIÓN
# ============================================================

def safe_str(value):
    return "" if value is None else str(value).strip()


section = next(
    (
        item
        for item in outline.get("sections", [])
        if safe_str(item.get("section_id")) == SECTION_ID
    ),
    None,
)

if section is None:
    raise RuntimeError(
        f"No se encontró la sección {SECTION_ID}"
    )

source_filenames = [
    safe_str(
        paper.get("source_filename")
        if isinstance(paper, dict)
        else paper
    )
    for paper in section.get("papers_to_use", [])
]

source_filenames = [
    source
    for source in source_filenames
    if source
]

if not source_filenames:
    raise RuntimeError(
        f"La sección {SECTION_ID} no tiene papers autorizados"
    )

print("Sección:", SECTION_ID)
print("Papers autorizados:", len(source_filenames))

# ============================================================
# 6. CONSULTAS
# ============================================================

thematic_query = build_section_query(section)
quantitative_query = build_quantitative_query(
    thematic_query
)

print("\nCONSULTA TEMÁTICA")
print(thematic_query)

print("\nCONSULTA CUANTITATIVA")
print(quantitative_query)

# ============================================================
# 7. PARÁMETROS
# ============================================================

final_top_k = int(
    cfg["policy"].get(
        "top_k_evidence_per_section",
        8,
    )
)

candidate_multiplier = 3
candidate_k = final_top_k * candidate_multiplier

max_evidence_chars = int(
    cfg["policy"].get(
        "max_evidence_chars",
        18000,
    )
)

valid_pairs = {
    (
        safe_str(row["source_filename"]),
        safe_str(row["chunk_id"]),
    )
    for _, row in chunks_df.iterrows()
}

print("\nCONFIGURACIÓN")
print("candidate_k por consulta:", candidate_k)
print("final_top_k:", final_top_k)

# ============================================================
# 8. CHROMA TEMÁTICO Y CUANTITATIVO
# ============================================================

chroma_thematic_rows = query_chroma_restricted(
    collection,
    chunks_df,
    thematic_query,
    source_filenames,
    candidate_k,
    max_evidence_chars=max_evidence_chars,
    valid_source_chunk_pairs=valid_pairs,
)

chroma_quantitative_rows = query_chroma_restricted(
    collection,
    chunks_df,
    quantitative_query,
    source_filenames,
    candidate_k,
    max_evidence_chars=max_evidence_chars,
    valid_source_chunk_pairs=valid_pairs,
)

# ============================================================
# 9. CSV TEMÁTICO Y CUANTITATIVO
# ============================================================

csv_thematic_rows = query_csv_ranked_restricted(
    chunks_df,
    thematic_query,
    source_filenames,
    candidate_k,
    max_evidence_chars=max_evidence_chars,
    valid_source_chunk_pairs=valid_pairs,
)

csv_quantitative_rows = query_csv_ranked_restricted(
    chunks_df,
    quantitative_query,
    source_filenames,
    candidate_k,
    max_evidence_chars=max_evidence_chars,
    valid_source_chunk_pairs=valid_pairs,
)

# ============================================================
# 10. FUSIÓN INTERNA DE CADA RECUPERADOR
# ============================================================

chroma_rows = reciprocal_rank_fusion_many(
    [
        ("chroma_thematic", chroma_thematic_rows),
        ("chroma_quantitative", chroma_quantitative_rows),
    ],
    rrf_k=60,
)

chroma_rows = dedupe_evidence(
    chroma_rows,
    valid_pairs,
)[:candidate_k]

csv_rows = reciprocal_rank_fusion_many(
    [
        ("csv_thematic", csv_thematic_rows),
        ("csv_quantitative", csv_quantitative_rows),
    ],
    rrf_k=60,
)

csv_rows = dedupe_evidence(
    csv_rows,
    valid_pairs,
)[:candidate_k]

# ============================================================
# 11. FUSIÓN GLOBAL
# ============================================================

fused_rows = reciprocal_rank_fusion(
    chroma_rows,
    csv_rows,
    rrf_k=60,
)

fused_rows = dedupe_evidence(
    fused_rows,
    valid_pairs,
)

# ============================================================
# 12. SELECCIÓN FINAL 3 + 3 + RRF
# ============================================================

final_rows = balanced_hybrid_selection(
    chroma_rows,
    csv_rows,
    fused_rows,
    final_top_k=final_top_k,
    chroma_quota=3,
    csv_quota=3,
    valid_source_chunk_pairs=valid_pairs,
)

# ============================================================
# 13. MOSTRAR RESULTADOS
# ============================================================

target_values = [
    "0.96",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]

print("\nTOP-K MULTI-QUERY RRF")
print("=" * 80)

for rank, row in enumerate(final_rows, start=1):
    text = safe_str(row.get("text"))

    values_found = [
        value
        for value in target_values
        if value in text
    ]

    print(
        f"Rank {rank}:",
        row.get("chunk_id"),
        "| selección:",
        row.get("selection_source"),
        "| valores:",
        values_found,
    )

print("\nCOBERTURA")
print("=" * 80)

coverage = {}

for value in target_values:
    matching_chunks = [
        safe_str(row.get("chunk_id"))
        for row in final_rows
        if value in safe_str(row.get("text"))
    ]

    coverage[value] = {
        "covered": bool(matching_chunks),
        "chunks": matching_chunks,
    }

    print(
        value,
        "→",
        "CUBIERTO" if matching_chunks else "NO CUBIERTO",
        matching_chunks,
    )

# ============================================================
# 14. GUARDAR REPORTE
# ============================================================

output_file = (
    OUTPUT_DIR
    / f"{SECTION_ID}_hybrid_retrieval.json"
)

payload = {
    "status": (
        "DIAGNOSTIC_HYBRID_MULTIQUERY_COMPLETED"
    ),
    "diagnostic_only": True,
    "openai_called": False,
    "pipeline_state_modified": False,
    "contractual_attempt_created": False,
    "section_id": SECTION_ID,
    "candidate_k_per_query": candidate_k,
    "final_top_k_count": final_top_k,
    "chroma_quota": 3,
    "csv_quota": 3,
    "thematic_query": thematic_query,
    "quantitative_query": quantitative_query,
    "chroma_thematic_candidates": (
        chroma_thematic_rows
    ),
    "chroma_quantitative_candidates": (
        chroma_quantitative_rows
    ),
    "csv_thematic_candidates": (
        csv_thematic_rows
    ),
    "csv_quantitative_candidates": (
        csv_quantitative_rows
    ),
    "chroma_fused_candidates": chroma_rows,
    "csv_fused_candidates": csv_rows,
    "global_fused_candidates": fused_rows,
    "final_top_k": final_rows,
    "coverage": coverage,
}

output_file.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nREPORTE GUARDADO")
print(output_file)
print("pipeline_state_modified: False")
print("openai_called: False")

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte
Módulo híbrido reconstruido: /content/tesis_codigo/src/tools/draft_writing/hybrid_retrieval.py
Importaciones correctas
Chunks cargados: 1364
Sección: S2
Papers autorizados: 4

CONSULTA TEMÁTICA
Modelos basados en Redes Neuronales Artificiales (ANN) para pronóstico de irradiancia solar Describir y analizar los modelos ANN clásicos aplicados al pronóstico de irradiancia solar y generación fotovoltaica, destacando sus arquitecturas, variables de entrada, desempeño y limitaciones. Los modelos ANN tradicionales, como Back Propagation y MLP, han demostrado alta precisión en predicciones a corto plazo utilizando variables meteorológicas y datos históricos. Estos modelos superan en muchos casos a métodos estadísticos clásicos, pero presentan limitaciones en escenarios con alta variabilidad climática y terrenos heterogéneos. La selección adecuada de variables y preprocesamiento es fundamental para mejorar la precisión d

In [52]:
import json
from pathlib import Path

# ============================================================
# CONFIGURACIÓN DE LA SELECCIÓN POR CUATRO CANALES
# ============================================================

FINAL_TOP_K = 8

# Reserva inicial:
# 2 Chroma temáticos
# 1 Chroma cuantitativo
# 2 CSV temáticos
# 1 CSV cuantitativo
#
# Los 2 espacios restantes se completan mediante RRF global.
CHANNEL_QUOTAS = {
    "chroma_thematic": 2,
    "chroma_quantitative": 1,
    "csv_thematic": 2,
    "csv_quantitative": 1,
}

TARGET_VALUES = [
    "0.96",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]


def candidate_key(row):
    return (
        str(row.get("source_filename", "")).strip(),
        str(row.get("chunk_id", "")).strip(),
    )


def select_channel_balanced(
    *,
    chroma_thematic_rows,
    chroma_quantitative_rows,
    csv_thematic_rows,
    csv_quantitative_rows,
    global_fused_rows,
    final_top_k,
    channel_quotas,
    valid_source_chunk_pairs,
):
    """
    Selección balanceada por canal de recuperación.

    1. Reserva cuotas separadas para cada combinación:
       recuperador + tipo de consulta.
    2. Evita duplicados por source_filename + chunk_id.
    3. Completa los lugares restantes con RRF global.
    """

    selected = []
    selected_keys = set()

    def add_from_channel(rows, quota, channel_name):
        added = 0

        for rank, original in enumerate(rows, start=1):
            if added >= quota:
                break

            row = dict(original)
            key = candidate_key(row)

            if not key[0] or not key[1]:
                continue

            if key in selected_keys:
                continue

            if key not in valid_source_chunk_pairs:
                continue

            row["selection_source"] = channel_name
            row["selection_channel_rank"] = rank
            row["hybrid_selection_method"] = (
                "four_channel_balanced_rrf"
            )

            selected.append(row)
            selected_keys.add(key)
            added += 1

    # --------------------------------------------------------
    # CUOTAS POR CANAL
    # --------------------------------------------------------

    add_from_channel(
        chroma_thematic_rows,
        channel_quotas["chroma_thematic"],
        "chroma_thematic_quota",
    )

    add_from_channel(
        chroma_quantitative_rows,
        channel_quotas["chroma_quantitative"],
        "chroma_quantitative_quota",
    )

    add_from_channel(
        csv_thematic_rows,
        channel_quotas["csv_thematic"],
        "csv_thematic_quota",
    )

    add_from_channel(
        csv_quantitative_rows,
        channel_quotas["csv_quantitative"],
        "csv_quantitative_quota",
    )

    # --------------------------------------------------------
    # COMPLETAR CON RRF GLOBAL
    # --------------------------------------------------------

    for rank, original in enumerate(global_fused_rows, start=1):
        if len(selected) >= final_top_k:
            break

        row = dict(original)
        key = candidate_key(row)

        if not key[0] or not key[1]:
            continue

        if key in selected_keys:
            continue

        if key not in valid_source_chunk_pairs:
            continue

        row["selection_source"] = "global_rrf_completion"
        row["selection_channel_rank"] = rank
        row["hybrid_selection_method"] = (
            "four_channel_balanced_rrf"
        )

        selected.append(row)
        selected_keys.add(key)

    return selected[:final_top_k]


# ============================================================
# CONSTRUIR RRF GLOBAL DIRECTAMENTE DESDE LOS CUATRO CANALES
# ============================================================

global_fused_rows = reciprocal_rank_fusion_many(
    [
        (
            "chroma_thematic",
            chroma_thematic_rows,
        ),
        (
            "chroma_quantitative",
            chroma_quantitative_rows,
        ),
        (
            "csv_thematic",
            csv_thematic_rows,
        ),
        (
            "csv_quantitative",
            csv_quantitative_rows,
        ),
    ],
    rrf_k=60,
)

global_fused_rows = dedupe_evidence(
    global_fused_rows,
    valid_pairs,
)

# ============================================================
# SELECCIÓN FINAL
# ============================================================

final_rows_four_channel = select_channel_balanced(
    chroma_thematic_rows=chroma_thematic_rows,
    chroma_quantitative_rows=chroma_quantitative_rows,
    csv_thematic_rows=csv_thematic_rows,
    csv_quantitative_rows=csv_quantitative_rows,
    global_fused_rows=global_fused_rows,
    final_top_k=FINAL_TOP_K,
    channel_quotas=CHANNEL_QUOTAS,
    valid_source_chunk_pairs=valid_pairs,
)

# ============================================================
# MOSTRAR TOP-K
# ============================================================

print("TOP-K BALANCEADO POR CUATRO CANALES")
print("=" * 90)

for rank, row in enumerate(
    final_rows_four_channel,
    start=1,
):
    text = str(row.get("text", ""))

    values_found = [
        value
        for value in TARGET_VALUES
        if value in text
    ]

    print(
        f"Rank {rank}:",
        row.get("chunk_id"),
        "| canal:",
        row.get("selection_source"),
        "| valores:",
        values_found,
    )

# ============================================================
# MEDIR COBERTURA
# ============================================================

coverage = {}

print("\nCOBERTURA")
print("=" * 90)

for value in TARGET_VALUES:
    matching_chunks = [
        str(row.get("chunk_id", ""))
        for row in final_rows_four_channel
        if value in str(row.get("text", ""))
    ]

    coverage[value] = {
        "covered": bool(matching_chunks),
        "chunks": matching_chunks,
    }

    print(
        value,
        "→",
        "CUBIERTO" if matching_chunks else "NO CUBIERTO",
        matching_chunks,
    )

covered_count = sum(
    1
    for result in coverage.values()
    if result["covered"]
)

coverage_rate = covered_count / len(TARGET_VALUES)

print("\nRESUMEN")
print("=" * 90)
print(
    "Valores cubiertos:",
    f"{covered_count}/{len(TARGET_VALUES)}",
)
print(
    "Cobertura:",
    f"{coverage_rate:.1%}",
)

# ============================================================
# GUARDAR REPORTE AISLADO
# ============================================================

output_dir = Path(
    "/content/proyecto_estado_arte/"
    "experimento_paper_02/"
    "05_outputs/"
    "05_draft_experiments/"
    "hybrid_four_channel_balanced"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_file = (
    output_dir
    / "S2_hybrid_retrieval.json"
)

payload = {
    "status": (
        "DIAGNOSTIC_FOUR_CHANNEL_BALANCED_COMPLETED"
    ),
    "diagnostic_only": True,
    "openai_called": False,
    "pipeline_state_modified": False,
    "contractual_attempt_created": False,
    "section_id": "S2",
    "selection_method": (
        "four_channel_balanced_rrf"
    ),
    "final_top_k": FINAL_TOP_K,
    "channel_quotas": CHANNEL_QUOTAS,
    "candidate_k_per_query": candidate_k,
    "thematic_query": thematic_query,
    "quantitative_query": quantitative_query,
    "chroma_thematic_candidates": (
        chroma_thematic_rows
    ),
    "chroma_quantitative_candidates": (
        chroma_quantitative_rows
    ),
    "csv_thematic_candidates": (
        csv_thematic_rows
    ),
    "csv_quantitative_candidates": (
        csv_quantitative_rows
    ),
    "global_fused_candidates": (
        global_fused_rows
    ),
    "final_top_k_rows": (
        final_rows_four_channel
    ),
    "coverage": coverage,
    "covered_values_count": covered_count,
    "total_target_values": len(TARGET_VALUES),
    "coverage_rate": coverage_rate,
}

output_file.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nREPORTE GUARDADO")
print(output_file)
print("pipeline_state_modified: False")
print("openai_called: False")

TOP-K BALANCEADO POR CUATRO CANALES
Rank 1: 696fb1df0f31a0b4_chunk_0015 | canal: chroma_thematic_quota | valores: []
Rank 2: 696fb1df0f31a0b4_chunk_0014 | canal: chroma_thematic_quota | valores: []
Rank 3: 42891eb1891ec233_chunk_0008 | canal: chroma_quantitative_quota | valores: []
Rank 4: 12ed2391bde9cd16_chunk_0005 | canal: csv_thematic_quota | valores: []
Rank 5: 696fb1df0f31a0b4_chunk_0002 | canal: csv_thematic_quota | valores: []
Rank 6: 12ed2391bde9cd16_chunk_0024 | canal: csv_quantitative_quota | valores: []
Rank 7: 42891eb1891ec233_chunk_0026 | canal: global_rrf_completion | valores: ['1.34']
Rank 8: 42891eb1891ec233_chunk_0002 | canal: global_rrf_completion | valores: []

COBERTURA
0.96 → NO CUBIERTO []
1.34 → CUBIERTO ['42891eb1891ec233_chunk_0026']
10% → NO CUBIERTO []
58.7% → NO CUBIERTO []
6.11% → NO CUBIERTO []
99% → NO CUBIERTO []

RESUMEN
Valores cubiertos: 1/6
Cobertura: 16.7%

REPORTE GUARDADO
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_expe

In [53]:
import sys
import inspect
from pathlib import Path

CODE_ROOT = Path("/content/tesis_codigo").resolve()

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("BUSCANDO POLÍTICA DE RETRIEVAL DEL AGENTE 06")
print("=" * 80)

patterns = [
    "query_chroma_restricted(",
    "query_csv_restricted(",
    "len(rows) < top_k",
    "rows.extend(",
]

matches = []

for path in sorted((CODE_ROOT / "src").rglob("*.py")):
    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        continue

    lines = text.splitlines()

    for line_number, line in enumerate(lines, start=1):
        if any(pattern in line for pattern in patterns):
            matches.append(
                {
                    "path": path,
                    "line_number": line_number,
                    "line": line,
                }
            )

for match in matches:
    relative = match["path"].relative_to(CODE_ROOT)

    print(
        f"\n{relative}:{match['line_number']}"
    )
    print(match["line"])

print("\n" + "=" * 80)
print("ARCHIVOS ENCONTRADOS")

unique_paths = []

for match in matches:
    path = match["path"]

    if path not in unique_paths:
        unique_paths.append(path)

for path in unique_paths:
    print(path.relative_to(CODE_ROOT))

print("\n" + "=" * 80)
print("CONTEXTO DE CADA ARCHIVO")

for path in unique_paths:
    print("\n\nARCHIVO:", path.relative_to(CODE_ROOT))
    print("-" * 80)

    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()

    relevant_numbers = [
        match["line_number"]
        for match in matches
        if match["path"] == path
    ]

    if not relevant_numbers:
        continue

    start = max(1, min(relevant_numbers) - 20)
    end = min(len(lines), max(relevant_numbers) + 30)

    for number in range(start, end + 1):
        marker = ">>" if number in relevant_numbers else "  "

        print(
            f"{marker} {number:04d}: "
            f"{lines[number - 1]}"
        )

BUSCANDO POLÍTICA DE RETRIEVAL DEL AGENTE 06

src/agents/extraction_agent.py:833
                            retrieval_trace_rows.extend(trace_rows)

src/tools/draft_writing/retrieval.py:79
def query_chroma_restricted(

src/tools/draft_writing/retrieval.py:127
def query_csv_restricted(

src/tools/draft_writing/retrieval.py:177
    rows = query_chroma_restricted(

src/tools/draft_writing/retrieval.py:187
    if len(rows) < top_k:

src/tools/draft_writing/retrieval.py:188
        rows.extend(

src/tools/draft_writing/retrieval.py:189
            query_csv_restricted(

src/tools/extraction/card_extraction.py:174
            retrieval_trace_rows.extend(

src/tools/extraction/card_extraction.py:419
            retrieval_trace_rows.extend(

ARCHIVOS ENCONTRADOS
src/agents/extraction_agent.py
src/tools/draft_writing/retrieval.py
src/tools/extraction/card_extraction.py

CONTEXTO DE CADA ARCHIVO


ARCHIVO: src/agents/extraction_agent.py
----------------------------------------------------------

In [54]:
from pathlib import Path

retrieval_path = Path(
    "/content/tesis_codigo/src/tools/draft_writing/retrieval.py"
)

code = retrieval_path.read_text(encoding="utf-8")

addition = r'''


def retrieve_section_evidence_hybrid_experimental(
    section,
    collection,
    chunks_df,
    top_k,
    max_evidence_chars=18000,
    candidate_multiplier=3,
    chroma_quota=3,
    csv_quota=3,
    rrf_k=60,
):
    """
    Variante experimental del retrieval del Agente 06.

    Política:
    1. Recupera más candidatos en Chroma.
    2. Recupera más candidatos en CSV léxico ordenado.
    3. Fusiona ambos rankings mediante RRF.
    4. Reserva una cuota mínima para Chroma y CSV.
    5. Completa los espacios restantes con RRF.
    6. Devuelve únicamente top_k evidencias finales.

    Esta función no sustituye retrieve_section_evidence().
    """

    from src.tools.draft_writing.hybrid_retrieval import (
        query_csv_ranked_restricted,
        reciprocal_rank_fusion,
        balanced_hybrid_selection,
    )

    source_filenames = [
        safe_str(
            paper.get("source_filename")
            if isinstance(paper, dict)
            else paper
        )
        for paper in (section.get("papers_to_use") or [])
    ]

    source_filenames = [
        source
        for source in source_filenames
        if source
    ]

    if not source_filenames:
        return []

    if top_k <= 0:
        return []

    if candidate_multiplier <= 0:
        raise ValueError(
            "candidate_multiplier debe ser mayor que cero"
        )

    valid_source_chunk_pairs = _valid_pairs_from_chunks(
        chunks_df
    )

    query = build_section_query(section)

    candidate_k = max(
        top_k,
        top_k * candidate_multiplier,
    )

    chroma_rows = query_chroma_restricted(
        collection,
        chunks_df,
        query,
        source_filenames,
        candidate_k,
        max_evidence_chars=max_evidence_chars,
        valid_source_chunk_pairs=(
            valid_source_chunk_pairs
        ),
    )

    csv_rows = query_csv_ranked_restricted(
        chunks_df,
        query,
        source_filenames,
        candidate_k,
        max_evidence_chars=max_evidence_chars,
        valid_source_chunk_pairs=(
            valid_source_chunk_pairs
        ),
    )

    fused_rows = reciprocal_rank_fusion(
        chroma_rows,
        csv_rows,
        rrf_k=rrf_k,
    )

    fused_rows = dedupe_evidence(
        fused_rows,
        valid_source_chunk_pairs,
    )

    final_rows = balanced_hybrid_selection(
        chroma_rows,
        csv_rows,
        fused_rows,
        final_top_k=top_k,
        chroma_quota=chroma_quota,
        csv_quota=csv_quota,
        valid_source_chunk_pairs=(
            valid_source_chunk_pairs
        ),
    )

    return dedupe_evidence(
        final_rows,
        valid_source_chunk_pairs,
    )[:top_k]
'''

if "def retrieve_section_evidence_hybrid_experimental(" not in code:
    code += addition
    retrieval_path.write_text(code, encoding="utf-8")
    print("Función experimental agregada:")
    print(retrieval_path)
else:
    print("La función experimental ya existe")

Función experimental agregada:
/content/tesis_codigo/src/tools/draft_writing/retrieval.py


In [55]:
import sys
import importlib
from pathlib import Path

CODE_ROOT = Path("/content/tesis_codigo").resolve()

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

module_name = "src.tools.draft_writing.retrieval"

if module_name in sys.modules:
    del sys.modules[module_name]

importlib.invalidate_caches()

from src.tools.draft_writing.retrieval import (
    retrieve_section_evidence,
    retrieve_section_evidence_hybrid_experimental,
)

print("Baseline disponible:")
print(retrieve_section_evidence)

print("\nVariante experimental disponible:")
print(retrieve_section_evidence_hybrid_experimental)

Baseline disponible:
<function retrieve_section_evidence at 0x7148cc1054e0>

Variante experimental disponible:
<function retrieve_section_evidence_hybrid_experimental at 0x7148cc63e3e0>


In [56]:
from pathlib import Path

CODE_ROOT = Path("/content/tesis_codigo").resolve()

target = "retrieve_section_evidence"

for path in sorted((CODE_ROOT / "src").rglob("*.py")):
    text = path.read_text(encoding="utf-8")

    if target not in text:
        continue

    print("\nARCHIVO:", path.relative_to(CODE_ROOT))
    print("=" * 80)

    for number, line in enumerate(
        text.splitlines(),
        start=1,
    ):
        if target in line:
            print(
                f"{number:04d}: {line}"
            )


ARCHIVO: src/agents/draft_writing_agent.py
0038:                 sid=str(section.get('section_id','')).strip();section_query=build_section_query(section);evidence=retrieve_section_evidence(section,self.runtime.collection,bundle['chunks'],int(policy.get('top_k_evidence_per_section',8)),int(policy.get('max_evidence_chars',18000)));retrieval_rounds += 1 if section.get('papers_to_use') else 0

ARCHIVO: src/tools/draft_writing/retrieval.py
0160: def retrieve_section_evidence(
0203: def retrieve_section_evidence_hybrid_experimental(
0225:     Esta función no sustituye retrieve_section_evidence().


In [57]:
import sys
import re
import importlib
from pathlib import Path

# ============================================================
# 1. RUTAS
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()

BASELINE_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent.py"
)

EXPERIMENTAL_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent_hybrid_experimental.py"
)

assert CODE_ROOT.exists(), f"No existe CODE_ROOT: {CODE_ROOT}"
assert BASELINE_AGENT_PATH.exists(), (
    f"No existe el agente baseline: {BASELINE_AGENT_PATH}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("Agente baseline:")
print(BASELINE_AGENT_PATH)

print("\nAgente experimental:")
print(EXPERIMENTAL_AGENT_PATH)

# ============================================================
# 2. LEER EL AGENTE BASELINE
# ============================================================

baseline_code = BASELINE_AGENT_PATH.read_text(
    encoding="utf-8"
)

assert "retrieve_section_evidence" in baseline_code, (
    "No encontré retrieve_section_evidence en el agente baseline"
)

# ============================================================
# 3. CREAR COPIA EXPERIMENTAL
# ============================================================

experimental_code = baseline_code

# ------------------------------------------------------------
# 3.1. Cambiar la importación
# ------------------------------------------------------------
#
# Se reemplaza solo el símbolo importado.
# La función baseline permanece intacta en su archivo original.

experimental_code = experimental_code.replace(
    "retrieve_section_evidence,",
    "retrieve_section_evidence_hybrid_experimental,",
)

experimental_code = experimental_code.replace(
    "retrieve_section_evidence\n",
    "retrieve_section_evidence_hybrid_experimental\n",
)

# También cubre importaciones escritas en una sola línea.
experimental_code = re.sub(
    r"\bretrieve_section_evidence\b",
    "retrieve_section_evidence_hybrid_experimental",
    experimental_code,
)

# ------------------------------------------------------------
# 3.2. Evitar duplicación accidental del nombre
# ------------------------------------------------------------

experimental_code = experimental_code.replace(
    "retrieve_section_evidence_hybrid_experimental_hybrid_experimental",
    "retrieve_section_evidence_hybrid_experimental",
)

# ============================================================
# 4. AGREGAR MARCA EXPLÍCITA DE VARIANTE
# ============================================================

header = '''"""
VARIANTE EXPERIMENTAL DEL AGENTE 06.

Esta implementación conserva el agente baseline y cambia únicamente
la política de recuperación de evidencia:

Baseline:
    Chroma -> fallback CSV condicionado -> top_k

Experimental:
    Chroma ampliado + CSV léxico ordenado
    -> RRF
    -> selección balanceada 3 + 3 + RRF
    -> top_k

No debe utilizarse como reemplazo contractual hasta completar
la evaluación comparativa.
"""

'''

# Evita duplicar el encabezado si se ejecuta otra vez.
if not experimental_code.startswith(
    '"""VARIANTE EXPERIMENTAL'
):
    experimental_code = header + experimental_code

# ============================================================
# 5. GUARDAR EL NUEVO AGENTE
# ============================================================

EXPERIMENTAL_AGENT_PATH.write_text(
    experimental_code,
    encoding="utf-8",
)

print("\nArchivo experimental creado:")
print(EXPERIMENTAL_AGENT_PATH)

# ============================================================
# 6. VERIFICACIONES TEXTUALES
# ============================================================

saved_code = EXPERIMENTAL_AGENT_PATH.read_text(
    encoding="utf-8"
)

baseline_call_present = bool(
    re.search(
        r"\bretrieve_section_evidence\s*\(",
        saved_code,
    )
)

experimental_call_present = bool(
    re.search(
        r"\bretrieve_section_evidence_hybrid_experimental\s*\(",
        saved_code,
    )
)

experimental_symbol_present = (
    "retrieve_section_evidence_hybrid_experimental"
    in saved_code
)

print("\nVERIFICACIÓN TEXTUAL")
print("=" * 80)
print(
    "Contiene símbolo experimental:",
    experimental_symbol_present,
)
print(
    "Contiene llamada experimental:",
    experimental_call_present,
)
print(
    "Conserva llamada baseline:",
    baseline_call_present,
)

assert experimental_symbol_present, (
    "No se agregó el símbolo experimental"
)

assert experimental_call_present, (
    "No se sustituyó la llamada de retrieval"
)

assert not baseline_call_present, (
    "Todavía existe una llamada a retrieve_section_evidence baseline"
)

# ============================================================
# 7. COMPROBAR SINTAXIS DEL ARCHIVO
# ============================================================

compile(
    saved_code,
    str(EXPERIMENTAL_AGENT_PATH),
    "exec",
)

print("\nSintaxis Python válida")

# ============================================================
# 8. LIMPIAR CACHÉ E IMPORTAR MÓDULO
# ============================================================

module_name = (
    "src.agents."
    "draft_writing_agent_hybrid_experimental"
)

if module_name in sys.modules:
    del sys.modules[module_name]

importlib.invalidate_caches()

experimental_module = importlib.import_module(
    module_name
)

print("Módulo experimental importado correctamente:")
print(experimental_module)

# ============================================================
# 9. IDENTIFICAR CLASES DEL MÓDULO
# ============================================================

classes = []

for name, value in vars(experimental_module).items():
    if isinstance(value, type):
        if value.__module__ == module_name:
            classes.append(name)

print("\nCLASES DEFINIDAS EN EL AGENTE EXPERIMENTAL")
print("=" * 80)

for class_name in classes:
    print("✓", class_name)

if not classes:
    raise RuntimeError(
        "No encontré ninguna clase definida en el módulo experimental"
    )

# ============================================================
# 10. MOSTRAR LA LÍNEA DONDE SE USA EL RETRIEVAL
# ============================================================

print("\nUSO DEL RETRIEVAL EN LA VARIANTE")
print("=" * 80)

for line_number, line in enumerate(
    saved_code.splitlines(),
    start=1,
):
    if (
        "retrieve_section_evidence_hybrid_experimental"
        in line
    ):
        print(
            f"{line_number:04d}: {line}"
        )

# ============================================================
# 11. LOCALIZAR DÓNDE SE IMPORTA EL AGENTE BASELINE
# ============================================================

print("\nARCHIVOS QUE IMPORTAN O USAN DraftWritingAgent")
print("=" * 80)

usage_matches = []

for path in sorted((CODE_ROOT / "src").rglob("*.py")):
    if path in {
        BASELINE_AGENT_PATH,
        EXPERIMENTAL_AGENT_PATH,
    }:
        continue

    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        continue

    for line_number, line in enumerate(
        text.splitlines(),
        start=1,
    ):
        if (
            "DraftWritingAgent" in line
            or "draft_writing_agent" in line
        ):
            usage_matches.append(
                {
                    "path": path,
                    "line_number": line_number,
                    "line": line,
                }
            )

for match in usage_matches:
    relative_path = match["path"].relative_to(
        CODE_ROOT
    )

    print(
        f"\n{relative_path}:{match['line_number']}"
    )
    print(match["line"])

print("\n" + "=" * 80)
print("RESULTADO")
print(
    "Agente baseline modificado:",
    False,
)
print(
    "Agente experimental creado:",
    EXPERIMENTAL_AGENT_PATH.exists(),
)
print(
    "PipelineState modificado:",
    False,
)
print(
    "OpenAI llamado:",
    False,
)

Agente baseline:
/content/tesis_codigo/src/agents/draft_writing_agent.py

Agente experimental:
/content/tesis_codigo/src/agents/draft_writing_agent_hybrid_experimental.py

Archivo experimental creado:
/content/tesis_codigo/src/agents/draft_writing_agent_hybrid_experimental.py

VERIFICACIÓN TEXTUAL
Contiene símbolo experimental: True
Contiene llamada experimental: True
Conserva llamada baseline: False

Sintaxis Python válida
Módulo experimental importado correctamente:
<module 'src.agents.draft_writing_agent_hybrid_experimental' from '/content/tesis_codigo/src/agents/draft_writing_agent_hybrid_experimental.py'>

CLASES DEFINIDAS EN EL AGENTE EXPERIMENTAL
✓ DraftWritingAgent

USO DEL RETRIEVAL EN LA VARIANTE
0057:                 sid=str(section.get('section_id','')).strip();section_query=build_section_query(section);evidence=retrieve_section_evidence_hybrid_experimental(section,self.runtime.collection,bundle['chunks'],int(policy.get('top_k_evidence_per_section',8)),int(policy.get('max_e

In [58]:
import sys
import re
import importlib
from pathlib import Path

# ============================================================
# 1. RUTAS
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()

BASELINE_RUNTIME_PATH = (
    CODE_ROOT
    / "src"
    / "adapters"
    / "draft_writing_runtime.py"
)

EXPERIMENTAL_RUNTIME_PATH = (
    CODE_ROOT
    / "src"
    / "adapters"
    / "draft_writing_hybrid_runtime.py"
)

assert CODE_ROOT.exists(), f"No existe CODE_ROOT: {CODE_ROOT}"
assert BASELINE_RUNTIME_PATH.exists(), (
    f"No existe el runtime baseline: {BASELINE_RUNTIME_PATH}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("Runtime baseline:")
print(BASELINE_RUNTIME_PATH)

print("\nRuntime experimental:")
print(EXPERIMENTAL_RUNTIME_PATH)

# ============================================================
# 2. LEER RUNTIME BASELINE
# ============================================================

baseline_code = BASELINE_RUNTIME_PATH.read_text(
    encoding="utf-8"
)

assert (
    "from src.agents.draft_writing_agent import DraftWritingAgent"
    in baseline_code
), "No encontré el import esperado del agente baseline"

# ============================================================
# 3. CREAR COPIA EXPERIMENTAL
# ============================================================

experimental_code = baseline_code.replace(
    "from src.agents.draft_writing_agent import DraftWritingAgent",
    (
        "from src.agents."
        "draft_writing_agent_hybrid_experimental "
        "import DraftWritingAgent"
    ),
)

header = '''"""
RUNTIME EXPERIMENTAL DEL AGENTE 06.

Construye DraftWritingAgent desde la variante híbrida experimental.

No sustituye el runtime baseline.
No modifica PipelineState por sí mismo.
"""

'''

if not experimental_code.startswith(
    '"""RUNTIME EXPERIMENTAL'
):
    experimental_code = header + experimental_code

EXPERIMENTAL_RUNTIME_PATH.write_text(
    experimental_code,
    encoding="utf-8",
)

print("\nRuntime experimental creado:")
print(EXPERIMENTAL_RUNTIME_PATH)

# ============================================================
# 4. VERIFICACIÓN TEXTUAL
# ============================================================

saved_code = EXPERIMENTAL_RUNTIME_PATH.read_text(
    encoding="utf-8"
)

baseline_import_present = (
    "from src.agents.draft_writing_agent import DraftWritingAgent"
    in saved_code
)

experimental_import_present = (
    "draft_writing_agent_hybrid_experimental"
    in saved_code
)

print("\nVERIFICACIÓN TEXTUAL")
print("=" * 80)
print(
    "Importa agente experimental:",
    experimental_import_present,
)
print(
    "Conserva import baseline:",
    baseline_import_present,
)

assert experimental_import_present, (
    "El runtime no importa el agente experimental"
)

assert not baseline_import_present, (
    "El runtime todavía importa el agente baseline"
)

# ============================================================
# 5. VALIDAR SINTAXIS
# ============================================================

compile(
    saved_code,
    str(EXPERIMENTAL_RUNTIME_PATH),
    "exec",
)

print("\nSintaxis Python válida")

# ============================================================
# 6. LIMPIAR CACHÉ E IMPORTAR
# ============================================================

module_name = (
    "src.adapters."
    "draft_writing_hybrid_runtime"
)

if module_name in sys.modules:
    del sys.modules[module_name]

importlib.invalidate_caches()

experimental_runtime = importlib.import_module(
    module_name
)

print("\nRuntime experimental importado:")
print(experimental_runtime)

# ============================================================
# 7. VERIFICAR FUNCIÓN DE CONSTRUCCIÓN
# ============================================================

required_functions = [
    "load_draft_configuration",
    "build_chroma_collection",
    "build_draft_agent_input",
    "build_real_draft_execution",
]

print("\nFUNCIONES DISPONIBLES")
print("=" * 80)

for function_name in required_functions:
    exists = hasattr(
        experimental_runtime,
        function_name,
    )

    print(
        "✓" if exists else "✗",
        function_name,
    )

    assert exists, (
        f"Falta la función {function_name}"
    )

# ============================================================
# 8. VERIFICAR QUÉ CLASE DE AGENTE IMPORTA
# ============================================================

agent_class = getattr(
    experimental_runtime,
    "DraftWritingAgent",
)

print("\nCLASE DE AGENTE UTILIZADA")
print("=" * 80)
print("Módulo:", agent_class.__module__)
print("Clase:", agent_class.__name__)

assert (
    agent_class.__module__
    == "src.agents.draft_writing_agent_hybrid_experimental"
), (
    "El runtime no está usando el agente híbrido experimental"
)

# ============================================================
# 9. RESULTADO
# ============================================================

print("\n" + "=" * 80)
print("RESULTADO")
print(
    "Runtime baseline modificado:",
    False,
)
print(
    "Runtime experimental creado:",
    EXPERIMENTAL_RUNTIME_PATH.exists(),
)
print(
    "Agente experimental conectado:",
    True,
)
print(
    "PipelineState modificado:",
    False,
)
print(
    "OpenAI llamado:",
    False,
)

Runtime baseline:
/content/tesis_codigo/src/adapters/draft_writing_runtime.py

Runtime experimental:
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py

Runtime experimental creado:
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py

VERIFICACIÓN TEXTUAL
Importa agente experimental: True
Conserva import baseline: False

Sintaxis Python válida

Runtime experimental importado:
<module 'src.adapters.draft_writing_hybrid_runtime' from '/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py'>

FUNCIONES DISPONIBLES
✓ load_draft_configuration
✓ build_chroma_collection
✓ build_draft_agent_input
✓ build_real_draft_execution

CLASE DE AGENTE UTILIZADA
Módulo: src.agents.draft_writing_agent_hybrid_experimental
Clase: DraftWritingAgent

RESULTADO
Runtime baseline modificado: False
Runtime experimental creado: True
Agente experimental conectado: True
PipelineState modificado: False
OpenAI llamado: False


In [59]:
import sys
import inspect
import importlib
from pathlib import Path

# ============================================================
# 1. CONFIGURAR REPOSITORIO
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()

assert CODE_ROOT.exists(), f"No existe CODE_ROOT: {CODE_ROOT}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

importlib.invalidate_caches()

# ============================================================
# 2. IMPORTAR RUNTIME EXPERIMENTAL
# ============================================================

from src.adapters import draft_writing_hybrid_runtime as hybrid_runtime

from src.agents.draft_writing_agent_hybrid_experimental import (
    DraftWritingAgent,
)

print("RUNTIME EXPERIMENTAL")
print("=" * 90)
print(hybrid_runtime.__file__)

print("\nAGENTE EXPERIMENTAL")
print("=" * 90)
print(inspect.getfile(DraftWritingAgent))

# ============================================================
# 3. INSPECCIONAR FUNCIONES PRINCIPALES DEL RUNTIME
# ============================================================

runtime_functions = [
    "load_draft_configuration",
    "build_chroma_collection",
    "build_draft_agent_input",
    "build_openai_draft_runtime",
    "build_real_draft_execution",
    "execute_draft_transaction",
]

for function_name in runtime_functions:
    print("\n\n" + "=" * 90)
    print("FUNCIÓN:", function_name)
    print("=" * 90)

    function = getattr(
        hybrid_runtime,
        function_name,
        None,
    )

    if function is None:
        print("NO EXISTE EN EL RUNTIME")
        continue

    print("Firma:")
    print(inspect.signature(function))

    print("\nCódigo fuente:")
    try:
        print(inspect.getsource(function))
    except (OSError, TypeError) as error:
        print(
            "No se pudo obtener el código fuente:",
            type(error).__name__,
            str(error),
        )

# ============================================================
# 4. INSPECCIONAR MÉTODOS DEL AGENTE
# ============================================================

print("\n\n" + "=" * 90)
print("MÉTODOS DE DraftWritingAgent")
print("=" * 90)

agent_methods = []

for method_name, method in inspect.getmembers(
    DraftWritingAgent,
    predicate=inspect.isfunction,
):
    if method_name.startswith("_"):
        continue

    agent_methods.append(method_name)

    print("\nMÉTODO:", method_name)
    print("Firma:", inspect.signature(method))

    try:
        print(inspect.getsource(method))
    except (OSError, TypeError) as error:
        print(
            "No se pudo obtener el código fuente:",
            type(error).__name__,
            str(error),
        )

# ============================================================
# 5. BUSCAR OPERACIONES QUE PODRÍAN MODIFICAR EL ESTADO
# ============================================================

print("\n\n" + "=" * 90)
print("BÚSQUEDA DE OPERACIONES CONTRACTUALES")
print("=" * 90)

source_path = Path(hybrid_runtime.__file__)
source_text = source_path.read_text(encoding="utf-8")

dangerous_terms = [
    "pipeline_state",
    "execute_draft_transaction",
    "commit",
    "prepare",
    "requested_transition",
    "attempt_number",
    "published_draft",
    "state_store",
]

for term in dangerous_terms:
    matching_lines = []

    for line_number, line in enumerate(
        source_text.splitlines(),
        start=1,
    ):
        if term.lower() in line.lower():
            matching_lines.append(
                (line_number, line)
            )

    print(f"\n{term}: {len(matching_lines)} coincidencias")

    for line_number, line in matching_lines:
        print(
            f"  {line_number:04d}: {line}"
        )

# ============================================================
# 6. RESUMEN
# ============================================================

print("\n\n" + "=" * 90)
print("RESUMEN")
print("=" * 90)

print(
    "Funciones encontradas:",
    [
        name
        for name in runtime_functions
        if hasattr(hybrid_runtime, name)
    ],
)

print(
    "Métodos públicos del agente:",
    agent_methods,
)

print("OpenAI llamado:", False)
print("PipelineState modificado:", False)
print("Intento contractual creado:", False)

RUNTIME EXPERIMENTAL
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py

AGENTE EXPERIMENTAL
/content/tesis_codigo/src/agents/draft_writing_agent_hybrid_experimental.py


FUNCIÓN: load_draft_configuration
Firma:
(project_dir, attempt_number=1, *, chroma_client_factory=None)

Código fuente:
def load_draft_configuration(project_dir,attempt_number=1,*,chroma_client_factory=None):
    root=Path(project_dir).resolve();active=json.loads((root/'active_experiment.json').read_text(encoding='utf-8'));eid=active['active_experiment_id'];exp=root/eid;outputs=exp/'05_outputs';outline=outputs/'04_outline';thematic=outputs/'03_thematic_analysis';draft=outputs/'05_draft';rag=active.get('rag_policy',{});policy=get_draft_writing_policy(active.get('draft_generation_policy',{}));generation=active.get('generation_profile',{});policy.update({'experiment_profile':active.get('experiment_profile',{}),'topic_profile':active.get('topic_profile',{}),'generation_profile':generation,'rag_policy':rag,

In [60]:
import sys
import json
import copy
import hashlib
import importlib
from pathlib import Path
from dataclasses import is_dataclass, replace as dataclass_replace, asdict

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()
PROJECT_DIR = Path("/content/proyecto_estado_arte").resolve()

EXPERIMENT_NAME = "agent06_hybrid_balanced_3_3_rrf"
ATTEMPT_NUMBER = 1

assert CODE_ROOT.exists(), f"No existe CODE_ROOT: {CODE_ROOT}"
assert PROJECT_DIR.exists(), f"No existe PROJECT_DIR: {PROJECT_DIR}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

importlib.invalidate_caches()

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

# ============================================================
# 2. IMPORTAR EL RUNTIME EXPERIMENTAL
# ============================================================

from src.adapters import draft_writing_hybrid_runtime as hybrid_runtime

print("\nRuntime experimental:")
print(hybrid_runtime.__file__)

# ============================================================
# 3. FUNCIONES AUXILIARES
# ============================================================

def sha256_file_local(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def replace_record(instance, **updates):
    """
    Reemplaza campos en modelos Pydantic, dataclasses
    u objetos Python normales.
    """

    if hasattr(instance, "model_copy"):
        return instance.model_copy(
            update=updates,
            deep=True,
        )

    if hasattr(instance, "copy"):
        try:
            return instance.copy(
                update=updates,
                deep=True,
            )
        except TypeError:
            pass

    if is_dataclass(instance):
        return dataclass_replace(
            instance,
            **updates,
        )

    cloned = copy.deepcopy(instance)

    for name, value in updates.items():
        setattr(cloned, name, value)

    return cloned


def to_serializable(value):
    """
    Convierte AgentResult y objetos asociados en estructuras JSON.
    """

    if value is None:
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, dict):
        return {
            str(key): to_serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            to_serializable(item)
            for item in value
        ]

    if hasattr(value, "model_dump"):
        return to_serializable(
            value.model_dump(mode="json")
        )

    if hasattr(value, "dict"):
        try:
            return to_serializable(
                value.dict()
            )
        except Exception:
            pass

    if is_dataclass(value):
        return to_serializable(
            asdict(value)
        )

    if hasattr(value, "value"):
        try:
            return to_serializable(value.value)
        except Exception:
            pass

    if hasattr(value, "__dict__"):
        return {
            key: to_serializable(item)
            for key, item in vars(value).items()
            if not key.startswith("_")
        }

    return str(value)


def enum_or_value(value):
    if hasattr(value, "value"):
        return value.value

    return str(value)


# ============================================================
# 4. CONSTRUIR AGENTE, INPUT Y CONFIGURACIÓN
# ============================================================

agent, baseline_input, cfg = (
    hybrid_runtime.build_real_draft_execution(
        PROJECT_DIR,
        attempt_number=ATTEMPT_NUMBER,
    )
)

print("\nAgente construido:")
print(agent.__class__.__module__)
print(agent.__class__.__name__)

assert (
    agent.__class__.__module__
    == "src.agents.draft_writing_agent_hybrid_experimental"
), (
    "Se construyó un agente distinto de la variante híbrida"
)

# ============================================================
# 5. DEFINIR SALIDA EXPERIMENTAL AISLADA
# ============================================================

EXPERIMENT_OUTPUT_DIR = (
    Path(cfg["experiment_dir"])
    / "05_outputs"
    / "05_draft_experiments"
    / EXPERIMENT_NAME
)

EXPERIMENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nSalida contractual original:")
print(cfg["output_dir"])

print("\nSalida experimental:")
print(EXPERIMENT_OUTPUT_DIR)

assert (
    EXPERIMENT_OUTPUT_DIR.resolve()
    != Path(cfg["output_dir"]).resolve()
), "La salida experimental coincide con la contractual"

# ============================================================
# 6. MODIFICAR SOLO LA COPIA DEL AGENT INPUT
# ============================================================

experimental_context = replace_record(
    baseline_input.agent_context,
    output_directory=str(EXPERIMENT_OUTPUT_DIR),
)

experimental_policy = dict(
    baseline_input.policy
)

# Impide reutilizar artefactos anteriores.
experimental_policy["force_rebuild"] = True

# Identificación explícita de la variante.
experimental_policy["experimental_run"] = True
experimental_policy["contractual_execution"] = False
experimental_policy["retrieval_variant"] = (
    "hybrid_thematic_candidate24_balanced_3_3_rrf"
)

# Evita que el fingerprint coincida con el baseline.
baseline_fingerprint = str(
    experimental_policy.get(
        "current_fingerprint",
        "",
    )
)

experimental_policy["current_fingerprint"] = (
    f"{baseline_fingerprint}"
    "::experimental_agent06_hybrid_3_3_rrf_v1"
)

experimental_input = replace_record(
    baseline_input,
    agent_context=experimental_context,
    policy=experimental_policy,
)

print("\nAgentInput experimental preparado")
print(
    "output_directory:",
    experimental_input.agent_context.output_directory,
)
print(
    "force_rebuild:",
    experimental_input.policy.get("force_rebuild"),
)
print(
    "retrieval_variant:",
    experimental_input.policy.get("retrieval_variant"),
)

# ============================================================
# 7. REGISTRAR HASH DEL PIPELINESTATE ANTES
# ============================================================

pipeline_state_path = Path(
    cfg["state_path"]
).resolve()

assert pipeline_state_path.is_file(), (
    f"No existe PipelineState: {pipeline_state_path}"
)

state_hash_before = sha256_file_local(
    pipeline_state_path
)

print("\nPipelineState:")
print(pipeline_state_path)
print("SHA-256 antes:", state_hash_before)

# ============================================================
# 8. EJECUTAR AGENTE 06 EXPERIMENTAL
# ============================================================
#
# ESTA LÍNEA LLAMA A OPENAI.
# No usa execute_draft_transaction ni state_store.

print("\n" + "=" * 80)
print("EJECUTANDO AGENTE 06 HÍBRIDO EXPERIMENTAL")
print("=" * 80)

result = agent.execute(
    experimental_input
)

print("\nEjecución terminada")

# ============================================================
# 9. VERIFICAR PIPELINESTATE DESPUÉS
# ============================================================

state_hash_after = sha256_file_local(
    pipeline_state_path
)

pipeline_state_modified = (
    state_hash_before != state_hash_after
)

print("\nSHA-256 después:", state_hash_after)
print(
    "PipelineState modificado:",
    pipeline_state_modified,
)

if pipeline_state_modified:
    raise RuntimeError(
        "El PipelineState cambió durante la ejecución experimental"
    )

# ============================================================
# 10. EXTRAER RESULTADO PRINCIPAL
# ============================================================

execution_status = enum_or_value(
    result.execution_status
)

quality_status = enum_or_value(
    result.quality_status
)

transition_action = enum_or_value(
    result.requested_transition.action
)

decision_code = str(
    result.decision.code
)

tool_usage = to_serializable(
    result.tool_usage
)

print("\n" + "=" * 80)
print("RESULTADO DEL AGENTE 06 EXPERIMENTAL")
print("=" * 80)

print("execution_status:", execution_status)
print("quality_status:", quality_status)
print("decision_code:", decision_code)
print("requested_transition:", transition_action)
print("tool_usage:", tool_usage)

print("\nFailure reason codes:")
print(
    list(result.failure_reason_codes or ())
)

print("\nWarnings:")

for warning in result.warnings or ():
    warning_data = to_serializable(warning)
    print(warning_data)

# ============================================================
# 11. GUARDAR AGENTRESULT EXPERIMENTAL
# ============================================================

result_payload = {
    "experiment_type": (
        "agent06_hybrid_retrieval_generation"
    ),
    "contractual_execution": False,
    "pipeline_state_modified": False,
    "attempt_number_created": False,
    "openai_called": True,
    "baseline_preserved": True,
    "retrieval_variant": (
        "hybrid_thematic_candidate24_balanced_3_3_rrf"
    ),
    "project_dir": str(PROJECT_DIR),
    "experiment_id": cfg["experiment_id"],
    "experimental_output_directory": str(
        EXPERIMENT_OUTPUT_DIR
    ),
    "contractual_output_directory": str(
        cfg["output_dir"]
    ),
    "pipeline_state_path": str(
        pipeline_state_path
    ),
    "pipeline_state_sha256_before": (
        state_hash_before
    ),
    "pipeline_state_sha256_after": (
        state_hash_after
    ),
    "execution_status": execution_status,
    "quality_status": quality_status,
    "decision_code": decision_code,
    "requested_transition": transition_action,
    "agent_result": to_serializable(result),
}

result_file = (
    EXPERIMENT_OUTPUT_DIR
    / "experimental_agent_result.json"
)

result_file.write_text(
    json.dumps(
        result_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nResultado guardado:")
print(result_file)

# ============================================================
# 12. MOSTRAR ARCHIVOS GENERADOS
# ============================================================

print("\n" + "=" * 80)
print("ARCHIVOS GENERADOS")
print("=" * 80)

generated_files = sorted(
    path
    for path in EXPERIMENT_OUTPUT_DIR.rglob("*")
    if path.is_file()
)

for path in generated_files:
    print(
        path.relative_to(
            EXPERIMENT_OUTPUT_DIR
        )
    )

# ============================================================
# 13. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN FINAL")
print("=" * 80)

print(
    "Agente usado:",
    agent.__class__.__module__,
)

print(
    "Salida aislada:",
    EXPERIMENT_OUTPUT_DIR,
)

print(
    "OpenAI llamado:",
    True,
)

print(
    "PipelineState modificado:",
    pipeline_state_modified,
)

print(
    "Intento contractual creado:",
    False,
)

print(
    "Baseline sobrescrito:",
    False,
)

print(
    "Calidad obtenida:",
    quality_status,
)

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte

Runtime experimental:
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py

Agente construido:
src.agents.draft_writing_agent_hybrid_experimental
DraftWritingAgent

Salida contractual original:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft

Salida experimental:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_balanced_3_3_rrf

AgentInput experimental preparado
output_directory: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_balanced_3_3_rrf
force_rebuild: True
retrieval_variant: hybrid_thematic_candidate24_balanced_3_3_rrf

PipelineState:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/00_orchestrator_planner/pipeline_state.json
SHA-256 antes: dd11c625d900c4699ced201c9862dab6deb3468e6b1d0e2b2ef2385786136da1

EJECUTANDO AGENTE 06 HÍBRIDO EXPERIMENTAL

Ejecución

In [61]:
import sys
import json
import copy
import hashlib
import importlib
from enum import Enum
from pathlib import Path
from dataclasses import (
    is_dataclass,
    replace as dataclass_replace,
    asdict,
)

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()
PROJECT_DIR = Path("/content/proyecto_estado_arte").resolve()

EXPERIMENT_NAME = (
    "agent06_hybrid_balanced_3_3_rrf_fixed_import"
)

ATTEMPT_NUMBER = 1

EXPERIMENTAL_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent_hybrid_experimental.py"
)

BASELINE_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent.py"
)

assert CODE_ROOT.exists(), (
    f"No existe CODE_ROOT: {CODE_ROOT}"
)

assert PROJECT_DIR.exists(), (
    f"No existe PROJECT_DIR: {PROJECT_DIR}"
)

assert EXPERIMENTAL_AGENT_PATH.is_file(), (
    "No existe el agente experimental: "
    f"{EXPERIMENTAL_AGENT_PATH}"
)

assert BASELINE_AGENT_PATH.is_file(), (
    "No existe el agente baseline: "
    f"{BASELINE_AGENT_PATH}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

# ============================================================
# 2. REGISTRAR HASH DEL AGENTE BASELINE
# ============================================================

def sha256_file_local(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


baseline_agent_hash_before = sha256_file_local(
    BASELINE_AGENT_PATH
)

print("\nHash agente baseline:")
print(baseline_agent_hash_before)

# ============================================================
# 3. CORREGIR IMPORT DEL AGENTE EXPERIMENTAL
# ============================================================

experimental_code = EXPERIMENTAL_AGENT_PATH.read_text(
    encoding="utf-8"
)

required_import = (
    "from src.tools.draft_writing.retrieval import "
    "retrieve_section_evidence_hybrid_experimental"
)

if required_import not in experimental_code:
    class_marker = "class DraftWritingAgent"

    if class_marker not in experimental_code:
        raise RuntimeError(
            "No se encontró class DraftWritingAgent "
            "en el agente experimental"
        )

    experimental_code = experimental_code.replace(
        class_marker,
        required_import
        + "\n\n"
        + class_marker,
        1,
    )

    EXPERIMENTAL_AGENT_PATH.write_text(
        experimental_code,
        encoding="utf-8",
    )

    print(
        "\nImport experimental agregado correctamente"
    )
else:
    print(
        "\nEl import experimental ya estaba presente"
    )

# ============================================================
# 4. VERIFICAR QUE EL BASELINE NO CAMBIÓ
# ============================================================

baseline_agent_hash_after_patch = (
    sha256_file_local(BASELINE_AGENT_PATH)
)

assert (
    baseline_agent_hash_before
    == baseline_agent_hash_after_patch
), "El agente baseline fue modificado accidentalmente"

print("Agente baseline modificado:", False)

# ============================================================
# 5. VERIFICAR SINTAXIS DEL AGENTE EXPERIMENTAL
# ============================================================

patched_code = EXPERIMENTAL_AGENT_PATH.read_text(
    encoding="utf-8"
)

assert required_import in patched_code, (
    "El import experimental no quedó guardado"
)

assert (
    "retrieve_section_evidence_hybrid_experimental("
    in patched_code
), (
    "El agente no contiene la llamada al retrieval "
    "experimental"
)

compile(
    patched_code,
    str(EXPERIMENTAL_AGENT_PATH),
    "exec",
)

print("Sintaxis del agente experimental:", "válida")

# ============================================================
# 6. LIMPIAR CACHÉ DE MÓDULOS
# ============================================================

modules_to_clear = [
    "src.agents.draft_writing_agent_hybrid_experimental",
    "src.adapters.draft_writing_hybrid_runtime",
    "src.tools.draft_writing.retrieval",
    "src.tools.draft_writing.hybrid_retrieval",
]

for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

# ============================================================
# 7. IMPORTAR Y VERIFICAR LA FUNCIÓN EXPERIMENTAL
# ============================================================

from src.tools.draft_writing.retrieval import (
    retrieve_section_evidence_hybrid_experimental,
)

experimental_agent_module = importlib.import_module(
    "src.agents."
    "draft_writing_agent_hybrid_experimental"
)

hybrid_runtime = importlib.import_module(
    "src.adapters.draft_writing_hybrid_runtime"
)

print("\nMódulo del agente:")
print(experimental_agent_module.__file__)

print("\nRuntime experimental:")
print(hybrid_runtime.__file__)

# Verificar que el nombre exista realmente en los globals
# del módulo donde se ejecuta DraftWritingAgent.execute.

function_in_agent_globals = hasattr(
    experimental_agent_module,
    "retrieve_section_evidence_hybrid_experimental",
)

print(
    "\nFunción disponible en globals del agente:",
    function_in_agent_globals,
)

assert function_in_agent_globals, (
    "La función experimental todavía no está disponible "
    "en el módulo del agente"
)

agent_global_function = getattr(
    experimental_agent_module,
    "retrieve_section_evidence_hybrid_experimental",
)

assert callable(agent_global_function), (
    "El símbolo experimental no es invocable"
)

# ============================================================
# 8. FUNCIONES AUXILIARES
# ============================================================

def replace_record(instance, **updates):
    """
    Reemplaza campos en modelos Pydantic, dataclasses
    u objetos Python convencionales.
    """

    if hasattr(instance, "model_copy"):
        return instance.model_copy(
            update=updates,
            deep=True,
        )

    if hasattr(instance, "copy"):
        try:
            return instance.copy(
                update=updates,
                deep=True,
            )
        except TypeError:
            pass

    if is_dataclass(instance):
        return dataclass_replace(
            instance,
            **updates,
        )

    cloned = copy.deepcopy(instance)

    for field_name, field_value in updates.items():
        setattr(
            cloned,
            field_name,
            field_value,
        )

    return cloned


def to_serializable(value):
    """
    Convierte AgentResult y objetos relacionados
    en estructuras compatibles con JSON.
    """

    if value is None:
        return None

    if isinstance(value, Enum):
        return value.value

    if isinstance(value, Path):
        return str(value)

    if isinstance(
        value,
        (str, int, float, bool),
    ):
        return value

    if isinstance(value, dict):
        return {
            str(key): to_serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            to_serializable(item)
            for item in value
        ]

    if hasattr(value, "model_dump"):
        try:
            return to_serializable(
                value.model_dump(mode="python")
            )
        except Exception:
            pass

    if is_dataclass(value):
        return to_serializable(
            asdict(value)
        )

    if hasattr(value, "dict"):
        try:
            return to_serializable(
                value.dict()
            )
        except Exception:
            pass

    if hasattr(value, "__dict__"):
        return {
            key: to_serializable(item)
            for key, item in vars(value).items()
            if not key.startswith("_")
        }

    return str(value)


def enum_or_value(value):
    if isinstance(value, Enum):
        return value.value

    if hasattr(value, "value"):
        return value.value

    return str(value)


# ============================================================
# 9. CONSTRUIR AGENTE, INPUT Y CONFIGURACIÓN
# ============================================================

agent, baseline_input, cfg = (
    hybrid_runtime.build_real_draft_execution(
        PROJECT_DIR,
        attempt_number=ATTEMPT_NUMBER,
    )
)

print("\nAgente construido:")
print(agent.__class__.__module__)
print(agent.__class__.__name__)

assert (
    agent.__class__.__module__
    == "src.agents."
       "draft_writing_agent_hybrid_experimental"
), (
    "El runtime no construyó el agente experimental"
)

# ============================================================
# 10. CREAR SALIDA EXPERIMENTAL AISLADA
# ============================================================

EXPERIMENT_OUTPUT_DIR = (
    Path(cfg["experiment_dir"])
    / "05_outputs"
    / "05_draft_experiments"
    / EXPERIMENT_NAME
)

EXPERIMENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

contractual_output_dir = Path(
    cfg["output_dir"]
).resolve()

assert (
    EXPERIMENT_OUTPUT_DIR.resolve()
    != contractual_output_dir
), (
    "La carpeta experimental coincide con "
    "la carpeta contractual"
)

print("\nSalida contractual:")
print(contractual_output_dir)

print("\nSalida experimental:")
print(EXPERIMENT_OUTPUT_DIR)

# ============================================================
# 11. CREAR AGENTINPUT EXPERIMENTAL
# ============================================================

experimental_context = replace_record(
    baseline_input.agent_context,
    output_directory=str(
        EXPERIMENT_OUTPUT_DIR
    ),
)

experimental_policy = dict(
    baseline_input.policy
)

experimental_policy["force_rebuild"] = True
experimental_policy["experimental_run"] = True
experimental_policy["contractual_execution"] = False
experimental_policy["retrieval_variant"] = (
    "hybrid_thematic_candidate24_balanced_3_3_rrf"
)

original_fingerprint = str(
    experimental_policy.get(
        "current_fingerprint",
        "",
    )
)

experimental_policy["current_fingerprint"] = (
    original_fingerprint
    + "::experimental_hybrid_fixed_import_v1"
)

experimental_input = replace_record(
    baseline_input,
    agent_context=experimental_context,
    policy=experimental_policy,
)

print("\nAgentInput experimental")
print(
    "output_directory:",
    experimental_input.agent_context.output_directory,
)
print(
    "force_rebuild:",
    experimental_input.policy.get("force_rebuild"),
)
print(
    "retrieval_variant:",
    experimental_input.policy.get(
        "retrieval_variant"
    ),
)

# ============================================================
# 12. HASH DEL PIPELINESTATE ANTES
# ============================================================

pipeline_state_path = Path(
    cfg["state_path"]
).resolve()

assert pipeline_state_path.is_file(), (
    f"No existe PipelineState: {pipeline_state_path}"
)

state_hash_before = sha256_file_local(
    pipeline_state_path
)

print("\nPipelineState:")
print(pipeline_state_path)
print("SHA-256 antes:", state_hash_before)

# ============================================================
# 13. EJECUTAR AGENTE EXPERIMENTAL
# ============================================================
#
# A partir de esta línea sí puede haber llamadas a OpenAI.
# No se usa execute_draft_transaction ni StateStore.

print("\n" + "=" * 90)
print("EJECUTANDO AGENTE 06 HÍBRIDO CORREGIDO")
print("=" * 90)

result = agent.execute(
    experimental_input
)

print("\nEjecución terminada")

# ============================================================
# 14. VERIFICAR PIPELINESTATE DESPUÉS
# ============================================================

state_hash_after = sha256_file_local(
    pipeline_state_path
)

pipeline_state_modified = (
    state_hash_before != state_hash_after
)

print("\nSHA-256 después:", state_hash_after)
print(
    "PipelineState modificado:",
    pipeline_state_modified,
)

if pipeline_state_modified:
    raise RuntimeError(
        "El PipelineState cambió durante "
        "la ejecución experimental"
    )

# ============================================================
# 15. MOSTRAR RESULTADO
# ============================================================

execution_status = enum_or_value(
    result.execution_status
)

quality_status = enum_or_value(
    result.quality_status
)

transition_action = enum_or_value(
    result.requested_transition.action
)

decision_code = str(
    result.decision.code
)

tool_usage = to_serializable(
    result.tool_usage
)

llm_calls = int(
    tool_usage.get("llm_calls", 0)
    if isinstance(tool_usage, dict)
    else 0
)

openai_invoked = llm_calls > 0

print("\n" + "=" * 90)
print("RESULTADO DEL AGENTE 06 EXPERIMENTAL")
print("=" * 90)

print("execution_status:", execution_status)
print("quality_status:", quality_status)
print("decision_code:", decision_code)
print(
    "requested_transition:",
    transition_action,
)
print("tool_usage:", tool_usage)
print("OpenAI realmente invocado:", openai_invoked)

print("\nFailure reason codes:")
print(
    list(result.failure_reason_codes or ())
)

print("\nWarnings:")

if result.warnings:
    for warning in result.warnings:
        print(to_serializable(warning))
else:
    print([])

# ============================================================
# 16. GUARDAR RESULTADO EXPERIMENTAL
# ============================================================

result_payload = {
    "experiment_type": (
        "agent06_hybrid_retrieval_generation"
    ),
    "contractual_execution": False,
    "pipeline_state_modified": False,
    "attempt_number_created": False,
    "openai_invoked": openai_invoked,
    "llm_calls": llm_calls,
    "baseline_preserved": True,
    "import_error_corrected": True,
    "retrieval_variant": (
        "hybrid_thematic_candidate24_"
        "balanced_3_3_rrf"
    ),
    "project_dir": str(PROJECT_DIR),
    "experiment_id": cfg["experiment_id"],
    "experimental_output_directory": str(
        EXPERIMENT_OUTPUT_DIR
    ),
    "contractual_output_directory": str(
        contractual_output_dir
    ),
    "pipeline_state_path": str(
        pipeline_state_path
    ),
    "pipeline_state_sha256_before": (
        state_hash_before
    ),
    "pipeline_state_sha256_after": (
        state_hash_after
    ),
    "execution_status": execution_status,
    "quality_status": quality_status,
    "decision_code": decision_code,
    "requested_transition": transition_action,
    "agent_result": to_serializable(result),
}

result_file = (
    EXPERIMENT_OUTPUT_DIR
    / "experimental_agent_result.json"
)

result_file.write_text(
    json.dumps(
        result_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nResultado guardado:")
print(result_file)

# ============================================================
# 17. MOSTRAR ARCHIVOS GENERADOS
# ============================================================

print("\n" + "=" * 90)
print("ARCHIVOS GENERADOS")
print("=" * 90)

generated_files = sorted(
    path
    for path in EXPERIMENT_OUTPUT_DIR.rglob("*")
    if path.is_file()
)

for path in generated_files:
    print(
        path.relative_to(
            EXPERIMENT_OUTPUT_DIR
        )
    )

# ============================================================
# 18. VERIFICAR NUEVAMENTE EL BASELINE
# ============================================================

baseline_agent_hash_final = sha256_file_local(
    BASELINE_AGENT_PATH
)

baseline_modified = (
    baseline_agent_hash_before
    != baseline_agent_hash_final
)

assert not baseline_modified, (
    "El agente baseline cambió durante la ejecución"
)

# ============================================================
# 19. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 90)
print("RESUMEN FINAL")
print("=" * 90)

print(
    "Agente usado:",
    agent.__class__.__module__,
)

print(
    "Función experimental importada:",
    function_in_agent_globals,
)

print(
    "Salida aislada:",
    EXPERIMENT_OUTPUT_DIR,
)

print(
    "OpenAI realmente invocado:",
    openai_invoked,
)

print(
    "Llamadas LLM:",
    llm_calls,
)

print(
    "PipelineState modificado:",
    pipeline_state_modified,
)

print(
    "Intento contractual creado:",
    False,
)

print(
    "Agente baseline modificado:",
    baseline_modified,
)

print(
    "Calidad obtenida:",
    quality_status,
)

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte

Hash agente baseline:
544201c3be8763a3ccdfde22f570af3391a0366db11b40918395e08d90ad231c

Import experimental agregado correctamente
Agente baseline modificado: False
Sintaxis del agente experimental: válida

Módulo del agente:
/content/tesis_codigo/src/agents/draft_writing_agent_hybrid_experimental.py

Runtime experimental:
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py

Función disponible en globals del agente: True

Agente construido:
src.agents.draft_writing_agent_hybrid_experimental
DraftWritingAgent

Salida contractual:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft

Salida experimental:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_balanced_3_3_rrf_fixed_import

AgentInput experimental
output_directory: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_balanced_3_

In [62]:
import json
from collections import Counter, defaultdict
from pathlib import Path

# ============================================================
# 1. RUTAS DEL EXPERIMENTO
# ============================================================

EXPERIMENT_DIR = Path(
    "/content/proyecto_estado_arte/"
    "experimento_paper_02/"
    "05_outputs/"
    "05_draft_experiments/"
    "agent06_hybrid_balanced_3_3_rrf_fixed_import"
).resolve()

RAW_DIR = EXPERIMENT_DIR / "raw_section_outputs"
GLOBAL_REPORT_PATH = (
    EXPERIMENT_DIR / "draft_validation_report.json"
)

OUTPUT_PATH = (
    EXPERIMENT_DIR
    / "diagnostic_validation_summary.json"
)

assert EXPERIMENT_DIR.is_dir(), (
    f"No existe la carpeta experimental: {EXPERIMENT_DIR}"
)

assert RAW_DIR.is_dir(), (
    f"No existe raw_section_outputs: {RAW_DIR}"
)

assert GLOBAL_REPORT_PATH.is_file(), (
    f"No existe el reporte global: {GLOBAL_REPORT_PATH}"
)

print("EXPERIMENTO")
print("=" * 100)
print(EXPERIMENT_DIR)

# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

TARGET_VALUES = [
    "0.96",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]


def load_json(path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )


def safe_list(value):
    return value if isinstance(value, list) else []


def normalize_reason(item):
    if isinstance(item, dict):
        return str(
            item.get("reason")
            or item.get("code")
            or item.get("error")
            or "UNKNOWN"
        )

    return str(item)


def shortened(value, limit=220):
    text = str(value).replace("\n", " ").strip()

    if len(text) <= limit:
        return text

    return text[:limit] + "..."


def citation_key(row):
    return (
        str(row.get("source_filename", "")).strip(),
        str(row.get("chunk_id", "")).strip(),
    )


# ============================================================
# 3. CARGAR REPORTE GLOBAL
# ============================================================

global_report = load_json(
    GLOBAL_REPORT_PATH
)

print("\nREPORTE GLOBAL")
print("=" * 100)

print(
    "validation_ok:",
    global_report.get("validation_ok"),
)

print(
    "failed_section:",
    global_report.get("failed_section"),
)

print(
    "section_attempts:",
    global_report.get("section_attempts"),
)

print(
    "published_draft:",
    global_report.get("published_draft"),
)

global_last_errors = safe_list(
    global_report.get("last_attempt_errors")
)

print(
    "last_attempt_errors:",
    len(global_last_errors),
)

# ============================================================
# 4. LOCALIZAR INTENTOS
# ============================================================

validation_files = sorted(
    RAW_DIR.glob("S2_attempt_*_validation.json")
)

rag_trace_files = sorted(
    RAW_DIR.glob("S2_attempt_*_rag_trace.json")
)

raw_output_files = sorted(
    path
    for path in RAW_DIR.glob("S2_attempt_*.txt")
    if "_validation" not in path.name
    and "_rag_trace" not in path.name
)

assert validation_files, (
    "No se encontraron archivos de validación"
)

print("\nARCHIVOS ENCONTRADOS")
print("=" * 100)

print(
    "Validaciones:",
    len(validation_files),
)

print(
    "Trazas RAG:",
    len(rag_trace_files),
)

print(
    "Salidas crudas:",
    len(raw_output_files),
)

# ============================================================
# 5. ANALIZAR CADA INTENTO
# ============================================================

attempt_summaries = []
reason_counter = Counter()
category_counter = Counter()
retrieved_values_by_attempt = {}
retrieved_chunks_by_attempt = {}
citations_by_attempt = {}

for validation_path in validation_files:
    validation = load_json(
        validation_path
    )

    attempt = int(
        validation.get(
            "generation_attempt",
            0,
        )
    )

    rag_path = (
        RAW_DIR
        / f"S2_attempt_{attempt}_rag_trace.json"
    )

    raw_path = (
        RAW_DIR
        / f"S2_attempt_{attempt}.txt"
    )

    rag_trace = (
        load_json(rag_path)
        if rag_path.is_file()
        else {}
    )

    retrieved_chunks = safe_list(
        rag_trace.get("retrieved_chunks")
    )

    all_retrieved_text = "\n".join(
        str(row.get("text", ""))
        for row in retrieved_chunks
    )

    covered_values = {
        value: [
            str(row.get("chunk_id", ""))
            for row in retrieved_chunks
            if value in str(row.get("text", ""))
        ]
        for value in TARGET_VALUES
    }

    retrieved_values_by_attempt[str(attempt)] = (
        covered_values
    )

    retrieved_chunks_by_attempt[str(attempt)] = [
        {
            "source_filename": str(
                row.get("source_filename", "")
            ),
            "chunk_id": str(
                row.get("chunk_id", "")
            ),
            "score": row.get("score"),
            "retrieval_method": str(
                row.get("retrieval_method", "")
            ),
            "target_values": [
                value
                for value in TARGET_VALUES
                if value in str(row.get("text", ""))
            ],
        }
        for row in retrieved_chunks
    ]

    llm_citations = safe_list(
        rag_trace.get("llm_citations")
    )

    normalized_citations = safe_list(
        rag_trace.get("normalized_citations")
    )

    allowed_citations = safe_list(
        rag_trace.get("allowed_citations")
    )

    citations_by_attempt[str(attempt)] = {
        "allowed_citations": allowed_citations,
        "llm_citations": llm_citations,
        "normalized_citations": (
            normalized_citations
        ),
    }

    categories = {
        "validation_errors": safe_list(
            validation.get("validation_errors")
        ),
        "invalid_citations": safe_list(
            validation.get("invalid_citations")
        ),
        "unsupported_claims": safe_list(
            validation.get("unsupported_claims")
        ),
        "substantive_sentences_without_claim": safe_list(
            validation.get(
                "substantive_sentences_without_claim"
            )
        ),
        "substantive_sentences_without_citation": safe_list(
            validation.get(
                "substantive_sentences_without_citation"
            )
        ),
        "claim_sentence_mismatches": safe_list(
            validation.get(
                "claim_sentence_mismatches"
            )
        ),
        "numeric_support_errors": safe_list(
            validation.get(
                "numeric_support_errors"
            )
        ),
    }

    for category, rows in categories.items():
        category_counter[category] += len(rows)

        for item in rows:
            reason_counter[
                normalize_reason(item)
            ] += 1

    raw_text = (
        raw_path.read_text(encoding="utf-8")
        if raw_path.is_file()
        else ""
    )

    attempt_summary = {
        "attempt": attempt,
        "validation_ok": bool(
            validation.get("validation_ok")
        ),
        "word_count": validation.get(
            "word_count"
        ),
        "citation_count": validation.get(
            "citation_count"
        ),
        "error_counts": {
            category: len(rows)
            for category, rows
            in categories.items()
        },
        "covered_target_values": [
            value
            for value, chunk_ids
            in covered_values.items()
            if chunk_ids
        ],
        "retrieved_chunk_count": len(
            retrieved_chunks
        ),
        "allowed_citation_count": len(
            allowed_citations
        ),
        "llm_citation_count": len(
            llm_citations
        ),
        "normalized_citation_count": len(
            normalized_citations
        ),
        "raw_output_chars": len(raw_text),
    }

    attempt_summaries.append(
        attempt_summary
    )

    # --------------------------------------------------------
    # IMPRESIÓN POR INTENTO
    # --------------------------------------------------------

    print("\n")
    print("=" * 100)
    print(f"INTENTO {attempt}")
    print("=" * 100)

    print(
        "validation_ok:",
        validation.get("validation_ok"),
    )

    print(
        "word_count:",
        validation.get("word_count"),
    )

    print(
        "citation_count:",
        validation.get("citation_count"),
    )

    print(
        "retrieved_chunks:",
        len(retrieved_chunks),
    )

    print(
        "llm_citations:",
        len(llm_citations),
    )

    print(
        "normalized_citations:",
        len(normalized_citations),
    )

    print("\nCOBERTURA CUANTITATIVA")

    for value in TARGET_VALUES:
        matching_chunks = covered_values[value]

        print(
            value,
            "→",
            "CUBIERTO" if matching_chunks
            else "NO CUBIERTO",
            matching_chunks,
        )

    print("\nCHUNKS RECUPERADOS")

    for rank, row in enumerate(
        retrieved_chunks,
        start=1,
    ):
        found_values = [
            value
            for value in TARGET_VALUES
            if value in str(row.get("text", ""))
        ]

        print(
            f"{rank:02d}.",
            row.get("chunk_id"),
            "| método:",
            row.get("retrieval_method"),
            "| score:",
            row.get("score"),
            "| valores:",
            found_values,
        )

    print("\nERRORES POR CATEGORÍA")

    for category, rows in categories.items():
        print(
            f"{category}: {len(rows)}"
        )

        for index, item in enumerate(
            rows,
            start=1,
        ):
            print(
                f"  {index:02d}.",
                shortened(item),
            )

# ============================================================
# 6. RESUMEN COMPARATIVO DE INTENTOS
# ============================================================

print("\n")
print("=" * 100)
print("RESUMEN COMPARATIVO")
print("=" * 100)

for summary in attempt_summaries:
    print(
        f"Intento {summary['attempt']}:",
        f"validation_ok={summary['validation_ok']}",
        f"words={summary['word_count']}",
        f"citations={summary['citation_count']}",
        f"retrieved={summary['retrieved_chunk_count']}",
        f"values={summary['covered_target_values']}",
    )

print("\nERRORES ACUMULADOS POR CATEGORÍA")

for category, count in category_counter.most_common():
    print(
        category,
        "→",
        count,
    )

print("\nRAZONES DE ERROR MÁS FRECUENTES")

for reason, count in reason_counter.most_common():
    print(
        reason,
        "→",
        count,
    )

# ============================================================
# 7. IDENTIFICAR EL INTENTO MÁS CERCANO A APROBAR
# ============================================================

def total_relevant_errors(summary):
    counts = summary["error_counts"]

    return sum(
        count
        for category, count in counts.items()
        if category != "validation_errors"
    )


best_attempt = min(
    attempt_summaries,
    key=lambda summary: (
        total_relevant_errors(summary),
        -len(
            summary["covered_target_values"]
        ),
        summary["attempt"],
    ),
)

print("\nINTENTO MÁS CERCANO A APROBAR")
print("=" * 100)

print(
    "Intento:",
    best_attempt["attempt"],
)

print(
    "Errores relevantes:",
    total_relevant_errors(best_attempt),
)

print(
    "Valores cubiertos:",
    best_attempt["covered_target_values"],
)

print(
    "Conteos:",
    best_attempt["error_counts"],
)

# ============================================================
# 8. GUARDAR DIAGNÓSTICO
# ============================================================

diagnostic_payload = {
    "experiment_directory": str(
        EXPERIMENT_DIR
    ),
    "diagnostic_only": True,
    "openai_called": False,
    "pipeline_state_modified": False,
    "global_validation_report": (
        global_report
    ),
    "attempt_summaries": (
        attempt_summaries
    ),
    "retrieved_values_by_attempt": (
        retrieved_values_by_attempt
    ),
    "retrieved_chunks_by_attempt": (
        retrieved_chunks_by_attempt
    ),
    "citations_by_attempt": (
        citations_by_attempt
    ),
    "error_categories_total": dict(
        category_counter
    ),
    "error_reasons_total": dict(
        reason_counter
    ),
    "best_attempt": best_attempt,
}

OUTPUT_PATH.write_text(
    json.dumps(
        diagnostic_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nDIAGNÓSTICO GUARDADO")
print("=" * 100)

print(OUTPUT_PATH)

print("\nOpenAI llamado:", False)
print("PipelineState modificado:", False)
print("Intento contractual creado:", False)

EXPERIMENTO
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_balanced_3_3_rrf_fixed_import

REPORTE GLOBAL
validation_ok: False
failed_section: S2
section_attempts: 3
published_draft: False
last_attempt_errors: 2

ARCHIVOS ENCONTRADOS
Validaciones: 3
Trazas RAG: 3
Salidas crudas: 3


INTENTO 1
validation_ok: False
word_count: 304
citation_count: 6
retrieved_chunks: 8
llm_citations: 4
normalized_citations: 6

COBERTURA CUANTITATIVA
0.96 → NO CUBIERTO []
1.34 → CUBIERTO ['42891eb1891ec233_chunk_0026']
10% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
58.7% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
6.11% → CUBIERTO ['696fb1df0f31a0b4_chunk_0003']
99% → NO CUBIERTO []

CHUNKS RECUPERADOS
01. 696fb1df0f31a0b4_chunk_0015 | método: chroma_restricted | score: 0.6259012222290039 | valores: []
02. 696fb1df0f31a0b4_chunk_0014 | método: chroma_restricted | score: 0.6024516820907593 | valores: []
03. 42891eb1891ec233_chunk_0008 | método: chroma_restric

In [63]:
import sys
import re
import json
import importlib
from pathlib import Path

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()
PROJECT_DIR = Path("/content/proyecto_estado_arte").resolve()

EXPERIMENT_DIR = Path(
    "/content/proyecto_estado_arte/"
    "experimento_paper_02/"
    "05_outputs/"
    "05_draft_experiments/"
    "agent06_hybrid_balanced_3_3_rrf_fixed_import"
).resolve()

RAW_DIR = EXPERIMENT_DIR / "raw_section_outputs"
OUTPUT_FILE = (
    EXPERIMENT_DIR
    / "numeric_origin_diagnostic.json"
)

SECTION_ID = "S2"

TARGET_VALUES = [
    "0.96",
    "99%",
    "0.21",
    "1.34",
    "10%",
    "58.7%",
    "6.11%",
]

assert CODE_ROOT.is_dir(), (
    f"No existe CODE_ROOT: {CODE_ROOT}"
)

assert PROJECT_DIR.is_dir(), (
    f"No existe PROJECT_DIR: {PROJECT_DIR}"
)

assert EXPERIMENT_DIR.is_dir(), (
    f"No existe EXPERIMENT_DIR: {EXPERIMENT_DIR}"
)

assert RAW_DIR.is_dir(), (
    f"No existe RAW_DIR: {RAW_DIR}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

importlib.invalidate_caches()

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)

# ============================================================
# 2. IMPORTAR EL RUNTIME EXPERIMENTAL
# ============================================================

from src.adapters import draft_writing_hybrid_runtime as hybrid_runtime

print("\nRuntime:")
print(hybrid_runtime.__file__)

# ============================================================
# 3. FUNCIONES AUXILIARES
# ============================================================

def safe_str(value):
    return "" if value is None else str(value)


def load_json(path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )


def normalize_for_percent_search(text):
    """
    Conserva el texto original, pero crea una variante donde
    se tolera un espacio entre número y símbolo de porcentaje.
    """
    return re.sub(
        r"(\d)\s+%",
        r"\1%",
        safe_str(text),
    )


def contains_value(text, value):
    normalized = normalize_for_percent_search(text)
    return value in normalized


def find_contexts(text, value, radius=180):
    """
    Devuelve fragmentos alrededor de cada aparición literal.
    """
    normalized = normalize_for_percent_search(text)
    contexts = []

    start = 0

    while True:
        index = normalized.find(value, start)

        if index == -1:
            break

        left = max(0, index - radius)
        right = min(
            len(normalized),
            index + len(value) + radius,
        )

        contexts.append(
            normalized[left:right]
            .replace("\n", " ")
            .strip()
        )

        start = index + len(value)

    return contexts


def extract_numeric_tokens(text):
    """
    Extrae números enteros, decimales y porcentajes.
    Se usa únicamente para diagnóstico.
    """
    normalized = normalize_for_percent_search(text)

    pattern = re.compile(
        r"(?<![\w.])"
        r"[-+]?"
        r"(?:\d+\.\d+|\d+)"
        r"(?:%|°C|kW|MW|W/m²|W/m2)?"
        r"(?![\w.])"
    )

    return sorted(
        set(pattern.findall(normalized))
    )


def serializable(value):
    if value is None:
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, dict):
        return {
            str(key): serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            serializable(item)
            for item in value
        ]

    if hasattr(value, "model_dump"):
        return serializable(
            value.model_dump(mode="python")
        )

    if hasattr(value, "__dict__"):
        return {
            key: serializable(item)
            for key, item in vars(value).items()
            if not key.startswith("_")
        }

    return str(value)


# ============================================================
# 4. CONSTRUIR AGENTE E INPUT SIN EJECUTAR EL LLM
# ============================================================

agent, agent_input, cfg = (
    hybrid_runtime.build_real_draft_execution(
        PROJECT_DIR,
        attempt_number=1,
    )
)

print("\nAgente:")
print(agent.__class__.__module__)
print(agent.__class__.__name__)

assert (
    agent.__class__.__module__
    == "src.agents."
       "draft_writing_agent_hybrid_experimental"
), (
    "No se construyó el agente experimental"
)

# ============================================================
# 5. CARGAR DEPENDENCIAS DEL AGENTE
# ============================================================

agent_module = importlib.import_module(
    "src.agents."
    "draft_writing_agent_hybrid_experimental"
)

validate_draft_dependencies = getattr(
    agent_module,
    "validate_draft_dependencies",
)

bundle = validate_draft_dependencies(
    agent_input
)

outline = bundle["outline"]
chunks_df = bundle["chunks"]

sections = outline.get("sections") or []

section = next(
    (
        item
        for item in sections
        if safe_str(item.get("section_id")).strip()
        == SECTION_ID
    ),
    None,
)

if section is None:
    raise RuntimeError(
        f"No se encontró la sección {SECTION_ID}"
    )

print("\nSección encontrada:")
print(section.get("section_id"))
print(section.get("section_title"))

# ============================================================
# 6. RECONSTRUIR EL CONTEXTO CUANTITATIVO DE S2
# ============================================================

policy = dict(agent_input.policy)

max_quantitative_rows = int(
    policy.get(
        "max_quantitative_rows_per_section",
        12,
    )
)

quantitative_context = agent._quant_context(
    section,
    bundle,
    max_quantitative_rows,
)

quantitative_context_text = safe_str(
    quantitative_context
)

print("\nCONTEXTO CUANTITATIVO")
print("=" * 100)

print(
    "Tipo:",
    type(quantitative_context).__name__,
)

print(
    "Caracteres:",
    len(quantitative_context_text),
)

print(
    "Tokens numéricos:",
    extract_numeric_tokens(
        quantitative_context_text
    ),
)

# ============================================================
# 7. CARGAR LAS TRAZAS RAG Y SALIDAS DEL LLM
# ============================================================

attempt_records = []

for attempt in (1, 2, 3):
    rag_path = (
        RAW_DIR
        / f"{SECTION_ID}_attempt_{attempt}_rag_trace.json"
    )

    validation_path = (
        RAW_DIR
        / f"{SECTION_ID}_attempt_{attempt}_validation.json"
    )

    raw_path = (
        RAW_DIR
        / f"{SECTION_ID}_attempt_{attempt}.txt"
    )

    assert rag_path.is_file(), (
        f"No existe: {rag_path}"
    )

    assert validation_path.is_file(), (
        f"No existe: {validation_path}"
    )

    assert raw_path.is_file(), (
        f"No existe: {raw_path}"
    )

    rag_trace = load_json(rag_path)
    validation = load_json(validation_path)
    raw_output = raw_path.read_text(
        encoding="utf-8"
    )

    retrieved_chunks = (
        rag_trace.get("retrieved_chunks")
        or []
    )

    evidence_text = "\n".join(
        safe_str(row.get("text"))
        for row in retrieved_chunks
    )

    attempt_records.append(
        {
            "attempt": attempt,
            "rag_path": str(rag_path),
            "validation_path": str(
                validation_path
            ),
            "raw_path": str(raw_path),
            "rag_trace": rag_trace,
            "validation": validation,
            "raw_output": raw_output,
            "retrieved_chunks": (
                retrieved_chunks
            ),
            "evidence_text": evidence_text,
        }
    )

# ============================================================
# 8. DETERMINAR EL ORIGEN DE CADA VALOR
# ============================================================

origin_report = {}

print("\n")
print("=" * 100)
print("ORIGEN DE LOS VALORES NUMÉRICOS")
print("=" * 100)

for value in TARGET_VALUES:
    in_quantitative_context = contains_value(
        quantitative_context_text,
        value,
    )

    occurrences_by_attempt = {}

    for record in attempt_records:
        attempt = record["attempt"]

        evidence_chunks = [
            {
                "source_filename": safe_str(
                    row.get("source_filename")
                ),
                "chunk_id": safe_str(
                    row.get("chunk_id")
                ),
                "retrieval_method": safe_str(
                    row.get("retrieval_method")
                ),
            }
            for row in record["retrieved_chunks"]
            if contains_value(
                row.get("text"),
                value,
            )
        ]

        in_raw_output = contains_value(
            record["raw_output"],
            value,
        )

        numeric_errors = (
            record["validation"].get(
                "numeric_support_errors"
            )
            or []
        )

        rejected_by_validator = any(
            value in safe_str(error)
            for error in numeric_errors
        )

        occurrences_by_attempt[
            str(attempt)
        ] = {
            "in_retrieved_evidence": bool(
                evidence_chunks
            ),
            "evidence_chunks": (
                evidence_chunks
            ),
            "in_raw_llm_output": (
                in_raw_output
            ),
            "rejected_by_validator": (
                rejected_by_validator
            ),
            "raw_output_contexts": (
                find_contexts(
                    record["raw_output"],
                    value,
                )
            ),
        }

    origin_report[value] = {
        "in_quantitative_context": (
            in_quantitative_context
        ),
        "quantitative_context_occurrences": (
            find_contexts(
                quantitative_context_text,
                value,
            )
        ),
        "attempts": occurrences_by_attempt,
    }

    print("\nVALOR:", value)
    print("-" * 100)

    print(
        "En contexto cuantitativo:",
        in_quantitative_context,
    )

    for attempt, data in (
        occurrences_by_attempt.items()
    ):
        print(
            f"Intento {attempt}:",
            "evidencia=",
            data["in_retrieved_evidence"],
            "| salida_llm=",
            data["in_raw_llm_output"],
            "| rechazado=",
            data["rejected_by_validator"],
            "| chunks=",
            [
                row["chunk_id"]
                for row
                in data["evidence_chunks"]
            ],
        )

# ============================================================
# 9. MOSTRAR EL CONTEXTO EXACTO DE LOS VALORES RECHAZADOS
# ============================================================

REJECTED_VALUES = [
    "0.96",
    "99%",
    "0.21",
]

print("\n")
print("=" * 100)
print("CONTEXTO EXACTO DE LOS VALORES RECHAZADOS")
print("=" * 100)

for value in REJECTED_VALUES:
    print("\n")
    print("#" * 100)
    print("VALOR:", value)
    print("#" * 100)

    quantitative_contexts = (
        origin_report[value][
            "quantitative_context_occurrences"
        ]
    )

    print("\nEN CONTEXTO CUANTITATIVO:")

    if quantitative_contexts:
        for index, context in enumerate(
            quantitative_contexts,
            start=1,
        ):
            print(
                f"  {index}. {context}"
            )
    else:
        print("  No aparece.")

    for attempt in ("1", "2", "3"):
        contexts = (
            origin_report[value]
            ["attempts"][attempt]
            ["raw_output_contexts"]
        )

        print(
            f"\nEN SALIDA DEL INTENTO {attempt}:"
        )

        if contexts:
            for index, context in enumerate(
                contexts,
                start=1,
            ):
                print(
                    f"  {index}. {context}"
                )
        else:
            print("  No aparece.")

# ============================================================
# 10. COMPARAR UNIVERSOS NUMÉRICOS
# ============================================================

quantitative_numbers = set(
    extract_numeric_tokens(
        quantitative_context_text
    )
)

print("\n")
print("=" * 100)
print("COMPARACIÓN DE UNIVERSOS NUMÉRICOS")
print("=" * 100)

numeric_comparison = {}

for record in attempt_records:
    attempt = record["attempt"]

    evidence_numbers = set(
        extract_numeric_tokens(
            record["evidence_text"]
        )
    )

    output_numbers = set(
        extract_numeric_tokens(
            record["raw_output"]
        )
    )

    output_not_in_evidence = sorted(
        output_numbers - evidence_numbers
    )

    output_in_quant_not_evidence = sorted(
        (
            output_numbers
            & quantitative_numbers
        )
        - evidence_numbers
    )

    output_neither_evidence_nor_quant = sorted(
        output_numbers
        - evidence_numbers
        - quantitative_numbers
    )

    numeric_comparison[str(attempt)] = {
        "evidence_numbers": sorted(
            evidence_numbers
        ),
        "output_numbers": sorted(
            output_numbers
        ),
        "output_not_in_evidence": (
            output_not_in_evidence
        ),
        "output_in_quantitative_context_but_not_evidence": (
            output_in_quant_not_evidence
        ),
        "output_in_neither_evidence_nor_quantitative_context": (
            output_neither_evidence_nor_quant
        ),
    }

    print(f"\nINTENTO {attempt}")
    print("-" * 100)

    print(
        "Números en salida pero no en evidencia:",
        output_not_in_evidence,
    )

    print(
        "De ellos, presentes en contexto cuantitativo:",
        output_in_quant_not_evidence,
    )

    print(
        "Presentes en ninguna de las dos fuentes:",
        output_neither_evidence_nor_quant,
    )

# ============================================================
# 11. CLASIFICAR EL PROBLEMA
# ============================================================

classification = {}

for value in REJECTED_VALUES:
    in_quant = origin_report[value][
        "in_quantitative_context"
    ]

    in_any_evidence = any(
        data["in_retrieved_evidence"]
        for data
        in origin_report[value][
            "attempts"
        ].values()
    )

    in_any_output = any(
        data["in_raw_llm_output"]
        for data
        in origin_report[value][
            "attempts"
        ].values()
    )

    if in_quant and not in_any_evidence:
        diagnosis = (
            "VALIDATOR_CONTEXT_MISMATCH"
        )
    elif (
        not in_quant
        and not in_any_evidence
        and in_any_output
    ):
        diagnosis = (
            "LIKELY_NUMERIC_HALLUCINATION"
        )
    elif in_any_evidence:
        diagnosis = (
            "VALUE_PRESENT_IN_RAG_EVIDENCE"
        )
    else:
        diagnosis = (
            "VALUE_NOT_USED_OR_ORIGIN_UNRESOLVED"
        )

    classification[value] = diagnosis

print("\n")
print("=" * 100)
print("CLASIFICACIÓN PRELIMINAR")
print("=" * 100)

for value, diagnosis in classification.items():
    print(
        value,
        "→",
        diagnosis,
    )

# ============================================================
# 12. GUARDAR REPORTE
# ============================================================

payload = {
    "diagnostic_type": (
        "numeric_origin_and_validator_context"
    ),
    "diagnostic_only": True,
    "openai_called": False,
    "pipeline_state_modified": False,
    "section_id": SECTION_ID,
    "experimental_directory": str(
        EXPERIMENT_DIR
    ),
    "quantitative_context": (
        serializable(
            quantitative_context
        )
    ),
    "quantitative_numeric_tokens": sorted(
        quantitative_numbers
    ),
    "origin_report": origin_report,
    "numeric_comparison": (
        numeric_comparison
    ),
    "classification": classification,
}

OUTPUT_FILE.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n")
print("=" * 100)
print("REPORTE GUARDADO")
print("=" * 100)

print(OUTPUT_FILE)

print("\nOpenAI llamado:", False)
print("PipelineState modificado:", False)
print("Intento contractual creado:", False)

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte
EXPERIMENT_DIR: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_balanced_3_3_rrf_fixed_import

Runtime:
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py

Agente:
src.agents.draft_writing_agent_hybrid_experimental
DraftWritingAgent

Sección encontrada:
S2
Modelos basados en Redes Neuronales Artificiales (ANN) para pronóstico de irradiancia solar

CONTEXTO CUANTITATIVO
Tipo: dict
Caracteres: 10105
Tokens numéricos: ['0.21', '0.96', '1', '1.0', '1.34', '10', '2', '2001', '2008', '2013', '3', '36', '58.7', '58.7%', '6', '6.11', '6.11%', '7', '7.0', '9', '99%', '99.0']


ORIGEN DE LOS VALORES NUMÉRICOS

VALOR: 0.96
----------------------------------------------------------------------------------------------------
En contexto cuantitativo: True
Intento 1: evidencia= False | salida_llm= True | rechazado= True | chunks= []
Intento 2: evidencia= False 

In [64]:
import sys
import inspect
import importlib
from pathlib import Path

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()

assert CODE_ROOT.is_dir(), (
    f"No existe CODE_ROOT: {CODE_ROOT}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

importlib.invalidate_caches()

print("CODE_ROOT:", CODE_ROOT)

# ============================================================
# 2. TÉRMINOS QUE DEBEMOS LOCALIZAR
# ============================================================

SEARCH_TERMS = [
    "UNSUPPORTED_NUMERIC_VALUE",
    "numeric_support_errors",
    "validate_generated_section",
    "validate_numeric",
    "numeric_value",
    "_quant_context",
    "max_quantitative_rows_per_section",
    "build_section_prompt",
]

# ============================================================
# 3. BUSCAR COINCIDENCIAS EN TODO src/
# ============================================================

matches_by_file = {}

for path in sorted((CODE_ROOT / "src").rglob("*.py")):
    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        continue

    lines = text.splitlines()
    file_matches = []

    for line_number, line in enumerate(lines, start=1):
        matching_terms = [
            term
            for term in SEARCH_TERMS
            if term in line
        ]

        if matching_terms:
            file_matches.append(
                {
                    "line_number": line_number,
                    "line": line,
                    "terms": matching_terms,
                }
            )

    if file_matches:
        matches_by_file[path] = file_matches

# ============================================================
# 4. MOSTRAR RESUMEN
# ============================================================

print("\nARCHIVOS ENCONTRADOS")
print("=" * 100)

for path in matches_by_file:
    print(path.relative_to(CODE_ROOT))

# ============================================================
# 5. MOSTRAR CONTEXTO AMPLIO DE CADA COINCIDENCIA
# ============================================================

print("\n")
print("=" * 100)
print("CONTEXTO DE LAS COINCIDENCIAS")
print("=" * 100)

for path, matches in matches_by_file.items():
    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()

    relevant_numbers = sorted(
        {
            match["line_number"]
            for match in matches
        }
    )

    print("\n\nARCHIVO:")
    print(path.relative_to(CODE_ROOT))
    print("-" * 100)

    # Agrupar coincidencias cercanas para no imprimir
    # el mismo bloque varias veces.
    ranges = []

    for number in relevant_numbers:
        start = max(1, number - 25)
        end = min(len(lines), number + 45)

        if not ranges:
            ranges.append([start, end])
            continue

        previous_start, previous_end = ranges[-1]

        if start <= previous_end + 5:
            ranges[-1][1] = max(previous_end, end)
        else:
            ranges.append([start, end])

    for start, end in ranges:
        print(
            f"\n--- Líneas {start} a {end} ---"
        )

        for number in range(start, end + 1):
            marker = (
                ">>"
                if number in relevant_numbers
                else "  "
            )

            print(
                f"{marker} {number:04d}: "
                f"{lines[number - 1]}"
            )

# ============================================================
# 6. IMPORTAR EL AGENTE EXPERIMENTAL
# ============================================================

agent_module_name = (
    "src.agents."
    "draft_writing_agent_hybrid_experimental"
)

if agent_module_name in sys.modules:
    del sys.modules[agent_module_name]

importlib.invalidate_caches()

agent_module = importlib.import_module(
    agent_module_name
)

DraftWritingAgent = getattr(
    agent_module,
    "DraftWritingAgent",
)

print("\n")
print("=" * 100)
print("MÉTODO _quant_context")
print("=" * 100)

quant_method = getattr(
    DraftWritingAgent,
    "_quant_context",
    None,
)

if quant_method is None:
    print("No existe _quant_context")
else:
    print("Firma:")
    print(inspect.signature(quant_method))

    print("\nCódigo:")
    print(inspect.getsource(quant_method))

# ============================================================
# 7. INSPECCIONAR FUNCIONES IMPORTADAS EN EL AGENTE
# ============================================================

FUNCTION_NAMES = [
    "validate_generated_section",
    "build_section_prompt",
    "normalize_generated_section",
]

print("\n")
print("=" * 100)
print("FUNCIONES UTILIZADAS POR EL AGENTE")
print("=" * 100)

for function_name in FUNCTION_NAMES:
    function = getattr(
        agent_module,
        function_name,
        None,
    )

    print("\n\nFUNCIÓN:", function_name)
    print("-" * 100)

    if function is None:
        print("No está disponible en globals del agente")
        continue

    print("Módulo:")
    print(function.__module__)

    print("Firma:")
    print(inspect.signature(function))

    print("\nArchivo:")
    try:
        print(inspect.getfile(function))
    except TypeError:
        print("No disponible")

    print("\nCódigo fuente:")
    try:
        print(inspect.getsource(function))
    except (OSError, TypeError) as error:
        print(
            "No se pudo obtener:",
            type(error).__name__,
            str(error),
        )

# ============================================================
# 8. LOCALIZAR LA LÍNEA EXACTA DEL ERROR
# ============================================================

print("\n")
print("=" * 100)
print("LÍNEAS EXACTAS CON UNSUPPORTED_NUMERIC_VALUE")
print("=" * 100)

unsupported_locations = []

for path, matches in matches_by_file.items():
    for match in matches:
        if (
            "UNSUPPORTED_NUMERIC_VALUE"
            in match["line"]
        ):
            unsupported_locations.append(
                {
                    "path": str(
                        path.relative_to(CODE_ROOT)
                    ),
                    "line_number": (
                        match["line_number"]
                    ),
                    "line": match["line"],
                }
            )

for location in unsupported_locations:
    print(
        f"{location['path']}:"
        f"{location['line_number']}"
    )
    print(location["line"])

if not unsupported_locations:
    print(
        "No se encontró la cadena literal "
        "UNSUPPORTED_NUMERIC_VALUE"
    )

# ============================================================
# 9. RESUMEN
# ============================================================

print("\n")
print("=" * 100)
print("RESUMEN")
print("=" * 100)

print(
    "Archivos relevantes:",
    len(matches_by_file),
)

print(
    "Ubicaciones del error numérico:",
    len(unsupported_locations),
)

print("Archivos modificados:", False)
print("OpenAI llamado:", False)
print("PipelineState modificado:", False)
print("Intento contractual creado:", False)

CODE_ROOT: /content/tesis_codigo

ARCHIVOS ENCONTRADOS
src/agents/draft_writing_agent.py
src/agents/draft_writing_agent_hybrid_experimental.py
src/config/draft_writing_policy_config.py
src/tools/draft_writing/artifacts.py
src/tools/draft_writing/prompting.py
src/tools/draft_writing/validation.py
src/tools/quantitative_extraction/artifacts.py
src/tools/quantitative_extraction/normalization.py


CONTEXTO DE LAS COINCIDENCIAS


ARCHIVO:
src/agents/draft_writing_agent.py
----------------------------------------------------------------------------------------------------

--- Líneas 1 a 110 ---
   0001: from __future__ import annotations
   0002: from datetime import datetime,timezone
   0003: import json
   0004: from pathlib import Path
   0005: import pandas as pd
   0006: from src.contracts.agent_result import *
   0007: from src.contracts.agent_input import ArtifactReference
   0008: from src.state.fingerprints import sha256_file
   0009: from src.tools.draft_writing import *
   0010: 

In [65]:
import sys
import re
import json
import hashlib
import importlib
from pathlib import Path

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()
PROJECT_DIR = Path("/content/proyecto_estado_arte").resolve()

EXPERIMENTAL_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent_hybrid_experimental.py"
)

BASELINE_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent.py"
)

AUGMENTATION_MODULE_PATH = (
    CODE_ROOT
    / "src"
    / "tools"
    / "draft_writing"
    / "quantitative_augmentation.py"
)

SECTION_ID = "S2"
FINAL_TOP_K = 8
QUANTITATIVE_QUOTA = 2

TARGET_VALUES = [
    "0.96",
    "1.34",
    "0.21",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]

assert CODE_ROOT.is_dir(), (
    f"No existe CODE_ROOT: {CODE_ROOT}"
)

assert PROJECT_DIR.is_dir(), (
    f"No existe PROJECT_DIR: {PROJECT_DIR}"
)

assert EXPERIMENTAL_AGENT_PATH.is_file(), (
    f"No existe el agente experimental: {EXPERIMENTAL_AGENT_PATH}"
)

assert BASELINE_AGENT_PATH.is_file(), (
    f"No existe el agente baseline: {BASELINE_AGENT_PATH}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

# ============================================================
# 2. HASH DEL AGENTE BASELINE
# ============================================================

def sha256_file_local(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


baseline_hash_before = sha256_file_local(
    BASELINE_AGENT_PATH
)

print("\nHash baseline antes:")
print(baseline_hash_before)

# ============================================================
# 3. CREAR MÓDULO DE AMPLIACIÓN CUANTITATIVA
# ============================================================

augmentation_code = r'''
from __future__ import annotations

import math
import re
from typing import Any


def _safe_str(value: Any) -> str:
    if value is None:
        return ""

    try:
        if isinstance(value, float) and math.isnan(value):
            return ""
    except TypeError:
        pass

    text = str(value).strip()

    if text.casefold() == "nan":
        return ""

    return text


def _normalize_percent_spacing(text: Any) -> str:
    return re.sub(
        r"(\d)\s+%",
        r"\1%",
        _safe_str(text),
    )


def _candidate_key(
    row: dict[str, Any],
) -> tuple[str, str]:
    return (
        _safe_str(row.get("source_filename")),
        _safe_str(row.get("chunk_id")),
    )


def _split_chunk_ids(value: Any) -> list[str]:
    text = _safe_str(value)

    if not text:
        return []

    parts = re.split(
        r"[;,|\n]+",
        text,
    )

    output = []
    seen = set()

    for part in parts:
        chunk_id = part.strip()

        if not chunk_id:
            continue

        if chunk_id in seen:
            continue

        seen.add(chunk_id)
        output.append(chunk_id)

    return output


def _row_numeric_literals(
    row: dict[str, Any],
) -> list[str]:
    """
    Extrae únicamente los valores explícitos de una fila
    cuantitativa confirmada.

    No toma años, índices o números de otros campos.
    """
    values = []

    for field_name in (
        "value",
        "raw_value",
        "numeric_value",
    ):
        value = _safe_str(
            row.get(field_name)
        )

        if not value:
            continue

        candidates = re.findall(
            r"(?<!\w)"
            r"[+-]?"
            r"(?:\d+\.\d+|\d+)"
            r"%?"
            r"(?!\w)",
            _normalize_percent_spacing(value),
        )

        for candidate in candidates:
            if candidate not in values:
                values.append(candidate)

    return values


def _confirmed_quantitative_rows(
    quantitative_context: dict[str, Any],
) -> list[dict[str, Any]]:
    rows = (
        quantitative_context.get(
            "quantitative_results"
        )
        if isinstance(
            quantitative_context,
            dict,
        )
        else []
    )

    if not isinstance(rows, list):
        return []

    confirmed = []

    for row in rows:
        if not isinstance(row, dict):
            continue

        verification_status = _safe_str(
            row.get("verification_status")
        ).casefold()

        found_in_source_chunk = _safe_str(
            row.get(
                "value_found_in_source_chunk"
            )
        ).casefold()

        is_confirmed = (
            verification_status
            == "confirmed_in_source_chunk"
            or found_in_source_chunk
            in {
                "true",
                "1",
                "yes",
                "sí",
                "si",
            }
        )

        if not is_confirmed:
            continue

        source_filename = _safe_str(
            row.get("source_filename")
        )

        chunk_ids = _split_chunk_ids(
            row.get(
                "source_chunk_ids_checked"
            )
        )

        numeric_literals = (
            _row_numeric_literals(row)
        )

        if (
            not source_filename
            or not chunk_ids
            or not numeric_literals
        ):
            continue

        normalized = dict(row)
        normalized["_source_filename"] = (
            source_filename
        )
        normalized["_chunk_ids"] = chunk_ids
        normalized["_numeric_literals"] = (
            numeric_literals
        )

        confirmed.append(normalized)

    return confirmed


def build_quantitative_chunk_candidates(
    chunks_df,
    quantitative_context,
    *,
    authorized_source_filenames=None,
    max_evidence_chars=18000,
    valid_source_chunk_pairs=None,
):
    """
    Construye candidatos documentales a partir de filas
    cuantitativas verificadas.

    Un chunk solo recibe crédito por un valor cuando el valor
    aparece literalmente en su texto.
    """
    authorized_sources = {
        _safe_str(source)
        for source
        in (
            authorized_source_filenames
            or []
        )
        if _safe_str(source)
    }

    confirmed_rows = (
        _confirmed_quantitative_rows(
            quantitative_context
        )
    )

    requirements_by_pair = {}

    for row in confirmed_rows:
        source_filename = row[
            "_source_filename"
        ]

        if (
            authorized_sources
            and source_filename
            not in authorized_sources
        ):
            continue

        for chunk_id in row["_chunk_ids"]:
            pair = (
                source_filename,
                chunk_id,
            )

            requirements_by_pair.setdefault(
                pair,
                {
                    "numeric_literals": set(),
                    "metrics": set(),
                    "verification_statuses": set(),
                },
            )

            item = requirements_by_pair[pair]

            item["numeric_literals"].update(
                row["_numeric_literals"]
            )

            metric = _safe_str(
                row.get("metric")
            )

            if metric:
                item["metrics"].add(metric)

            verification_status = _safe_str(
                row.get(
                    "verification_status"
                )
            )

            if verification_status:
                item[
                    "verification_statuses"
                ].add(verification_status)

    if not requirements_by_pair:
        return []

    candidates = []

    for _, source_row in chunks_df.iterrows():
        source_filename = _safe_str(
            source_row.get(
                "source_filename"
            )
        )

        chunk_id = _safe_str(
            source_row.get("chunk_id")
        )

        pair = (
            source_filename,
            chunk_id,
        )

        if pair not in requirements_by_pair:
            continue

        if (
            valid_source_chunk_pairs
            is not None
            and pair
            not in valid_source_chunk_pairs
        ):
            continue

        text = _safe_str(
            source_row.get("text")
        )

        normalized_text = (
            _normalize_percent_spacing(text)
        )

        requirements = (
            requirements_by_pair[pair]
        )

        matched_values = sorted(
            value
            for value
            in requirements[
                "numeric_literals"
            ]
            if value in normalized_text
        )

        if not matched_values:
            continue

        # Prioriza chunks que respaldan más valores.
        score = float(
            len(matched_values)
        )

        candidates.append(
            {
                "source_filename": (
                    source_filename
                ),
                "chunk_id": chunk_id,
                "text": text[
                    :max_evidence_chars
                ],
                "score": score,
                "retrieval_method": (
                    "quantitative_confirmed_chunk"
                ),
                "quantitative_values": (
                    matched_values
                ),
                "quantitative_metrics": sorted(
                    requirements["metrics"]
                ),
                "verification_statuses": sorted(
                    requirements[
                        "verification_statuses"
                    ]
                ),
            }
        )

    candidates.sort(
        key=lambda row: (
            -len(
                row.get(
                    "quantitative_values",
                    [],
                )
            ),
            -float(
                row.get("score", 0.0)
            ),
            _safe_str(
                row.get("source_filename")
            ),
            _safe_str(
                row.get("chunk_id")
            ),
        )
    )

    return candidates


def augment_evidence_with_quantitative_chunks(
    base_evidence,
    chunks_df,
    quantitative_context,
    *,
    authorized_source_filenames=None,
    final_top_k=8,
    quantitative_quota=2,
    max_evidence_chars=18000,
    valid_source_chunk_pairs=None,
):
    """
    Combina retrieval híbrido y evidencia cuantitativa citable.

    Política:
    1. Reserva hasta quantitative_quota posiciones para chunks
       cuantitativos confirmados.
    2. Completa el resto con la evidencia híbrida original.
    3. Deduplica por source_filename + chunk_id.
    4. Nunca acepta una fila cuantitativa sin chunk fuente.
    5. Nunca inventa una cita sintética.
    """
    if final_top_k <= 0:
        return []

    quantitative_quota = max(
        0,
        min(
            int(quantitative_quota),
            int(final_top_k),
        ),
    )

    quantitative_candidates = (
        build_quantitative_chunk_candidates(
            chunks_df,
            quantitative_context,
            authorized_source_filenames=(
                authorized_source_filenames
            ),
            max_evidence_chars=(
                max_evidence_chars
            ),
            valid_source_chunk_pairs=(
                valid_source_chunk_pairs
            ),
        )
    )

    selected = []
    selected_keys = set()

    def add_rows(
        rows,
        limit,
        selection_source,
    ):
        added = 0

        for original in rows:
            if added >= limit:
                break

            row = dict(original)
            key = _candidate_key(row)

            if not key[0] or not key[1]:
                continue

            if key in selected_keys:
                continue

            if (
                valid_source_chunk_pairs
                is not None
                and key
                not in valid_source_chunk_pairs
            ):
                continue

            row["selection_source"] = (
                selection_source
            )

            row[
                "hybrid_selection_method"
            ] = (
                "hybrid_plus_confirmed_quantitative"
            )

            selected.append(row)
            selected_keys.add(key)
            added += 1

    # Conserva primero la mayor parte de la evidencia híbrida.
    base_quota = (
        final_top_k
        - quantitative_quota
    )

    add_rows(
        base_evidence,
        base_quota,
        "hybrid_base_quota",
    )

    # Añade chunks cuantitativos verificables.
    add_rows(
        quantitative_candidates,
        quantitative_quota,
        "confirmed_quantitative_quota",
    )

    # Si hubo duplicados o faltaron candidatos cuantitativos,
    # completa con el resto del ranking híbrido.
    remaining = final_top_k - len(selected)

    if remaining > 0:
        add_rows(
            base_evidence,
            remaining,
            "hybrid_completion",
        )

    # Último respaldo: más chunks cuantitativos válidos.
    remaining = final_top_k - len(selected)

    if remaining > 0:
        add_rows(
            quantitative_candidates,
            remaining,
            "quantitative_completion",
        )

    return selected[:final_top_k]
'''

AUGMENTATION_MODULE_PATH.write_text(
    augmentation_code,
    encoding="utf-8",
)

print("\nMódulo creado:")
print(AUGMENTATION_MODULE_PATH)

compile(
    augmentation_code,
    str(AUGMENTATION_MODULE_PATH),
    "exec",
)

print("Sintaxis del módulo:", "válida")

# ============================================================
# 4. MODIFICAR SOLO EL AGENTE EXPERIMENTAL
# ============================================================

agent_code = EXPERIMENTAL_AGENT_PATH.read_text(
    encoding="utf-8"
)

augmentation_import = (
    "from src.tools.draft_writing."
    "quantitative_augmentation import "
    "augment_evidence_with_quantitative_chunks"
)

if augmentation_import not in agent_code:
    import_anchor = (
        "from src.tools.draft_writing.retrieval "
        "import "
        "retrieve_section_evidence_hybrid_experimental"
    )

    if import_anchor not in agent_code:
        raise RuntimeError(
            "No se encontró el import del retrieval "
            "experimental en el agente"
        )

    agent_code = agent_code.replace(
        import_anchor,
        import_anchor
        + "\n"
        + augmentation_import,
        1,
    )

# La copia actual usa una línea compacta.
old_loop_line = (
    "sid=str(section.get('section_id','')).strip();"
    "section_query=build_section_query(section);"
    "evidence=retrieve_section_evidence_hybrid_experimental("
    "section,self.runtime.collection,bundle['chunks'],"
    "int(policy.get('top_k_evidence_per_section',8)),"
    "int(policy.get('max_evidence_chars',18000)));"
    "retrieval_rounds += 1 if section.get('papers_to_use') else 0"
)

new_loop_block = """sid=str(section.get('section_id','')).strip()
                section_query=build_section_query(section)
                quantitative_context=self._quant_context(
                    section,
                    bundle,
                    int(policy.get(
                        'max_quantitative_rows_per_section',
                        12,
                    )),
                )
                base_evidence=retrieve_section_evidence_hybrid_experimental(
                    section,
                    self.runtime.collection,
                    bundle['chunks'],
                    int(policy.get(
                        'top_k_evidence_per_section',
                        8,
                    )),
                    int(policy.get(
                        'max_evidence_chars',
                        18000,
                    )),
                )
                authorized_sources={
                    str(
                        paper.get(
                            'source_filename',
                            '',
                        )
                    ).strip()
                    for paper
                    in (
                        section.get(
                            'papers_to_use',
                        )
                        or []
                    )
                    if isinstance(paper,dict)
                    and str(
                        paper.get(
                            'source_filename',
                            '',
                        )
                    ).strip()
                }
                valid_pairs={
                    (
                        str(row['source_filename']).strip(),
                        str(row['chunk_id']).strip(),
                    )
                    for _,row
                    in bundle['chunks'].iterrows()
                }
                evidence=augment_evidence_with_quantitative_chunks(
                    base_evidence,
                    bundle['chunks'],
                    quantitative_context,
                    authorized_source_filenames=authorized_sources,
                    final_top_k=int(policy.get(
                        'top_k_evidence_per_section',
                        8,
                    )),
                    quantitative_quota=int(policy.get(
                        'quantitative_evidence_quota',
                        2,
                    )),
                    max_evidence_chars=int(policy.get(
                        'max_evidence_chars',
                        18000,
                    )),
                    valid_source_chunk_pairs=valid_pairs,
                )
                retrieval_rounds += (
                    1
                    if section.get('papers_to_use')
                    else 0
                )"""

if old_loop_line in agent_code:
    agent_code = agent_code.replace(
        old_loop_line,
        new_loop_block,
        1,
    )
elif (
    "augment_evidence_with_quantitative_chunks("
    not in agent_code
):
    raise RuntimeError(
        "No se encontró la línea compacta del retrieval "
        "y el agente tampoco parece estar modificado"
    )

old_prompt_call = (
    "prompt=build_section_prompt("
    "section,evidence,"
    "self._quant_context("
    "section,bundle,"
    "int(policy.get("
    "'max_quantitative_rows_per_section',12"
    "))),previous,policy)"
)

new_prompt_call = (
    "prompt=build_section_prompt("
    "section,"
    "evidence,"
    "quantitative_context,"
    "previous,"
    "policy"
    ")"
)

if old_prompt_call in agent_code:
    agent_code = agent_code.replace(
        old_prompt_call,
        new_prompt_call,
        1,
    )

EXPERIMENTAL_AGENT_PATH.write_text(
    agent_code,
    encoding="utf-8",
)

print("\nAgente experimental actualizado:")
print(EXPERIMENTAL_AGENT_PATH)

# ============================================================
# 5. VERIFICAR BASELINE
# ============================================================

baseline_hash_after = sha256_file_local(
    BASELINE_AGENT_PATH
)

baseline_modified = (
    baseline_hash_before
    != baseline_hash_after
)

assert not baseline_modified, (
    "El agente baseline fue modificado"
)

print("Agente baseline modificado:", False)

# ============================================================
# 6. VALIDAR SINTAXIS DEL AGENTE EXPERIMENTAL
# ============================================================

patched_agent_code = (
    EXPERIMENTAL_AGENT_PATH.read_text(
        encoding="utf-8"
    )
)

compile(
    patched_agent_code,
    str(EXPERIMENTAL_AGENT_PATH),
    "exec",
)

assert (
    "augment_evidence_with_quantitative_chunks("
    in patched_agent_code
), "El agente no usa la ampliación cuantitativa"

assert (
    "quantitative_context"
    in patched_agent_code
), "El agente no conserva el contexto cuantitativo"

print("Sintaxis del agente:", "válida")

# ============================================================
# 7. LIMPIAR CACHÉ
# ============================================================

modules_to_clear = [
    "src.tools.draft_writing.quantitative_augmentation",
    "src.tools.draft_writing.hybrid_retrieval",
    "src.tools.draft_writing.retrieval",
    "src.agents.draft_writing_agent_hybrid_experimental",
    "src.adapters.draft_writing_hybrid_runtime",
]

for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

# ============================================================
# 8. IMPORTAR FUNCIONES
# ============================================================

from src.tools.draft_writing.quantitative_augmentation import (
    build_quantitative_chunk_candidates,
    augment_evidence_with_quantitative_chunks,
)

from src.tools.draft_writing.retrieval import (
    retrieve_section_evidence_hybrid_experimental,
)

hybrid_runtime = importlib.import_module(
    "src.adapters.draft_writing_hybrid_runtime"
)

agent, agent_input, cfg = (
    hybrid_runtime.build_real_draft_execution(
        PROJECT_DIR,
        attempt_number=1,
    )
)

print("\nAgente construido:")
print(agent.__class__.__module__)

assert (
    agent.__class__.__module__
    == "src.agents."
       "draft_writing_agent_hybrid_experimental"
), "No se construyó el agente experimental"

# ============================================================
# 9. CARGAR DEPENDENCIAS SIN EJECUTAR LLM
# ============================================================

agent_module = importlib.import_module(
    "src.agents."
    "draft_writing_agent_hybrid_experimental"
)

validate_draft_dependencies = getattr(
    agent_module,
    "validate_draft_dependencies",
)

bundle = validate_draft_dependencies(
    agent_input
)

section = next(
    (
        item
        for item
        in (
            bundle["outline"].get(
                "sections"
            )
            or []
        )
        if str(
            item.get(
                "section_id",
                "",
            )
        ).strip()
        == SECTION_ID
    ),
    None,
)

if section is None:
    raise RuntimeError(
        f"No se encontró la sección {SECTION_ID}"
    )

policy = dict(agent_input.policy)

top_k = int(
    policy.get(
        "top_k_evidence_per_section",
        FINAL_TOP_K,
    )
)

max_evidence_chars = int(
    policy.get(
        "max_evidence_chars",
        18000,
    )
)

quantitative_context = agent._quant_context(
    section,
    bundle,
    int(
        policy.get(
            "max_quantitative_rows_per_section",
            12,
        )
    ),
)

authorized_sources = {
    str(
        paper.get(
            "source_filename",
            "",
        )
    ).strip()
    for paper
    in (
        section.get("papers_to_use")
        or []
    )
    if isinstance(paper, dict)
    and str(
        paper.get(
            "source_filename",
            "",
        )
    ).strip()
}

valid_pairs = {
    (
        str(row["source_filename"]).strip(),
        str(row["chunk_id"]).strip(),
    )
    for _, row
    in bundle["chunks"].iterrows()
}

# ============================================================
# 10. RETRIEVAL HÍBRIDO BASE
# ============================================================

base_evidence = (
    retrieve_section_evidence_hybrid_experimental(
        section,
        agent.runtime.collection,
        bundle["chunks"],
        top_k,
        max_evidence_chars,
    )
)

# ============================================================
# 11. CANDIDATOS CUANTITATIVOS
# ============================================================

quantitative_candidates = (
    build_quantitative_chunk_candidates(
        bundle["chunks"],
        quantitative_context,
        authorized_source_filenames=(
            authorized_sources
        ),
        max_evidence_chars=(
            max_evidence_chars
        ),
        valid_source_chunk_pairs=(
            valid_pairs
        ),
    )
)

# ============================================================
# 12. EVIDENCIA FINAL AMPLIADA
# ============================================================

final_evidence = (
    augment_evidence_with_quantitative_chunks(
        base_evidence,
        bundle["chunks"],
        quantitative_context,
        authorized_source_filenames=(
            authorized_sources
        ),
        final_top_k=top_k,
        quantitative_quota=(
            QUANTITATIVE_QUOTA
        ),
        max_evidence_chars=(
            max_evidence_chars
        ),
        valid_source_chunk_pairs=(
            valid_pairs
        ),
    )
)

# ============================================================
# 13. MOSTRAR CANDIDATOS CUANTITATIVOS
# ============================================================

print("\n")
print("=" * 100)
print("CANDIDATOS CUANTITATIVOS CONFIRMADOS")
print("=" * 100)

for rank, row in enumerate(
    quantitative_candidates,
    start=1,
):
    print(
        f"{rank:02d}.",
        row.get("chunk_id"),
        "| valores:",
        row.get("quantitative_values"),
        "| métricas:",
        row.get("quantitative_metrics"),
    )

# ============================================================
# 14. MOSTRAR EVIDENCIA FINAL
# ============================================================

print("\n")
print("=" * 100)
print("EVIDENCIA FINAL HÍBRIDA + CUANTITATIVA")
print("=" * 100)

coverage = {}

for rank, row in enumerate(
    final_evidence,
    start=1,
):
    text = re.sub(
        r"(\d)\s+%",
        r"\1%",
        str(row.get("text", "")),
    )

    found_values = [
        value
        for value in TARGET_VALUES
        if value in text
    ]

    print(
        f"{rank:02d}.",
        row.get("chunk_id"),
        "| selección:",
        row.get("selection_source"),
        "| método:",
        row.get("retrieval_method"),
        "| valores:",
        found_values,
    )

print("\n")
print("=" * 100)
print("COBERTURA FINAL")
print("=" * 100)

for value in TARGET_VALUES:
    matching_chunks = []

    for row in final_evidence:
        text = re.sub(
            r"(\d)\s+%",
            r"\1%",
            str(row.get("text", "")),
        )

        if value in text:
            matching_chunks.append(
                str(row.get("chunk_id", ""))
            )

    coverage[value] = {
        "covered": bool(matching_chunks),
        "chunks": matching_chunks,
    }

    print(
        value,
        "→",
        "CUBIERTO"
        if matching_chunks
        else "NO CUBIERTO",
        matching_chunks,
    )

covered_count = sum(
    1
    for item in coverage.values()
    if item["covered"]
)

print("\nValores cubiertos:")
print(
    f"{covered_count}/{len(TARGET_VALUES)}"
)

# ============================================================
# 15. GUARDAR DIAGNÓSTICO
# ============================================================

diagnostic_dir = (
    Path(cfg["experiment_dir"])
    / "05_outputs"
    / "05_draft_experiments"
    / "hybrid_quantitative_augmentation_diagnostic"
)

diagnostic_dir.mkdir(
    parents=True,
    exist_ok=True,
)

diagnostic_file = (
    diagnostic_dir
    / f"{SECTION_ID}_evidence.json"
)

payload = {
    "diagnostic_only": True,
    "openai_called": False,
    "pipeline_state_modified": False,
    "baseline_modified": False,
    "section_id": SECTION_ID,
    "selection_method": (
        "hybrid_plus_confirmed_quantitative"
    ),
    "final_top_k": top_k,
    "quantitative_quota": (
        QUANTITATIVE_QUOTA
    ),
    "base_evidence": base_evidence,
    "quantitative_candidates": (
        quantitative_candidates
    ),
    "final_evidence": final_evidence,
    "coverage": coverage,
    "covered_values_count": (
        covered_count
    ),
    "target_values_count": len(
        TARGET_VALUES
    ),
}

diagnostic_file.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n")
print("=" * 100)
print("DIAGNÓSTICO GUARDADO")
print("=" * 100)

print(diagnostic_file)

print("\nOpenAI llamado:", False)
print("PipelineState modificado:", False)
print("Agente baseline modificado:", False)
print("Intento contractual creado:", False)

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte

Hash baseline antes:
544201c3be8763a3ccdfde22f570af3391a0366db11b40918395e08d90ad231c

Módulo creado:
/content/tesis_codigo/src/tools/draft_writing/quantitative_augmentation.py
Sintaxis del módulo: válida

Agente experimental actualizado:
/content/tesis_codigo/src/agents/draft_writing_agent_hybrid_experimental.py
Agente baseline modificado: False
Sintaxis del agente: válida

Agente construido:
src.agents.draft_writing_agent_hybrid_experimental


CANDIDATOS CUANTITATIVOS CONFIRMADOS
01. 42891eb1891ec233_chunk_0023 | valores: ['0.21', '0.96', '1.34', '2', '7'] | métricas: ['MBE', 'R', 'RMSE', 'neurons_hidden_layer']
02. 42891eb1891ec233_chunk_0022 | valores: ['0.96', '1.34', '2', '7'] | métricas: ['MBE', 'R', 'RMSE', 'neurons_hidden_layer']
03. 696fb1df0f31a0b4_chunk_0003 | valores: ['58.7', '58.7%', '6.11', '6.11%'] | métricas: ['MAPE', 'maximum_error', 'percentage_predictions_error_less_than_10%']
04. 42891eb1

In [66]:
import sys
import re
import json
import hashlib
import importlib
from pathlib import Path

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path("/content/tesis_codigo").resolve()
PROJECT_DIR = Path("/content/proyecto_estado_arte").resolve()

AUGMENTATION_MODULE_PATH = (
    CODE_ROOT
    / "src"
    / "tools"
    / "draft_writing"
    / "quantitative_augmentation.py"
)

EXPERIMENTAL_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent_hybrid_experimental.py"
)

BASELINE_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent.py"
)

SECTION_ID = "S2"
QUANTITATIVE_QUOTA = 2

TARGET_VALUES = [
    "0.96",
    "1.34",
    "0.21",
    "10%",
    "58.7%",
    "6.11%",
    "99%",
]

assert CODE_ROOT.is_dir(), (
    f"No existe CODE_ROOT: {CODE_ROOT}"
)

assert PROJECT_DIR.is_dir(), (
    f"No existe PROJECT_DIR: {PROJECT_DIR}"
)

assert AUGMENTATION_MODULE_PATH.is_file(), (
    f"No existe el módulo cuantitativo: "
    f"{AUGMENTATION_MODULE_PATH}"
)

assert EXPERIMENTAL_AGENT_PATH.is_file(), (
    f"No existe el agente experimental: "
    f"{EXPERIMENTAL_AGENT_PATH}"
)

assert BASELINE_AGENT_PATH.is_file(), (
    f"No existe el agente baseline: "
    f"{BASELINE_AGENT_PATH}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def sha256_file_local(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def normalize_percent_spacing(text):
    return re.sub(
        r"(\d)\s+%",
        r"\1%",
        "" if text is None else str(text),
    )


baseline_hash_before = sha256_file_local(
    BASELINE_AGENT_PATH
)

print("\nHash baseline antes:")
print(baseline_hash_before)

# ============================================================
# 3. AGREGAR FUNCIÓN GREEDY AL MÓDULO CUANTITATIVO
# ============================================================

module_code = AUGMENTATION_MODULE_PATH.read_text(
    encoding="utf-8"
)

greedy_function_code = r'''


def augment_evidence_with_quantitative_chunks_greedy(
    base_evidence,
    chunks_df,
    quantitative_context,
    *,
    authorized_source_filenames=None,
    final_top_k=8,
    quantitative_quota=2,
    max_evidence_chars=18000,
    valid_source_chunk_pairs=None,
):
    """
    Combina evidencia híbrida y chunks cuantitativos confirmados
    mediante cobertura marginal.

    Política:
    1. Conserva final_top_k - quantitative_quota evidencias base.
    2. Detecta qué valores cuantitativos ya están cubiertos.
    3. Selecciona cada chunk cuantitativo según la cantidad de
       valores todavía no cubiertos que añade.
    4. Favorece diversidad de fuentes cuando existe empate.
    5. No acepta chunks sin coincidencia literal del valor.
    """

    if final_top_k <= 0:
        return []

    quantitative_quota = max(
        0,
        min(
            int(quantitative_quota),
            int(final_top_k),
        ),
    )

    quantitative_candidates = (
        build_quantitative_chunk_candidates(
            chunks_df,
            quantitative_context,
            authorized_source_filenames=(
                authorized_source_filenames
            ),
            max_evidence_chars=(
                max_evidence_chars
            ),
            valid_source_chunk_pairs=(
                valid_source_chunk_pairs
            ),
        )
    )

    selected = []
    selected_keys = set()
    selected_sources = set()

    def key_of(row):
        return (
            _safe_str(
                row.get("source_filename")
            ),
            _safe_str(
                row.get("chunk_id")
            ),
        )

    def valid_row(row):
        key = key_of(row)

        if not key[0] or not key[1]:
            return False

        if key in selected_keys:
            return False

        if (
            valid_source_chunk_pairs
            is not None
            and key
            not in valid_source_chunk_pairs
        ):
            return False

        return True

    def append_row(
        original,
        selection_source,
    ):
        row = dict(original)
        key = key_of(row)

        row["selection_source"] = (
            selection_source
        )

        row[
            "hybrid_selection_method"
        ] = (
            "hybrid_plus_confirmed_"
            "quantitative_greedy"
        )

        selected.append(row)
        selected_keys.add(key)
        selected_sources.add(key[0])

    # --------------------------------------------------------
    # A. CONSERVAR CUOTA BASE
    # --------------------------------------------------------

    base_quota = (
        int(final_top_k)
        - quantitative_quota
    )

    for original in base_evidence:
        if len(selected) >= base_quota:
            break

        if not valid_row(original):
            continue

        append_row(
            original,
            "hybrid_base_quota",
        )

    # --------------------------------------------------------
    # B. UNIVERSO DE VALORES CUANTITATIVOS CONFIRMADOS
    # --------------------------------------------------------

    quantitative_universe = set()

    for row in quantitative_candidates:
        quantitative_universe.update(
            _safe_str(value)
            for value
            in row.get(
                "quantitative_values",
                [],
            )
            if _safe_str(value)
        )

    # --------------------------------------------------------
    # C. VALORES YA CUBIERTOS POR LA EVIDENCIA BASE
    # --------------------------------------------------------

    covered_values = set()

    for row in selected:
        normalized_text = (
            _normalize_percent_spacing(
                row.get("text")
            )
        )

        for value in quantitative_universe:
            if value in normalized_text:
                covered_values.add(value)

    # --------------------------------------------------------
    # D. SELECCIÓN GREEDY POR COBERTURA MARGINAL
    # --------------------------------------------------------

    quantitative_added = 0

    while (
        quantitative_added
        < quantitative_quota
    ):
        eligible = [
            row
            for row in quantitative_candidates
            if valid_row(row)
        ]

        if not eligible:
            break

        scored = []

        for row in eligible:
            row_values = {
                _safe_str(value)
                for value
                in row.get(
                    "quantitative_values",
                    [],
                )
                if _safe_str(value)
            }

            marginal_values = (
                row_values
                - covered_values
            )

            source_filename = _safe_str(
                row.get("source_filename")
            )

            source_diversity_bonus = (
                1
                if source_filename
                not in selected_sources
                else 0
            )

            scored.append(
                (
                    len(marginal_values),
                    source_diversity_bonus,
                    len(row_values),
                    float(
                        row.get(
                            "score",
                            0.0,
                        )
                        or 0.0
                    ),
                    _safe_str(
                        row.get(
                            "source_filename"
                        )
                    ),
                    _safe_str(
                        row.get("chunk_id")
                    ),
                    row,
                    marginal_values,
                )
            )

        scored.sort(
            key=lambda item: (
                -item[0],
                -item[1],
                -item[2],
                -item[3],
                item[4],
                item[5],
            )
        )

        (
            marginal_count,
            _source_bonus,
            _total_value_count,
            _score,
            _source,
            _chunk,
            best_row,
            marginal_values,
        ) = scored[0]

        # Si no añade nada nuevo, se permite completar solo
        # cuando todavía quedan espacios vacíos.
        append_row(
            best_row,
            (
                "confirmed_quantitative_"
                "greedy_quota"
            ),
        )

        covered_values.update(
            _safe_str(value)
            for value
            in best_row.get(
                "quantitative_values",
                [],
            )
            if _safe_str(value)
        )

        selected[-1][
            "marginal_quantitative_values"
        ] = sorted(marginal_values)

        selected[-1][
            "marginal_quantitative_count"
        ] = int(marginal_count)

        quantitative_added += 1

    # --------------------------------------------------------
    # E. COMPLETAR SI HUBO MENOS CANDIDATOS CUANTITATIVOS
    # --------------------------------------------------------

    for original in base_evidence:
        if len(selected) >= final_top_k:
            break

        if not valid_row(original):
            continue

        append_row(
            original,
            "hybrid_completion",
        )

    for original in quantitative_candidates:
        if len(selected) >= final_top_k:
            break

        if not valid_row(original):
            continue

        append_row(
            original,
            "quantitative_completion",
        )

    return selected[:final_top_k]
'''

function_name = (
    "def "
    "augment_evidence_with_quantitative_chunks_greedy("
)

if function_name not in module_code:
    module_code += greedy_function_code

    AUGMENTATION_MODULE_PATH.write_text(
        module_code,
        encoding="utf-8",
    )

    print(
        "\nFunción greedy agregada al módulo"
    )
else:
    print(
        "\nLa función greedy ya existe"
    )

compile(
    AUGMENTATION_MODULE_PATH.read_text(
        encoding="utf-8"
    ),
    str(AUGMENTATION_MODULE_PATH),
    "exec",
)

print("Sintaxis del módulo:", "válida")

# ============================================================
# 4. MODIFICAR SOLO EL AGENTE EXPERIMENTAL
# ============================================================

agent_code = EXPERIMENTAL_AGENT_PATH.read_text(
    encoding="utf-8"
)

old_import = (
    "from src.tools.draft_writing."
    "quantitative_augmentation import "
    "augment_evidence_with_quantitative_chunks"
)

new_import = (
    "from src.tools.draft_writing."
    "quantitative_augmentation import "
    "augment_evidence_with_quantitative_chunks_greedy"
)

if old_import in agent_code:
    agent_code = agent_code.replace(
        old_import,
        new_import,
        1,
    )

elif new_import not in agent_code:
    raise RuntimeError(
        "No se encontró el import de ampliación "
        "cuantitativa en el agente experimental"
    )

agent_code = agent_code.replace(
    "evidence=augment_evidence_with_quantitative_chunks(",
    (
        "evidence="
        "augment_evidence_with_quantitative_chunks_greedy("
    ),
)

EXPERIMENTAL_AGENT_PATH.write_text(
    agent_code,
    encoding="utf-8",
)

patched_agent_code = (
    EXPERIMENTAL_AGENT_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    "augment_evidence_with_quantitative_chunks_greedy"
    in patched_agent_code
), (
    "El agente experimental no quedó conectado "
    "a la selección greedy"
)

compile(
    patched_agent_code,
    str(EXPERIMENTAL_AGENT_PATH),
    "exec",
)

print(
    "\nAgente experimental conectado a greedy"
)
print("Sintaxis del agente:", "válida")

# ============================================================
# 5. VERIFICAR QUE EL BASELINE SIGUE INTACTO
# ============================================================

baseline_hash_after = sha256_file_local(
    BASELINE_AGENT_PATH
)

baseline_modified = (
    baseline_hash_before
    != baseline_hash_after
)

assert not baseline_modified, (
    "El agente baseline fue modificado"
)

print("Agente baseline modificado:", False)

# ============================================================
# 6. LIMPIAR CACHÉ
# ============================================================

modules_to_clear = [
    (
        "src.tools.draft_writing."
        "quantitative_augmentation"
    ),
    (
        "src.tools.draft_writing."
        "hybrid_retrieval"
    ),
    "src.tools.draft_writing.retrieval",
    (
        "src.agents."
        "draft_writing_agent_hybrid_experimental"
    ),
    (
        "src.adapters."
        "draft_writing_hybrid_runtime"
    ),
]

for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

# ============================================================
# 7. IMPORTAR COMPONENTES ACTUALIZADOS
# ============================================================

from src.tools.draft_writing.quantitative_augmentation import (
    build_quantitative_chunk_candidates,
    augment_evidence_with_quantitative_chunks_greedy,
)

from src.tools.draft_writing.retrieval import (
    retrieve_section_evidence_hybrid_experimental,
)

hybrid_runtime = importlib.import_module(
    "src.adapters.draft_writing_hybrid_runtime"
)

agent, agent_input, cfg = (
    hybrid_runtime.build_real_draft_execution(
        PROJECT_DIR,
        attempt_number=1,
    )
)

print("\nAgente construido:")
print(agent.__class__.__module__)

assert (
    agent.__class__.__module__
    == "src.agents."
       "draft_writing_agent_hybrid_experimental"
), "No se construyó el agente experimental"

# ============================================================
# 8. CARGAR DEPENDENCIAS
# ============================================================

agent_module = importlib.import_module(
    "src.agents."
    "draft_writing_agent_hybrid_experimental"
)

validate_draft_dependencies = getattr(
    agent_module,
    "validate_draft_dependencies",
)

bundle = validate_draft_dependencies(
    agent_input
)

sections = (
    bundle["outline"].get("sections")
    or []
)

section = next(
    (
        item
        for item in sections
        if str(
            item.get("section_id", "")
        ).strip()
        == SECTION_ID
    ),
    None,
)

if section is None:
    raise RuntimeError(
        f"No se encontró la sección {SECTION_ID}"
    )

policy = dict(agent_input.policy)

top_k = int(
    policy.get(
        "top_k_evidence_per_section",
        8,
    )
)

max_evidence_chars = int(
    policy.get(
        "max_evidence_chars",
        18000,
    )
)

quantitative_context = agent._quant_context(
    section,
    bundle,
    int(
        policy.get(
            "max_quantitative_rows_per_section",
            12,
        )
    ),
)

authorized_sources = {
    str(
        paper.get(
            "source_filename",
            "",
        )
    ).strip()
    for paper
    in (
        section.get("papers_to_use")
        or []
    )
    if isinstance(paper, dict)
    and str(
        paper.get(
            "source_filename",
            "",
        )
    ).strip()
}

valid_pairs = {
    (
        str(
            row["source_filename"]
        ).strip(),
        str(
            row["chunk_id"]
        ).strip(),
    )
    for _, row
    in bundle["chunks"].iterrows()
}

# ============================================================
# 9. RECUPERAR EVIDENCIA HÍBRIDA BASE
# ============================================================

base_evidence = (
    retrieve_section_evidence_hybrid_experimental(
        section,
        agent.runtime.collection,
        bundle["chunks"],
        top_k,
        max_evidence_chars,
    )
)

# ============================================================
# 10. CREAR EVIDENCIA FINAL GREEDY
# ============================================================

quantitative_candidates = (
    build_quantitative_chunk_candidates(
        bundle["chunks"],
        quantitative_context,
        authorized_source_filenames=(
            authorized_sources
        ),
        max_evidence_chars=(
            max_evidence_chars
        ),
        valid_source_chunk_pairs=(
            valid_pairs
        ),
    )
)

final_evidence = (
    augment_evidence_with_quantitative_chunks_greedy(
        base_evidence,
        bundle["chunks"],
        quantitative_context,
        authorized_source_filenames=(
            authorized_sources
        ),
        final_top_k=top_k,
        quantitative_quota=(
            QUANTITATIVE_QUOTA
        ),
        max_evidence_chars=(
            max_evidence_chars
        ),
        valid_source_chunk_pairs=(
            valid_pairs
        ),
    )
)

# ============================================================
# 11. MOSTRAR EVIDENCIA FINAL
# ============================================================

print("\n")
print("=" * 100)
print("EVIDENCIA FINAL GREEDY")
print("=" * 100)

coverage = {}

for rank, row in enumerate(
    final_evidence,
    start=1,
):
    text = normalize_percent_spacing(
        row.get("text", "")
    )

    found_values = [
        value
        for value in TARGET_VALUES
        if value in text
    ]

    print(
        f"{rank:02d}.",
        row.get("chunk_id"),
        "| selección:",
        row.get("selection_source"),
        "| método:",
        row.get("retrieval_method"),
        "| marginales:",
        row.get(
            "marginal_quantitative_values",
            [],
        ),
        "| valores:",
        found_values,
    )

# ============================================================
# 12. COBERTURA FINAL
# ============================================================

print("\n")
print("=" * 100)
print("COBERTURA FINAL GREEDY")
print("=" * 100)

for value in TARGET_VALUES:
    matching_chunks = []

    for row in final_evidence:
        text = normalize_percent_spacing(
            row.get("text", "")
        )

        if value in text:
            matching_chunks.append(
                str(
                    row.get("chunk_id", "")
                )
            )

    coverage[value] = {
        "covered": bool(
            matching_chunks
        ),
        "chunks": matching_chunks,
    }

    print(
        value,
        "→",
        (
            "CUBIERTO"
            if matching_chunks
            else "NO CUBIERTO"
        ),
        matching_chunks,
    )

covered_count = sum(
    1
    for item in coverage.values()
    if item["covered"]
)

coverage_rate = (
    covered_count
    / len(TARGET_VALUES)
)

print("\nValores cubiertos:")
print(
    f"{covered_count}/"
    f"{len(TARGET_VALUES)}"
)

print(
    "Cobertura:",
    f"{coverage_rate:.1%}",
)

# ============================================================
# 13. GUARDAR DIAGNÓSTICO
# ============================================================

diagnostic_dir = (
    Path(cfg["experiment_dir"])
    / "05_outputs"
    / "05_draft_experiments"
    / "hybrid_quantitative_greedy_diagnostic"
)

diagnostic_dir.mkdir(
    parents=True,
    exist_ok=True,
)

diagnostic_file = (
    diagnostic_dir
    / f"{SECTION_ID}_evidence.json"
)

payload = {
    "diagnostic_only": True,
    "openai_called": False,
    "pipeline_state_modified": False,
    "baseline_modified": False,
    "section_id": SECTION_ID,
    "selection_method": (
        "hybrid_plus_confirmed_"
        "quantitative_greedy"
    ),
    "final_top_k": top_k,
    "quantitative_quota": (
        QUANTITATIVE_QUOTA
    ),
    "base_evidence": base_evidence,
    "quantitative_candidates": (
        quantitative_candidates
    ),
    "final_evidence": final_evidence,
    "coverage": coverage,
    "covered_values_count": (
        covered_count
    ),
    "target_values_count": len(
        TARGET_VALUES
    ),
    "coverage_rate": coverage_rate,
}

diagnostic_file.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n")
print("=" * 100)
print("DIAGNÓSTICO GUARDADO")
print("=" * 100)

print(diagnostic_file)

print("\nOpenAI llamado:", False)
print("PipelineState modificado:", False)
print("Agente baseline modificado:", False)
print("Intento contractual creado:", False)

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte

Hash baseline antes:
544201c3be8763a3ccdfde22f570af3391a0366db11b40918395e08d90ad231c

Función greedy agregada al módulo
Sintaxis del módulo: válida

Agente experimental conectado a greedy
Sintaxis del agente: válida
Agente baseline modificado: False

Agente construido:
src.agents.draft_writing_agent_hybrid_experimental


EVIDENCIA FINAL GREEDY
01. 696fb1df0f31a0b4_chunk_0015 | selección: hybrid_base_quota | método: chroma_restricted | marginales: [] | valores: []
02. 696fb1df0f31a0b4_chunk_0014 | selección: hybrid_base_quota | método: chroma_restricted | marginales: [] | valores: []
03. 42891eb1891ec233_chunk_0008 | selección: hybrid_base_quota | método: chroma_restricted | marginales: [] | valores: []
04. 12ed2391bde9cd16_chunk_0005 | selección: hybrid_base_quota | método: csv_lexical_ranked_experimental | marginales: [] | valores: []
05. 696fb1df0f31a0b4_chunk_0002 | selección: hybrid_base_quota | método: c

In [67]:
import sys
import json
import copy
import hashlib
import importlib
from enum import Enum
from pathlib import Path
from dataclasses import (
    is_dataclass,
    replace as dataclass_replace,
    asdict,
)

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path(
    "/content/tesis_codigo"
).resolve()

PROJECT_DIR = Path(
    "/content/proyecto_estado_arte"
).resolve()

EXPERIMENT_NAME = (
    "agent06_hybrid_quantitative_greedy_v1"
)

ATTEMPT_NUMBER = 1

BASELINE_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent.py"
)

EXPERIMENTAL_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent_hybrid_experimental.py"
)

assert CODE_ROOT.is_dir(), (
    f"No existe CODE_ROOT: {CODE_ROOT}"
)

assert PROJECT_DIR.is_dir(), (
    f"No existe PROJECT_DIR: {PROJECT_DIR}"
)

assert BASELINE_AGENT_PATH.is_file(), (
    f"No existe el agente baseline: "
    f"{BASELINE_AGENT_PATH}"
)

assert EXPERIMENTAL_AGENT_PATH.is_file(), (
    f"No existe el agente experimental: "
    f"{EXPERIMENTAL_AGENT_PATH}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(CODE_ROOT),
    )

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def sha256_file_local(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def replace_record(instance, **updates):
    """
    Reemplaza campos en modelos Pydantic,
    dataclasses u objetos Python convencionales.
    """

    if hasattr(instance, "model_copy"):
        return instance.model_copy(
            update=updates,
            deep=True,
        )

    if hasattr(instance, "copy"):
        try:
            return instance.copy(
                update=updates,
                deep=True,
            )
        except TypeError:
            pass

    if is_dataclass(instance):
        return dataclass_replace(
            instance,
            **updates,
        )

    cloned = copy.deepcopy(instance)

    for field_name, field_value in (
        updates.items()
    ):
        setattr(
            cloned,
            field_name,
            field_value,
        )

    return cloned


def to_serializable(value):
    """
    Convierte AgentResult y objetos relacionados
    en estructuras compatibles con JSON.
    """

    if value is None:
        return None

    if isinstance(value, Enum):
        return value.value

    if isinstance(value, Path):
        return str(value)

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    if isinstance(value, dict):
        return {
            str(key): to_serializable(item)
            for key, item
            in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return [
            to_serializable(item)
            for item in value
        ]

    if hasattr(value, "model_dump"):
        try:
            return to_serializable(
                value.model_dump(
                    mode="python"
                )
            )
        except Exception:
            pass

    if is_dataclass(value):
        return to_serializable(
            asdict(value)
        )

    if hasattr(value, "dict"):
        try:
            return to_serializable(
                value.dict()
            )
        except Exception:
            pass

    if hasattr(value, "__dict__"):
        return {
            key: to_serializable(item)
            for key, item
            in vars(value).items()
            if not key.startswith("_")
        }

    return str(value)


def enum_or_value(value):
    if isinstance(value, Enum):
        return value.value

    if hasattr(value, "value"):
        return value.value

    return str(value)


# ============================================================
# 3. REGISTRAR HASH DEL AGENTE BASELINE
# ============================================================

baseline_agent_hash_before = (
    sha256_file_local(
        BASELINE_AGENT_PATH
    )
)

print("\nHash agente baseline:")
print(baseline_agent_hash_before)

# ============================================================
# 4. VERIFICAR QUE EL AGENTE EXPERIMENTAL USA GREEDY
# ============================================================

experimental_code = (
    EXPERIMENTAL_AGENT_PATH.read_text(
        encoding="utf-8"
    )
)

required_symbols = [
    (
        "retrieve_section_evidence_"
        "hybrid_experimental"
    ),
    (
        "augment_evidence_with_"
        "quantitative_chunks_greedy"
    ),
    "quantitative_context",
    "quantitative_evidence_quota",
]

print("\nVERIFICACIÓN DEL AGENTE")
print("=" * 90)

for symbol in required_symbols:
    present = (
        symbol in experimental_code
    )

    print(
        "✓" if present else "✗",
        symbol,
    )

    assert present, (
        f"Falta el símbolo requerido: "
        f"{symbol}"
    )

compile(
    experimental_code,
    str(EXPERIMENTAL_AGENT_PATH),
    "exec",
)

print("Sintaxis del agente experimental: válida")

# ============================================================
# 5. LIMPIAR CACHÉ DE MÓDULOS
# ============================================================

modules_to_clear = [
    (
        "src.tools.draft_writing."
        "quantitative_augmentation"
    ),
    (
        "src.tools.draft_writing."
        "hybrid_retrieval"
    ),
    (
        "src.tools.draft_writing."
        "retrieval"
    ),
    (
        "src.agents."
        "draft_writing_agent_hybrid_experimental"
    ),
    (
        "src.adapters."
        "draft_writing_hybrid_runtime"
    ),
]

for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

# ============================================================
# 6. IMPORTAR RUNTIME EXPERIMENTAL ACTUALIZADO
# ============================================================

hybrid_runtime = importlib.import_module(
    "src.adapters."
    "draft_writing_hybrid_runtime"
)

print("\nRuntime experimental:")
print(hybrid_runtime.__file__)

# ============================================================
# 7. CONSTRUIR AGENTE, INPUT Y CONFIGURACIÓN
# ============================================================

agent, baseline_input, cfg = (
    hybrid_runtime.build_real_draft_execution(
        PROJECT_DIR,
        attempt_number=ATTEMPT_NUMBER,
    )
)

print("\nAgente construido:")
print(agent.__class__.__module__)
print(agent.__class__.__name__)

assert (
    agent.__class__.__module__
    == (
        "src.agents."
        "draft_writing_agent_hybrid_experimental"
    )
), (
    "El runtime no construyó el agente "
    "experimental actualizado"
)

# ============================================================
# 8. CREAR CARPETA EXPERIMENTAL AISLADA
# ============================================================

EXPERIMENT_OUTPUT_DIR = (
    Path(cfg["experiment_dir"])
    / "05_outputs"
    / "05_draft_experiments"
    / EXPERIMENT_NAME
)

EXPERIMENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

contractual_output_dir = Path(
    cfg["output_dir"]
).resolve()

assert (
    EXPERIMENT_OUTPUT_DIR.resolve()
    != contractual_output_dir
), (
    "La salida experimental coincide "
    "con la salida contractual"
)

print("\nSalida contractual:")
print(contractual_output_dir)

print("\nSalida experimental:")
print(EXPERIMENT_OUTPUT_DIR)

# ============================================================
# 9. CREAR AGENTINPUT EXPERIMENTAL
# ============================================================

experimental_context = replace_record(
    baseline_input.agent_context,
    output_directory=str(
        EXPERIMENT_OUTPUT_DIR
    ),
)

experimental_policy = dict(
    baseline_input.policy
)

experimental_policy.update(
    {
        "force_rebuild": True,
        "experimental_run": True,
        "contractual_execution": False,
        "retrieval_variant": (
            "hybrid_thematic_candidate24_"
            "balanced_3_3_rrf_plus_"
            "quantitative_greedy"
        ),
        "quantitative_evidence_quota": 2,
    }
)

original_fingerprint = str(
    experimental_policy.get(
        "current_fingerprint",
        "",
    )
)

experimental_policy[
    "current_fingerprint"
] = (
    original_fingerprint
    + "::experimental_hybrid_"
      "quantitative_greedy_v1"
)

experimental_input = replace_record(
    baseline_input,
    agent_context=experimental_context,
    policy=experimental_policy,
)

print("\nAgentInput experimental")
print(
    "output_directory:",
    experimental_input
    .agent_context
    .output_directory,
)

print(
    "force_rebuild:",
    experimental_input.policy.get(
        "force_rebuild"
    ),
)

print(
    "retrieval_variant:",
    experimental_input.policy.get(
        "retrieval_variant"
    ),
)

print(
    "quantitative_evidence_quota:",
    experimental_input.policy.get(
        "quantitative_evidence_quota"
    ),
)

# ============================================================
# 10. HASH DEL PIPELINESTATE ANTES
# ============================================================

pipeline_state_path = Path(
    cfg["state_path"]
).resolve()

assert pipeline_state_path.is_file(), (
    f"No existe PipelineState: "
    f"{pipeline_state_path}"
)

state_hash_before = (
    sha256_file_local(
        pipeline_state_path
    )
)

print("\nPipelineState:")
print(pipeline_state_path)

print(
    "SHA-256 antes:",
    state_hash_before,
)

# ============================================================
# 11. EJECUTAR AGENTE EXPERIMENTAL
# ============================================================
#
# DESDE ESTA LÍNEA SÍ SE LLAMA A OPENAI.
#
# No se usa execute_draft_transaction.
# No se usa StateStore.
# No se persiste requested_transition.

print("\n" + "=" * 90)
print(
    "EJECUTANDO AGENTE 06 "
    "HÍBRIDO + CUANTITATIVO GREEDY"
)
print("=" * 90)

result = agent.execute(
    experimental_input
)

print("\nEjecución terminada")

# ============================================================
# 12. VERIFICAR PIPELINESTATE DESPUÉS
# ============================================================

state_hash_after = (
    sha256_file_local(
        pipeline_state_path
    )
)

pipeline_state_modified = (
    state_hash_before
    != state_hash_after
)

print(
    "\nSHA-256 después:",
    state_hash_after,
)

print(
    "PipelineState modificado:",
    pipeline_state_modified,
)

if pipeline_state_modified:
    raise RuntimeError(
        "El PipelineState cambió durante "
        "la ejecución experimental"
    )

# ============================================================
# 13. EXTRAER RESULTADO
# ============================================================

execution_status = enum_or_value(
    result.execution_status
)

quality_status = enum_or_value(
    result.quality_status
)

decision_code = str(
    result.decision.code
)

transition_action = enum_or_value(
    result.requested_transition.action
)

tool_usage = to_serializable(
    result.tool_usage
)

llm_calls = int(
    tool_usage.get(
        "llm_calls",
        0,
    )
    if isinstance(
        tool_usage,
        dict,
    )
    else 0
)

retrieval_rounds = int(
    tool_usage.get(
        "retrieval_rounds",
        0,
    )
    if isinstance(
        tool_usage,
        dict,
    )
    else 0
)

validation_calls = int(
    tool_usage.get(
        "validation_calls",
        0,
    )
    if isinstance(
        tool_usage,
        dict,
    )
    else 0
)

openai_invoked = (
    llm_calls > 0
)

# ============================================================
# 14. MOSTRAR RESULTADO
# ============================================================

print("\n" + "=" * 90)
print(
    "RESULTADO DEL AGENTE 06 "
    "EXPERIMENTAL GREEDY"
)
print("=" * 90)

print(
    "execution_status:",
    execution_status,
)

print(
    "quality_status:",
    quality_status,
)

print(
    "decision_code:",
    decision_code,
)

print(
    "requested_transition:",
    transition_action,
)

print(
    "retrieval_rounds:",
    retrieval_rounds,
)

print(
    "llm_calls:",
    llm_calls,
)

print(
    "validation_calls:",
    validation_calls,
)

print(
    "OpenAI realmente invocado:",
    openai_invoked,
)

print("\nFailure reason codes:")
print(
    list(
        result.failure_reason_codes
        or ()
    )
)

print("\nWarnings:")

if result.warnings:
    for warning in result.warnings:
        print(
            to_serializable(warning)
        )
else:
    print([])

# ============================================================
# 15. GUARDAR AGENTRESULT EXPERIMENTAL
# ============================================================

result_payload = {
    "experiment_type": (
        "agent06_hybrid_quantitative_"
        "greedy_generation"
    ),
    "contractual_execution": False,
    "pipeline_state_modified": False,
    "attempt_number_created": False,
    "baseline_preserved": True,
    "openai_invoked": openai_invoked,
    "retrieval_variant": (
        "hybrid_thematic_candidate24_"
        "balanced_3_3_rrf_plus_"
        "quantitative_greedy"
    ),
    "quantitative_evidence_quota": 2,
    "diagnostic_coverage_before_generation": {
        "covered_values": 7,
        "target_values": 7,
        "coverage_rate": 1.0,
    },
    "project_dir": str(
        PROJECT_DIR
    ),
    "experiment_id": cfg[
        "experiment_id"
    ],
    "experimental_output_directory": str(
        EXPERIMENT_OUTPUT_DIR
    ),
    "contractual_output_directory": str(
        contractual_output_dir
    ),
    "pipeline_state_path": str(
        pipeline_state_path
    ),
    "pipeline_state_sha256_before": (
        state_hash_before
    ),
    "pipeline_state_sha256_after": (
        state_hash_after
    ),
    "execution_status": (
        execution_status
    ),
    "quality_status": (
        quality_status
    ),
    "decision_code": (
        decision_code
    ),
    "requested_transition": (
        transition_action
    ),
    "tool_usage": tool_usage,
    "agent_result": (
        to_serializable(result)
    ),
}

result_file = (
    EXPERIMENT_OUTPUT_DIR
    / "experimental_agent_result.json"
)

result_file.write_text(
    json.dumps(
        result_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nResultado guardado:")
print(result_file)

# ============================================================
# 16. MOSTRAR ARCHIVOS GENERADOS
# ============================================================

print("\n" + "=" * 90)
print("ARCHIVOS GENERADOS")
print("=" * 90)

generated_files = sorted(
    path
    for path
    in EXPERIMENT_OUTPUT_DIR.rglob("*")
    if path.is_file()
)

for path in generated_files:
    print(
        path.relative_to(
            EXPERIMENT_OUTPUT_DIR
        )
    )

# ============================================================
# 17. VERIFICAR BASELINE AL FINAL
# ============================================================

baseline_agent_hash_after = (
    sha256_file_local(
        BASELINE_AGENT_PATH
    )
)

baseline_modified = (
    baseline_agent_hash_before
    != baseline_agent_hash_after
)

assert not baseline_modified, (
    "El agente baseline cambió durante "
    "la ejecución experimental"
)

# ============================================================
# 18. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 90)
print("RESUMEN FINAL")
print("=" * 90)

print(
    "Agente usado:",
    agent.__class__.__module__,
)

print(
    "Retrieval:",
    experimental_input.policy.get(
        "retrieval_variant"
    ),
)

print(
    "Cobertura diagnóstica previa:",
    "7/7 = 100%",
)

print(
    "OpenAI realmente invocado:",
    openai_invoked,
)

print(
    "Llamadas LLM:",
    llm_calls,
)

print(
    "PipelineState modificado:",
    pipeline_state_modified,
)

print(
    "Intento contractual creado:",
    False,
)

print(
    "Agente baseline modificado:",
    baseline_modified,
)

print(
    "Baseline sobrescrito:",
    False,
)

print(
    "Calidad obtenida:",
    quality_status,
)

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte

Hash agente baseline:
544201c3be8763a3ccdfde22f570af3391a0366db11b40918395e08d90ad231c

VERIFICACIÓN DEL AGENTE
✓ retrieve_section_evidence_hybrid_experimental
✓ augment_evidence_with_quantitative_chunks_greedy
✓ quantitative_context
✓ quantitative_evidence_quota
Sintaxis del agente experimental: válida

Runtime experimental:
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py

Agente construido:
src.agents.draft_writing_agent_hybrid_experimental
DraftWritingAgent

Salida contractual:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft

Salida experimental:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_quantitative_greedy_v1

AgentInput experimental
output_directory: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_quantitative_greedy_v1
force_rebuild: True
retrieval_variant: h

In [68]:
import json
from collections import Counter
from pathlib import Path

# ============================================================
# 1. RUTAS
# ============================================================

EXPERIMENT_DIR = Path(
    "/content/proyecto_estado_arte/"
    "experimento_paper_02/"
    "05_outputs/"
    "05_draft_experiments/"
    "agent06_hybrid_quantitative_greedy_v1"
).resolve()

RAW_DIR = (
    EXPERIMENT_DIR
    / "raw_section_outputs"
)

GLOBAL_REPORT_PATH = (
    EXPERIMENT_DIR
    / "draft_validation_report.json"
)

OUTPUT_PATH = (
    EXPERIMENT_DIR
    / "global_validation_diagnostic.json"
)

assert EXPERIMENT_DIR.is_dir(), (
    f"No existe EXPERIMENT_DIR: {EXPERIMENT_DIR}"
)

assert RAW_DIR.is_dir(), (
    f"No existe RAW_DIR: {RAW_DIR}"
)

assert GLOBAL_REPORT_PATH.is_file(), (
    f"No existe el reporte global: {GLOBAL_REPORT_PATH}"
)

print("EXPERIMENTO")
print("=" * 100)
print(EXPERIMENT_DIR)

# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def load_json(path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def safe_list(value):
    return (
        value
        if isinstance(value, list)
        else []
    )


def safe_dict(value):
    return (
        value
        if isinstance(value, dict)
        else {}
    )


def shortened(value, limit=300):
    text = str(value).replace(
        "\n",
        " ",
    ).strip()

    if len(text) <= limit:
        return text

    return text[:limit] + "..."


def truth_label(value):
    return (
        "OK"
        if bool(value)
        else "FALLA"
    )


# ============================================================
# 3. CARGAR REPORTE GLOBAL
# ============================================================

global_report = load_json(
    GLOBAL_REPORT_PATH
)

print("\n")
print("=" * 100)
print("REPORTE GLOBAL COMPLETO")
print("=" * 100)

print(
    json.dumps(
        global_report,
        indent=2,
        ensure_ascii=False,
    )
)

# ============================================================
# 4. EXTRAER INDICADORES GLOBALES
# ============================================================

global_checks = {
    "validation_ok": global_report.get(
        "validation_ok"
    ),
    "all_section_validations_ok": (
        global_report.get(
            "all_section_validations_ok"
        )
    ),
    "global_length_valid": (
        global_report.get(
            "global_length_valid"
        )
    ),
}

global_counts = {
    "invalid_citation_count": int(
        global_report.get(
            "invalid_citation_count",
            0,
        )
        or 0
    ),
    "numeric_failure_count": int(
        global_report.get(
            "numeric_failure_count",
            0,
        )
        or 0
    ),
    "section_count": int(
        global_report.get(
            "section_count",
            0,
        )
        or 0
    ),
    "total_words": int(
        global_report.get(
            "total_words",
            0,
        )
        or 0
    ),
    "target_total_words": int(
        global_report.get(
            "target_total_words",
            0,
        )
        or 0
    ),
    "configured_min_total_words": int(
        global_report.get(
            "configured_min_total_words",
            0,
        )
        or 0
    ),
    "effective_min_total_words": int(
        global_report.get(
            "effective_min_total_words",
            0,
        )
        or 0
    ),
    "max_total_words": int(
        global_report.get(
            "max_total_words",
            0,
        )
        or 0
    ),
}

global_lists = {
    "invalid_sections": safe_list(
        global_report.get(
            "invalid_sections"
        )
    ),
    "sections_without_valid_citations": (
        safe_list(
            global_report.get(
                "sections_without_valid_citations"
            )
        )
    ),
    "sections_with_low_citation_density": (
        safe_list(
            global_report.get(
                "sections_with_low_citation_density"
            )
        )
    ),
    "sections_with_claim_support_errors": (
        safe_list(
            global_report.get(
                "sections_with_claim_support_errors"
            )
        )
    ),
    "sections_with_quantitative_support_errors": (
        safe_list(
            global_report.get(
                "sections_with_quantitative_support_errors"
            )
        )
    ),
    "sections_outside_word_range": (
        safe_list(
            global_report.get(
                "sections_outside_word_range"
            )
        )
    ),
}

# ============================================================
# 5. MOSTRAR CONDICIONES GLOBALES
# ============================================================

print("\n")
print("=" * 100)
print("CONDICIONES GLOBALES")
print("=" * 100)

for name, value in global_checks.items():
    print(
        f"{name}:",
        value,
        "→",
        truth_label(value),
    )

print("\nCONTEOS GLOBALES")

for name, value in global_counts.items():
    print(
        f"{name}:",
        value,
    )

print("\nLISTAS DE FALLOS")

for name, value in global_lists.items():
    print(
        f"{name}:",
        len(value),
        value,
    )

# ============================================================
# 6. CARGAR LAS VALIDACIONES DE TODAS LAS SECCIONES
# ============================================================

validation_files = sorted(
    RAW_DIR.glob(
        "S*_attempt_*_validation.json"
    )
)

assert validation_files, (
    "No se encontraron validaciones por sección"
)

section_attempts = []

for path in validation_files:
    data = load_json(path)

    section_id = str(
        data.get(
            "section_id",
            "",
        )
    ).strip()

    generation_attempt = int(
        data.get(
            "generation_attempt",
            0,
        )
        or 0
    )

    section_attempts.append(
        {
            "path": str(path),
            "section_id": section_id,
            "generation_attempt": (
                generation_attempt
            ),
            "validation_ok": bool(
                data.get(
                    "validation_ok"
                )
            ),
            "word_count": int(
                data.get(
                    "word_count",
                    0,
                )
                or 0
            ),
            "citation_count": int(
                data.get(
                    "citation_count",
                    0,
                )
                or 0
            ),
            "validation_errors": (
                safe_list(
                    data.get(
                        "validation_errors"
                    )
                )
            ),
            "invalid_citations": (
                safe_list(
                    data.get(
                        "invalid_citations"
                    )
                )
            ),
            "unsupported_claims": (
                safe_list(
                    data.get(
                        "unsupported_claims"
                    )
                )
            ),
            "substantive_sentences_without_claim": (
                safe_list(
                    data.get(
                        "substantive_sentences_without_claim"
                    )
                )
            ),
            "substantive_sentences_without_citation": (
                safe_list(
                    data.get(
                        "substantive_sentences_without_citation"
                    )
                )
            ),
            "claim_sentence_mismatches": (
                safe_list(
                    data.get(
                        "claim_sentence_mismatches"
                    )
                )
            ),
            "numeric_support_errors": (
                safe_list(
                    data.get(
                        "numeric_support_errors"
                    )
                )
            ),
        }
    )

# ============================================================
# 7. MOSTRAR RESULTADO POR SECCIÓN
# ============================================================

print("\n")
print("=" * 100)
print("VALIDACIÓN POR SECCIÓN")
print("=" * 100)

for row in section_attempts:
    print("\n")
    print("-" * 100)

    print(
        "Sección:",
        row["section_id"],
    )

    print(
        "Intento:",
        row["generation_attempt"],
    )

    print(
        "validation_ok:",
        row["validation_ok"],
        "→",
        truth_label(
            row["validation_ok"]
        ),
    )

    print(
        "word_count:",
        row["word_count"],
    )

    print(
        "citation_count:",
        row["citation_count"],
    )

    error_categories = {
        "validation_errors": (
            row["validation_errors"]
        ),
        "invalid_citations": (
            row["invalid_citations"]
        ),
        "unsupported_claims": (
            row["unsupported_claims"]
        ),
        "substantive_sentences_without_claim": (
            row[
                "substantive_sentences_without_claim"
            ]
        ),
        "substantive_sentences_without_citation": (
            row[
                "substantive_sentences_without_citation"
            ]
        ),
        "claim_sentence_mismatches": (
            row[
                "claim_sentence_mismatches"
            ]
        ),
        "numeric_support_errors": (
            row["numeric_support_errors"]
        ),
    }

    for category, errors in (
        error_categories.items()
    ):
        print(
            f"{category}:",
            len(errors),
        )

        for index, error in enumerate(
            errors,
            start=1,
        ):
            print(
                f"  {index:02d}.",
                shortened(error),
            )

# ============================================================
# 8. SUMAR PALABRAS DE LAS SECCIONES GENERADAS
# ============================================================

latest_by_section = {}

for row in section_attempts:
    section_id = row["section_id"]

    previous = latest_by_section.get(
        section_id
    )

    if (
        previous is None
        or row["generation_attempt"]
        > previous["generation_attempt"]
    ):
        latest_by_section[
            section_id
        ] = row

calculated_total_words = sum(
    row["word_count"]
    for row
    in latest_by_section.values()
)

print("\n")
print("=" * 100)
print("COMPROBACIÓN DE LONGITUD")
print("=" * 100)

for section_id in sorted(
    latest_by_section
):
    row = latest_by_section[
        section_id
    ]

    print(
        section_id,
        "→",
        row["word_count"],
        "palabras",
    )

print(
    "\nSuma calculada:",
    calculated_total_words,
)

print(
    "Total del reporte:",
    global_counts["total_words"],
)

print(
    "Mínimo efectivo:",
    global_counts[
        "effective_min_total_words"
    ],
)

print(
    "Máximo permitido:",
    global_counts[
        "max_total_words"
    ],
)

calculated_length_valid = (
    global_counts[
        "effective_min_total_words"
    ]
    <= calculated_total_words
    <= global_counts[
        "max_total_words"
    ]
)

print(
    "Longitud calculada válida:",
    calculated_length_valid,
)

# ============================================================
# 9. IDENTIFICAR TODAS LAS CONDICIONES NEGATIVAS
# ============================================================

failure_reasons = []

if not bool(
    global_report.get(
        "all_section_validations_ok"
    )
):
    failure_reasons.append(
        "ALL_SECTION_VALIDATIONS_NOT_OK"
    )

if int(
    global_report.get(
        "invalid_citation_count",
        0,
    )
    or 0
) > 0:
    failure_reasons.append(
        "INVALID_CITATIONS_PRESENT"
    )

if global_lists[
    "sections_without_valid_citations"
]:
    failure_reasons.append(
        "SECTIONS_WITHOUT_VALID_CITATIONS"
    )

if global_lists[
    "sections_with_low_citation_density"
]:
    failure_reasons.append(
        "LOW_CITATION_DENSITY"
    )

if global_lists[
    "sections_with_claim_support_errors"
]:
    failure_reasons.append(
        "CLAIM_SUPPORT_ERRORS"
    )

if global_lists[
    "sections_with_quantitative_support_errors"
]:
    failure_reasons.append(
        "QUANTITATIVE_SUPPORT_ERRORS"
    )

if int(
    global_report.get(
        "numeric_failure_count",
        0,
    )
    or 0
) > 0:
    failure_reasons.append(
        "NUMERIC_FAILURES_PRESENT"
    )

if not bool(
    global_report.get(
        "global_length_valid"
    )
):
    failure_reasons.append(
        "GLOBAL_LENGTH_INVALID"
    )

if global_lists[
    "sections_outside_word_range"
]:
    failure_reasons.append(
        "SECTIONS_OUTSIDE_WORD_RANGE"
    )

print("\n")
print("=" * 100)
print("CAUSAS EXACTAS DEL RECHAZO GLOBAL")
print("=" * 100)

if failure_reasons:
    for index, reason in enumerate(
        failure_reasons,
        start=1,
    ):
        print(
            f"{index}.",
            reason,
        )
else:
    print(
        "No se detectó una condición negativa "
        "entre las reglas conocidas."
    )

# ============================================================
# 10. DIAGNÓSTICO INTERPRETATIVO
# ============================================================

if failure_reasons == [
    "GLOBAL_LENGTH_INVALID"
]:
    diagnosis = (
        "Las secciones aprobaron individualmente, "
        "pero la suma total de palabras quedó fuera "
        "del intervalo global configurado."
    )

elif failure_reasons == [
    "SECTIONS_OUTSIDE_WORD_RANGE"
]:
    diagnosis = (
        "Las secciones aprobaron su validación documental, "
        "pero al menos una quedó fuera del rango individual "
        "de longitud."
    )

elif set(failure_reasons).issubset(
    {
        "GLOBAL_LENGTH_INVALID",
        "SECTIONS_OUTSIDE_WORD_RANGE",
    }
):
    diagnosis = (
        "El rechazo es exclusivamente de longitud: "
        "la evidencia, las citas, los claims y los "
        "valores numéricos pasaron la validación."
    )

elif (
    "NUMERIC_FAILURES_PRESENT"
    in failure_reasons
    or "QUANTITATIVE_SUPPORT_ERRORS"
    in failure_reasons
):
    diagnosis = (
        "Persisten fallos cuantitativos durante la "
        "reconstrucción de los reportes globales."
    )

elif (
    "INVALID_CITATIONS_PRESENT"
    in failure_reasons
    or "LOW_CITATION_DENSITY"
    in failure_reasons
    or "SECTIONS_WITHOUT_VALID_CITATIONS"
    in failure_reasons
):
    diagnosis = (
        "El rechazo global se relaciona con citas "
        "o densidad de citación."
    )

elif (
    "CLAIM_SUPPORT_ERRORS"
    in failure_reasons
    or "ALL_SECTION_VALIDATIONS_NOT_OK"
    in failure_reasons
):
    diagnosis = (
        "Alguna sección o claim no conserva una "
        "validación documental satisfactoria."
    )

else:
    diagnosis = (
        "Se requiere revisar la estructura completa "
        "del reporte global."
    )

print("\nDIAGNÓSTICO")
print("-" * 100)
print(diagnosis)

# ============================================================
# 11. GUARDAR REPORTE DE DIAGNÓSTICO
# ============================================================

payload = {
    "diagnostic_type": (
        "agent06_global_validation_failure"
    ),
    "diagnostic_only": True,
    "openai_called": False,
    "pipeline_state_modified": False,
    "contractual_attempt_created": False,
    "experiment_directory": str(
        EXPERIMENT_DIR
    ),
    "global_report": global_report,
    "global_checks": global_checks,
    "global_counts": global_counts,
    "global_failure_lists": global_lists,
    "section_attempts": section_attempts,
    "latest_attempt_by_section": (
        latest_by_section
    ),
    "calculated_total_words": (
        calculated_total_words
    ),
    "calculated_length_valid": (
        calculated_length_valid
    ),
    "failure_reasons": failure_reasons,
    "diagnosis": diagnosis,
}

OUTPUT_PATH.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n")
print("=" * 100)
print("DIAGNÓSTICO GUARDADO")
print("=" * 100)

print(OUTPUT_PATH)

print("\nOpenAI llamado:", False)
print("PipelineState modificado:", False)
print("Intento contractual creado:", False)

EXPERIMENTO
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_quantitative_greedy_v1


REPORTE GLOBAL COMPLETO
{
  "validation_ok": false,
  "invalid_citation_count": 0,
  "sections_without_valid_citations": [],
  "sections_with_low_citation_density": [],
  "sections_with_claim_support_errors": [],
  "sections_with_quantitative_support_errors": [],
  "numeric_failure_count": 0,
  "total_words": 1150,
  "target_total_words": 1000,
  "configured_min_total_words": 650,
  "effective_min_total_words": 490,
  "max_total_words": 1400,
  "source_free_organizational_section_count": 1,
  "global_length_valid": true,
  "section_count": 5,
  "all_section_validations_ok": true,
  "open_search_used": false,
  "ground_truth_used": false,
  "sections_outside_word_range": [
    "S2",
    "S3",
    "S4"
  ],
  "stage": "06_agente_redactor",
  "experiment_id": "experimento_paper_02",
  "validation_version": "legacy_notebook06_validation_v1",
  "generation_a

In [70]:
import sys
import re
import json
import hashlib
import importlib
from pathlib import Path

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path(
    "/content/tesis_codigo"
).resolve()

PROJECT_DIR = Path(
    "/content/proyecto_estado_arte"
).resolve()

EXPERIMENT_DIR = Path(
    "/content/proyecto_estado_arte/"
    "experimento_paper_02/"
    "05_outputs/"
    "05_draft_experiments/"
    "agent06_hybrid_quantitative_greedy_v1"
).resolve()

EXPERIMENTAL_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent_hybrid_experimental.py"
)

BASELINE_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent.py"
)

BUDGET_MODULE_PATH = (
    CODE_ROOT
    / "src"
    / "tools"
    / "draft_writing"
    / "source_aware_budgets.py"
)

RAW_DIR = (
    EXPERIMENT_DIR
    / "raw_section_outputs"
)

DIAGNOSTIC_OUTPUT_PATH = (
    EXPERIMENT_DIR
    / "source_aware_budget_diagnostic.json"
)

ORGANIZATIONAL_TARGET_WORDS = 40

assert CODE_ROOT.is_dir(), (
    f"No existe CODE_ROOT: {CODE_ROOT}"
)

assert PROJECT_DIR.is_dir(), (
    f"No existe PROJECT_DIR: {PROJECT_DIR}"
)

assert EXPERIMENT_DIR.is_dir(), (
    f"No existe EXPERIMENT_DIR: {EXPERIMENT_DIR}"
)

assert EXPERIMENTAL_AGENT_PATH.is_file(), (
    f"No existe el agente experimental: "
    f"{EXPERIMENTAL_AGENT_PATH}"
)

assert BASELINE_AGENT_PATH.is_file(), (
    f"No existe el agente baseline: "
    f"{BASELINE_AGENT_PATH}"
)

assert BUDGET_MODULE_PATH.is_file(), (
    f"No existe el módulo de presupuestos: "
    f"{BUDGET_MODULE_PATH}"
)

assert RAW_DIR.is_dir(), (
    f"No existe RAW_DIR: {RAW_DIR}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(CODE_ROOT),
    )

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)

# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def sha256_file_local(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


baseline_hash_before = sha256_file_local(
    BASELINE_AGENT_PATH
)

print("\nHash agente baseline:")
print(baseline_hash_before)

# ============================================================
# 3. VERIFICAR MÓDULO SOURCE-AWARE
# ============================================================

budget_module_code = (
    BUDGET_MODULE_PATH.read_text(
        encoding="utf-8"
    )
)

compile(
    budget_module_code,
    str(BUDGET_MODULE_PATH),
    "exec",
)

assert (
    "def assign_source_aware_section_budgets("
    in budget_module_code
), (
    "El módulo no contiene "
    "assign_source_aware_section_budgets"
)

print("\nMódulo de presupuestos:")
print(BUDGET_MODULE_PATH)
print("Sintaxis del módulo: válida")

# ============================================================
# 4. REPARAR EL AGENTE EXPERIMENTAL
# ============================================================

agent_code = (
    EXPERIMENTAL_AGENT_PATH.read_text(
        encoding="utf-8"
    )
)

budget_import = (
    "from src.tools.draft_writing."
    "source_aware_budgets import "
    "assign_source_aware_section_budgets"
)

retrieval_import = (
    "from src.tools.draft_writing.retrieval "
    "import "
    "retrieve_section_evidence_hybrid_experimental"
)

# ------------------------------------------------------------
# 4.1. Garantizar el import
# ------------------------------------------------------------

if budget_import not in agent_code:
    if retrieval_import not in agent_code:
        raise RuntimeError(
            "No se encontró el import del retrieval "
            "experimental"
        )

    agent_code = agent_code.replace(
        retrieval_import,
        retrieval_import
        + "\n"
        + budget_import,
        1,
    )

# ------------------------------------------------------------
# 4.2. Reemplazar cualquier asignación rota o anterior
# ------------------------------------------------------------

correct_assignment = (
    "policy['section_budgets'] = "
    "assign_source_aware_section_budgets(\n"
    "                sections,\n"
    "                policy.get("
    "'target_total_words', 1000),\n"
    "                organizational_target_words=int(\n"
    "                    policy.get(\n"
    "                        'organizational_target_words',\n"
    "                        40,\n"
    "                    )\n"
    "                ),\n"
    "            )"
)

# Caso A: línea rota actual, desde policy['section_budgets']
# hasta generated=[].
pattern = re.compile(
    r"policy\['section_budgets'\]\s*="
    r".*?"
    r"generated=\[\];all_evidence=\[\];attempt_logs=\{\}",
    flags=re.DOTALL,
)

replacement = (
    correct_assignment
    + "\n"
    + "            "
    + "generated=[];all_evidence=[];attempt_logs={}"
)

if pattern.search(agent_code):
    agent_code = pattern.sub(
        replacement,
        agent_code,
        count=1,
    )
else:
    # Caso B: la asignación está en una sola línea válida
    # pero todavía usa assign_section_budgets.
    old_pattern = re.compile(
        r"policy\['section_budgets'\]\s*="
        r"assign_section_budgets\("
        r"sections,"
        r"policy\.get\("
        r"'target_total_words',\s*1000"
        r"\)"
        r"\)"
    )

    if old_pattern.search(agent_code):
        agent_code = old_pattern.sub(
            correct_assignment,
            agent_code,
            count=1,
        )

    elif (
        "assign_source_aware_section_budgets("
        not in agent_code
    ):
        raise RuntimeError(
            "No se encontró una asignación reconocible "
            "de section_budgets"
        )

EXPERIMENTAL_AGENT_PATH.write_text(
    agent_code,
    encoding="utf-8",
)

print("\nAgente experimental reparado:")
print(EXPERIMENTAL_AGENT_PATH)

# ============================================================
# 5. COMPROBAR SINTAXIS Y CONTENIDO
# ============================================================

repaired_code = (
    EXPERIMENTAL_AGENT_PATH.read_text(
        encoding="utf-8"
    )
)

compile(
    repaired_code,
    str(EXPERIMENTAL_AGENT_PATH),
    "exec",
)

assert budget_import in repaired_code, (
    "Falta el import de presupuestos source-aware"
)

assert (
    "assign_source_aware_section_budgets("
    in repaired_code
), (
    "El agente no utiliza la función source-aware"
)

assert (
    "organizational_target_words=int("
    in repaired_code
), (
    "No quedó configurado organizational_target_words"
)

print("Sintaxis del agente experimental: válida")

# Mostrar el bloque reparado.
print("\nBLOQUE DE PRESUPUESTOS REPARADO")
print("=" * 90)

lines = repaired_code.splitlines()

for index, line in enumerate(
    lines,
    start=1,
):
    if (
        "policy['section_budgets']"
        in line
    ):
        start = max(1, index - 2)
        end = min(
            len(lines),
            index + 14,
        )

        for number in range(
            start,
            end + 1,
        ):
            marker = (
                ">>"
                if number == index
                else "  "
            )

            print(
                f"{marker} {number:04d}: "
                f"{lines[number - 1]}"
            )

        break

# ============================================================
# 6. VERIFICAR BASELINE
# ============================================================

baseline_hash_after_patch = (
    sha256_file_local(
        BASELINE_AGENT_PATH
    )
)

baseline_modified = (
    baseline_hash_before
    != baseline_hash_after_patch
)

assert not baseline_modified, (
    "El agente baseline fue modificado"
)

print("\nAgente baseline modificado:", False)

# ============================================================
# 7. LIMPIAR CACHÉ
# ============================================================

modules_to_clear = [
    (
        "src.tools.draft_writing."
        "source_aware_budgets"
    ),
    (
        "src.tools.draft_writing."
        "quantitative_augmentation"
    ),
    (
        "src.agents."
        "draft_writing_agent_hybrid_experimental"
    ),
    (
        "src.adapters."
        "draft_writing_hybrid_runtime"
    ),
]

for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

# ============================================================
# 8. IMPORTAR COMPONENTES
# ============================================================

from src.tools.draft_writing.source_aware_budgets import (
    assign_source_aware_section_budgets,
)

hybrid_runtime = importlib.import_module(
    "src.adapters."
    "draft_writing_hybrid_runtime"
)

agent, agent_input, cfg = (
    hybrid_runtime.build_real_draft_execution(
        PROJECT_DIR,
        attempt_number=1,
    )
)

print("\nAgente construido:")
print(agent.__class__.__module__)
print(agent.__class__.__name__)

assert (
    agent.__class__.__module__
    == (
        "src.agents."
        "draft_writing_agent_hybrid_experimental"
    )
), (
    "No se construyó el agente experimental"
)

# ============================================================
# 9. CARGAR OUTLINE
# ============================================================

agent_module = importlib.import_module(
    "src.agents."
    "draft_writing_agent_hybrid_experimental"
)

validate_draft_dependencies = getattr(
    agent_module,
    "validate_draft_dependencies",
)

bundle = validate_draft_dependencies(
    agent_input
)

sections = (
    bundle["outline"].get(
        "sections"
    )
    or []
)

target_total_words = int(
    agent_input.policy.get(
        "target_total_words",
        1000,
    )
)

budgets = (
    assign_source_aware_section_budgets(
        sections,
        target_total_words,
        organizational_target_words=(
            ORGANIZATIONAL_TARGET_WORDS
        ),
    )
)

# ============================================================
# 10. LEER CONTEOS DE LA EJECUCIÓN ANTERIOR
# ============================================================

word_counts = {}

for path in sorted(
    RAW_DIR.glob(
        "S*_attempt_*_validation.json"
    )
):
    data = load_json(path)

    section_id = str(
        data.get(
            "section_id",
            "",
        )
    ).strip()

    attempt = int(
        data.get(
            "generation_attempt",
            0,
        )
        or 0
    )

    current = word_counts.get(
        section_id
    )

    if (
        current is None
        or attempt > current["attempt"]
    ):
        word_counts[section_id] = {
            "attempt": attempt,
            "word_count": int(
                data.get(
                    "word_count",
                    0,
                )
                or 0
            ),
        }

global_report = load_json(
    EXPERIMENT_DIR
    / "draft_validation_report.json"
)

generated_llm_total = sum(
    item["word_count"]
    for item in word_counts.values()
)

s1_inferred_words = max(
    0,
    int(
        global_report.get(
            "total_words",
            0,
        )
        or 0
    )
    - generated_llm_total,
)

if "S1" not in word_counts:
    word_counts["S1"] = {
        "attempt": 0,
        "word_count": (
            s1_inferred_words
        ),
    }

# ============================================================
# 11. MOSTRAR PRESUPUESTOS
# ============================================================

print("\n")
print("=" * 100)
print("PRESUPUESTOS SOURCE-AWARE")
print("=" * 100)

budget_diagnostic = {}
outside_new_range = []

for section in sections:
    section_id = str(
        section.get(
            "section_id",
            "",
        )
    ).strip()

    budget = budgets[
        section_id
    ]

    actual_words = int(
        word_counts.get(
            section_id,
            {},
        ).get(
            "word_count",
            0,
        )
    )

    within_range = (
        budget["minimum_words"]
        <= actual_words
        <= budget["maximum_words"]
    )

    if not within_range:
        outside_new_range.append(
            section_id
        )

    budget_diagnostic[
        section_id
    ] = {
        "actual_words": actual_words,
        **budget,
        "within_new_range": (
            within_range
        ),
    }

    print(
        section_id,
        "| tipo:",
        budget["budget_type"],
        "| actual:",
        actual_words,
        "| objetivo:",
        budget["target_words"],
        "| rango:",
        (
            f"{budget['minimum_words']}"
            f"-"
            f"{budget['maximum_words']}"
        ),
        "|",
        (
            "VÁLIDA"
            if within_range
            else "FUERA DE RANGO"
        ),
    )

# ============================================================
# 12. SIMULAR VALIDACIÓN GLOBAL
# ============================================================

original_non_length_checks_ok = all(
    [
        bool(
            global_report.get(
                "all_section_validations_ok"
            )
        ),
        int(
            global_report.get(
                "invalid_citation_count",
                0,
            )
            or 0
        )
        == 0,
        not (
            global_report.get(
                "sections_without_valid_citations"
            )
            or []
        ),
        not (
            global_report.get(
                "sections_with_low_citation_density"
            )
            or []
        ),
        not (
            global_report.get(
                "sections_with_claim_support_errors"
            )
            or []
        ),
        not (
            global_report.get(
                "sections_with_quantitative_support_errors"
            )
            or []
        ),
        int(
            global_report.get(
                "numeric_failure_count",
                0,
            )
            or 0
        )
        == 0,
        bool(
            global_report.get(
                "global_length_valid"
            )
        ),
    ]
)

simulated_validation_ok = (
    original_non_length_checks_ok
    and not outside_new_range
)

print("\n")
print("=" * 100)
print("SIMULACIÓN CON LOS NUEVOS PRESUPUESTOS")
print("=" * 100)

print(
    "Controles documentales originales:",
    original_non_length_checks_ok,
)

print(
    "Secciones fuera del nuevo rango:",
    outside_new_range,
)

print(
    "Validación global simulada:",
    simulated_validation_ok,
)

# ============================================================
# 13. GUARDAR DIAGNÓSTICO
# ============================================================

payload = {
    "diagnostic_only": True,
    "openai_called": False,
    "pipeline_state_modified": False,
    "baseline_modified": False,
    "syntax_error_repaired": True,
    "target_total_words": (
        target_total_words
    ),
    "organizational_target_words": (
        ORGANIZATIONAL_TARGET_WORDS
    ),
    "original_global_report": (
        global_report
    ),
    "new_section_budgets": (
        budget_diagnostic
    ),
    "sections_outside_new_range": (
        outside_new_range
    ),
    "original_non_length_checks_ok": (
        original_non_length_checks_ok
    ),
    "simulated_validation_ok": (
        simulated_validation_ok
    ),
}

DIAGNOSTIC_OUTPUT_PATH.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n")
print("=" * 100)
print("DIAGNÓSTICO GUARDADO")
print("=" * 100)

print(DIAGNOSTIC_OUTPUT_PATH)

print("\nOpenAI llamado:", False)
print("PipelineState modificado:", False)
print("Agente baseline modificado:", False)
print("Intento contractual creado:", False)

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte
EXPERIMENT_DIR: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_quantitative_greedy_v1

Hash agente baseline:
544201c3be8763a3ccdfde22f570af3391a0366db11b40918395e08d90ad231c

Módulo de presupuestos:
/content/tesis_codigo/src/tools/draft_writing/source_aware_budgets.py
Sintaxis del módulo: válida

Agente experimental reparado:
/content/tesis_codigo/src/agents/draft_writing_agent_hybrid_experimental.py
Sintaxis del agente experimental: válida

BLOQUE DE PRESUPUESTOS REPARADO
   0056:             if not isinstance(sections,list) or not sections:raise ValueError('INVALID_OUTLINE_SCHEMA')
   0057:             policy['outline_sections']=sections
>> 0058:             policy['section_budgets'] = assign_source_aware_section_budgets(
   0059:                 sections,
   0060:                 policy.get('target_total_words', 1000),
   0061:                 organizational

In [71]:
import sys
import json
import copy
import hashlib
import importlib
from enum import Enum
from pathlib import Path
from dataclasses import (
    is_dataclass,
    replace as dataclass_replace,
    asdict,
)

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CODE_ROOT = Path(
    "/content/tesis_codigo"
).resolve()

PROJECT_DIR = Path(
    "/content/proyecto_estado_arte"
).resolve()

EXPERIMENT_NAME = (
    "agent06_hybrid_quantitative_greedy_"
    "source_aware_v1"
)

ATTEMPT_NUMBER = 1

BASELINE_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent.py"
)

EXPERIMENTAL_AGENT_PATH = (
    CODE_ROOT
    / "src"
    / "agents"
    / "draft_writing_agent_hybrid_experimental.py"
)

QUANTITATIVE_MODULE_PATH = (
    CODE_ROOT
    / "src"
    / "tools"
    / "draft_writing"
    / "quantitative_augmentation.py"
)

BUDGET_MODULE_PATH = (
    CODE_ROOT
    / "src"
    / "tools"
    / "draft_writing"
    / "source_aware_budgets.py"
)

assert CODE_ROOT.is_dir(), (
    f"No existe CODE_ROOT: {CODE_ROOT}"
)

assert PROJECT_DIR.is_dir(), (
    f"No existe PROJECT_DIR: {PROJECT_DIR}"
)

assert BASELINE_AGENT_PATH.is_file(), (
    f"No existe el agente baseline: "
    f"{BASELINE_AGENT_PATH}"
)

assert EXPERIMENTAL_AGENT_PATH.is_file(), (
    f"No existe el agente experimental: "
    f"{EXPERIMENTAL_AGENT_PATH}"
)

assert QUANTITATIVE_MODULE_PATH.is_file(), (
    f"No existe el módulo cuantitativo: "
    f"{QUANTITATIVE_MODULE_PATH}"
)

assert BUDGET_MODULE_PATH.is_file(), (
    f"No existe el módulo de presupuestos: "
    f"{BUDGET_MODULE_PATH}"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(CODE_ROOT),
    )

print("CODE_ROOT:", CODE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def sha256_file_local(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def replace_record(instance, **updates):
    if hasattr(instance, "model_copy"):
        return instance.model_copy(
            update=updates,
            deep=True,
        )

    if hasattr(instance, "copy"):
        try:
            return instance.copy(
                update=updates,
                deep=True,
            )
        except TypeError:
            pass

    if is_dataclass(instance):
        return dataclass_replace(
            instance,
            **updates,
        )

    cloned = copy.deepcopy(instance)

    for field_name, field_value in updates.items():
        setattr(
            cloned,
            field_name,
            field_value,
        )

    return cloned


def to_serializable(value):
    if value is None:
        return None

    if isinstance(value, Enum):
        return value.value

    if isinstance(value, Path):
        return str(value)

    if isinstance(
        value,
        (str, int, float, bool),
    ):
        return value

    if isinstance(value, dict):
        return {
            str(key): to_serializable(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            to_serializable(item)
            for item in value
        ]

    if hasattr(value, "model_dump"):
        try:
            return to_serializable(
                value.model_dump(
                    mode="python"
                )
            )
        except Exception:
            pass

    if is_dataclass(value):
        return to_serializable(
            asdict(value)
        )

    if hasattr(value, "dict"):
        try:
            return to_serializable(
                value.dict()
            )
        except Exception:
            pass

    if hasattr(value, "__dict__"):
        return {
            key: to_serializable(item)
            for key, item
            in vars(value).items()
            if not key.startswith("_")
        }

    return str(value)


def enum_or_value(value):
    if isinstance(value, Enum):
        return value.value

    if hasattr(value, "value"):
        return value.value

    return str(value)


# ============================================================
# 3. HASH DEL BASELINE
# ============================================================

baseline_hash_before = sha256_file_local(
    BASELINE_AGENT_PATH
)

print("\nHash agente baseline:")
print(baseline_hash_before)

# ============================================================
# 4. VERIFICAR SINTAXIS Y CONEXIONES EXPERIMENTALES
# ============================================================

experimental_code = (
    EXPERIMENTAL_AGENT_PATH.read_text(
        encoding="utf-8"
    )
)

quantitative_code = (
    QUANTITATIVE_MODULE_PATH.read_text(
        encoding="utf-8"
    )
)

budget_code = (
    BUDGET_MODULE_PATH.read_text(
        encoding="utf-8"
    )
)

compile(
    experimental_code,
    str(EXPERIMENTAL_AGENT_PATH),
    "exec",
)

compile(
    quantitative_code,
    str(QUANTITATIVE_MODULE_PATH),
    "exec",
)

compile(
    budget_code,
    str(BUDGET_MODULE_PATH),
    "exec",
)

required_agent_symbols = [
    "retrieve_section_evidence_hybrid_experimental",
    "augment_evidence_with_quantitative_chunks_greedy",
    "assign_source_aware_section_budgets",
    "quantitative_evidence_quota",
    "organizational_target_words",
]

print("\nVERIFICACIÓN DEL AGENTE EXPERIMENTAL")
print("=" * 100)

for symbol in required_agent_symbols:
    present = symbol in experimental_code

    print(
        "✓" if present else "✗",
        symbol,
    )

    assert present, (
        f"Falta el símbolo requerido: {symbol}"
    )

assert (
    "def augment_evidence_with_quantitative_chunks_greedy("
    in quantitative_code
), (
    "No existe la función greedy cuantitativa"
)

assert (
    "def assign_source_aware_section_budgets("
    in budget_code
), (
    "No existe la función source-aware"
)

print("Sintaxis de componentes experimentales: válida")

# ============================================================
# 5. LIMPIAR CACHÉ DE MÓDULOS
# ============================================================

modules_to_clear = [
    (
        "src.tools.draft_writing."
        "source_aware_budgets"
    ),
    (
        "src.tools.draft_writing."
        "quantitative_augmentation"
    ),
    (
        "src.tools.draft_writing."
        "hybrid_retrieval"
    ),
    (
        "src.tools.draft_writing."
        "retrieval"
    ),
    (
        "src.agents."
        "draft_writing_agent_hybrid_experimental"
    ),
    (
        "src.adapters."
        "draft_writing_hybrid_runtime"
    ),
]

for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

# ============================================================
# 6. IMPORTAR RUNTIME EXPERIMENTAL
# ============================================================

hybrid_runtime = importlib.import_module(
    "src.adapters."
    "draft_writing_hybrid_runtime"
)

print("\nRuntime experimental:")
print(hybrid_runtime.__file__)

# ============================================================
# 7. CONSTRUIR AGENTE
# ============================================================

agent, baseline_input, cfg = (
    hybrid_runtime.build_real_draft_execution(
        PROJECT_DIR,
        attempt_number=ATTEMPT_NUMBER,
    )
)

print("\nAgente construido:")
print(agent.__class__.__module__)
print(agent.__class__.__name__)

assert (
    agent.__class__.__module__
    == (
        "src.agents."
        "draft_writing_agent_hybrid_experimental"
    )
), (
    "El runtime no construyó el agente experimental"
)

# ============================================================
# 8. CREAR SALIDA EXPERIMENTAL AISLADA
# ============================================================

EXPERIMENT_OUTPUT_DIR = (
    Path(cfg["experiment_dir"])
    / "05_outputs"
    / "05_draft_experiments"
    / EXPERIMENT_NAME
)

EXPERIMENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

contractual_output_dir = Path(
    cfg["output_dir"]
).resolve()

assert (
    EXPERIMENT_OUTPUT_DIR.resolve()
    != contractual_output_dir
), (
    "La salida experimental coincide "
    "con la salida contractual"
)

print("\nSalida contractual:")
print(contractual_output_dir)

print("\nSalida experimental:")
print(EXPERIMENT_OUTPUT_DIR)

# ============================================================
# 9. CREAR AGENTINPUT EXPERIMENTAL
# ============================================================

experimental_context = replace_record(
    baseline_input.agent_context,
    output_directory=str(
        EXPERIMENT_OUTPUT_DIR
    ),
)

experimental_policy = dict(
    baseline_input.policy
)

experimental_policy.update(
    {
        "force_rebuild": True,
        "experimental_run": True,
        "contractual_execution": False,
        "retrieval_variant": (
            "hybrid_thematic_candidate24_"
            "balanced_3_3_rrf_plus_"
            "quantitative_greedy_plus_"
            "source_aware_budgets"
        ),
        "quantitative_evidence_quota": 2,
        "organizational_target_words": 40,
    }
)

original_fingerprint = str(
    experimental_policy.get(
        "current_fingerprint",
        "",
    )
)

experimental_policy[
    "current_fingerprint"
] = (
    original_fingerprint
    + "::experimental_hybrid_quantitative_"
      "greedy_source_aware_v1"
)

experimental_input = replace_record(
    baseline_input,
    agent_context=experimental_context,
    policy=experimental_policy,
)

print("\nAgentInput experimental")
print(
    "output_directory:",
    experimental_input
    .agent_context
    .output_directory,
)

print(
    "force_rebuild:",
    experimental_input.policy.get(
        "force_rebuild"
    ),
)

print(
    "retrieval_variant:",
    experimental_input.policy.get(
        "retrieval_variant"
    ),
)

print(
    "quantitative_evidence_quota:",
    experimental_input.policy.get(
        "quantitative_evidence_quota"
    ),
)

print(
    "organizational_target_words:",
    experimental_input.policy.get(
        "organizational_target_words"
    ),
)

# ============================================================
# 10. HASH DEL PIPELINESTATE ANTES
# ============================================================

pipeline_state_path = Path(
    cfg["state_path"]
).resolve()

assert pipeline_state_path.is_file(), (
    f"No existe PipelineState: "
    f"{pipeline_state_path}"
)

state_hash_before = sha256_file_local(
    pipeline_state_path
)

print("\nPipelineState:")
print(pipeline_state_path)

print(
    "SHA-256 antes:",
    state_hash_before,
)

# ============================================================
# 11. EJECUTAR AGENTE
# ============================================================
#
# DESDE ESTA LÍNEA SÍ SE LLAMA A OPENAI.
#
# No se usa StateStore.
# No se persiste requested_transition.
# No se crea un intento contractual.

print("\n" + "=" * 100)
print(
    "EJECUTANDO AGENTE 06 "
    "HÍBRIDO + CUANTITATIVO GREEDY "
    "+ PRESUPUESTOS SOURCE-AWARE"
)
print("=" * 100)

result = agent.execute(
    experimental_input
)

print("\nEjecución terminada")

# ============================================================
# 12. VERIFICAR PIPELINESTATE DESPUÉS
# ============================================================

state_hash_after = sha256_file_local(
    pipeline_state_path
)

pipeline_state_modified = (
    state_hash_before
    != state_hash_after
)

print(
    "\nSHA-256 después:",
    state_hash_after,
)

print(
    "PipelineState modificado:",
    pipeline_state_modified,
)

if pipeline_state_modified:
    raise RuntimeError(
        "El PipelineState cambió durante "
        "la ejecución experimental"
    )

# ============================================================
# 13. EXTRAER RESULTADO
# ============================================================

execution_status = enum_or_value(
    result.execution_status
)

quality_status = enum_or_value(
    result.quality_status
)

decision_code = str(
    result.decision.code
)

transition_action = enum_or_value(
    result.requested_transition.action
)

tool_usage = to_serializable(
    result.tool_usage
)

llm_calls = int(
    tool_usage.get(
        "llm_calls",
        0,
    )
    if isinstance(tool_usage, dict)
    else 0
)

retrieval_rounds = int(
    tool_usage.get(
        "retrieval_rounds",
        0,
    )
    if isinstance(tool_usage, dict)
    else 0
)

validation_calls = int(
    tool_usage.get(
        "validation_calls",
        0,
    )
    if isinstance(tool_usage, dict)
    else 0
)

openai_invoked = llm_calls > 0

# ============================================================
# 14. MOSTRAR RESULTADO
# ============================================================

print("\n" + "=" * 100)
print("RESULTADO DEL AGENTE 06 EXPERIMENTAL")
print("=" * 100)

print(
    "execution_status:",
    execution_status,
)

print(
    "quality_status:",
    quality_status,
)

print(
    "decision_code:",
    decision_code,
)

print(
    "requested_transition:",
    transition_action,
)

print(
    "retrieval_rounds:",
    retrieval_rounds,
)

print(
    "llm_calls:",
    llm_calls,
)

print(
    "validation_calls:",
    validation_calls,
)

print(
    "OpenAI realmente invocado:",
    openai_invoked,
)

print("\nFailure reason codes:")
print(
    list(
        result.failure_reason_codes
        or ()
    )
)

print("\nWarnings:")

if result.warnings:
    for warning in result.warnings:
        print(
            to_serializable(warning)
        )
else:
    print([])

# ============================================================
# 15. LEER REPORTE GLOBAL SI EXISTE
# ============================================================

global_report_path = (
    EXPERIMENT_OUTPUT_DIR
    / "draft_validation_report.json"
)

global_report = None

if global_report_path.is_file():
    global_report = json.loads(
        global_report_path.read_text(
            encoding="utf-8"
        )
    )

    print("\n" + "=" * 100)
    print("REPORTE GLOBAL")
    print("=" * 100)

    fields_to_show = [
        "validation_ok",
        "invalid_citation_count",
        "numeric_failure_count",
        "total_words",
        "target_total_words",
        "global_length_valid",
        "all_section_validations_ok",
        "sections_outside_word_range",
    ]

    for field in fields_to_show:
        print(
            f"{field}:",
            global_report.get(field),
        )
else:
    print(
        "\nNo se generó draft_validation_report.json"
    )

# ============================================================
# 16. GUARDAR RESULTADO EXPERIMENTAL
# ============================================================

result_payload = {
    "experiment_type": (
        "agent06_hybrid_quantitative_"
        "greedy_source_aware_generation"
    ),
    "contractual_execution": False,
    "pipeline_state_modified": False,
    "attempt_number_created": False,
    "baseline_preserved": True,
    "openai_invoked": openai_invoked,
    "retrieval_variant": (
        "hybrid_thematic_candidate24_"
        "balanced_3_3_rrf_plus_"
        "quantitative_greedy_plus_"
        "source_aware_budgets"
    ),
    "quantitative_evidence_quota": 2,
    "organizational_target_words": 40,
    "diagnostic_coverage_before_generation": {
        "covered_values": 7,
        "target_values": 7,
        "coverage_rate": 1.0,
    },
    "source_aware_budget_simulation": {
        "sections_outside_range": [],
        "simulated_validation_ok": True,
    },
    "project_dir": str(PROJECT_DIR),
    "experiment_id": cfg["experiment_id"],
    "experimental_output_directory": str(
        EXPERIMENT_OUTPUT_DIR
    ),
    "contractual_output_directory": str(
        contractual_output_dir
    ),
    "pipeline_state_path": str(
        pipeline_state_path
    ),
    "pipeline_state_sha256_before": (
        state_hash_before
    ),
    "pipeline_state_sha256_after": (
        state_hash_after
    ),
    "execution_status": execution_status,
    "quality_status": quality_status,
    "decision_code": decision_code,
    "requested_transition": transition_action,
    "tool_usage": tool_usage,
    "global_validation_report": global_report,
    "agent_result": to_serializable(result),
}

result_file = (
    EXPERIMENT_OUTPUT_DIR
    / "experimental_agent_result.json"
)

result_file.write_text(
    json.dumps(
        result_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nResultado guardado:")
print(result_file)

# ============================================================
# 17. MOSTRAR ARCHIVOS GENERADOS
# ============================================================

print("\n" + "=" * 100)
print("ARCHIVOS GENERADOS")
print("=" * 100)

generated_files = sorted(
    path
    for path
    in EXPERIMENT_OUTPUT_DIR.rglob("*")
    if path.is_file()
)

for path in generated_files:
    print(
        path.relative_to(
            EXPERIMENT_OUTPUT_DIR
        )
    )

# ============================================================
# 18. VERIFICAR BASELINE AL FINAL
# ============================================================

baseline_hash_after = sha256_file_local(
    BASELINE_AGENT_PATH
)

baseline_modified = (
    baseline_hash_before
    != baseline_hash_after
)

assert not baseline_modified, (
    "El agente baseline fue modificado"
)

# ============================================================
# 19. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 100)
print("RESUMEN FINAL")
print("=" * 100)

print(
    "Agente usado:",
    agent.__class__.__module__,
)

print(
    "Retrieval:",
    experimental_input.policy.get(
        "retrieval_variant"
    ),
)

print(
    "Cobertura cuantitativa previa:",
    "7/7 = 100%",
)

print(
    "Presupuestos source-aware:",
    True,
)

print(
    "OpenAI realmente invocado:",
    openai_invoked,
)

print(
    "Llamadas LLM:",
    llm_calls,
)

print(
    "PipelineState modificado:",
    pipeline_state_modified,
)

print(
    "Intento contractual creado:",
    False,
)

print(
    "Agente baseline modificado:",
    baseline_modified,
)

print(
    "Baseline sobrescrito:",
    False,
)

print(
    "Calidad obtenida:",
    quality_status,
)

print(
    "Decisión:",
    decision_code,
)

CODE_ROOT: /content/tesis_codigo
PROJECT_DIR: /content/proyecto_estado_arte

Hash agente baseline:
544201c3be8763a3ccdfde22f570af3391a0366db11b40918395e08d90ad231c

VERIFICACIÓN DEL AGENTE EXPERIMENTAL
✓ retrieve_section_evidence_hybrid_experimental
✓ augment_evidence_with_quantitative_chunks_greedy
✓ assign_source_aware_section_budgets
✓ quantitative_evidence_quota
✓ organizational_target_words
Sintaxis de componentes experimentales: válida

Runtime experimental:
/content/tesis_codigo/src/adapters/draft_writing_hybrid_runtime.py

Agente construido:
src.agents.draft_writing_agent_hybrid_experimental
DraftWritingAgent

Salida contractual:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft

Salida experimental:
/content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/agent06_hybrid_quantitative_greedy_source_aware_v1

AgentInput experimental
output_directory: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/05_draft_experiments/a